## Citi Bike NYC 2018

# Data Preparation

In [ ]:
import os
import pandas as pd
from typing import Optional

pd.set_option('display.max_columns', None)

DATA_DIR = os.path.abspath('data')
RAW_TRIPS_PATH = os.path.join(DATA_DIR, 'Trips_2018.csv')
HOLIDAYS_PATH = os.path.join(DATA_DIR, 'holidays_2018_nyc.csv')
EVENTS_PATH = os.path.join(DATA_DIR, 'events_2018_nyc.csv')

RAW_TRIPS_PATH, HOLIDAYS_PATH, EVENTS_PATH


In [ ]:
# Load holidays and events
holidays = pd.read_csv(HOLIDAYS_PATH, parse_dates=['date'])
# Normalize date to date only
holidays['date'] = holidays['date'].dt.normalize()

events = pd.read_csv(EVENTS_PATH, parse_dates=['start_datetime', 'end_datetime'])

print('Holidays:', holidays.shape)
print('Events:', events.shape)
holidays.head(2)


In [ ]:
events.head(2)

### Load raw rides (chunked) and validate schema
We will stream the large CSV to avoid memory pressure. The loader:
- Infers likely Citi Bike column names and standardizes to a canonical schema
- Parses datetimes safely
- drops invalid rows and outliers
- Tracks basic QA metrics (missing rates, outlier rates)



In [ ]:
CANONICAL_COLS = {
    "unnamed: 0": "id",
    "tripduration": "trip_duration_sec",
    "starttime": "start_time",
    "stoptime": "end_time",
    "start_station_id": "start_station_id",
    "start_station_latitude": "start_lat",
    "start_station_longitude": "start_lng",
    "end_station_id": "end_station_id",
    "end_station_latitude": "end_lat",
    "end_station_longitude": "end_lng",
    "bikeid": "bike_id",
    "usertype": "user_type",
    "birth_year": "birth_year",
    "gender": "gender",
}

CANONICAL_SET = set(CANONICAL_COLS.values())

DTYPES_BASE = {
    "trip_duration_sec": "float64",
    "start_station_id": "int",
    "end_station_id": "int",
    "bike_id": "string",
    "user_type": "string",
    "birth_year": "float64",
    "gender": "string",
    "start_lat": "float64",
    "start_lng": "float64",
    "end_lat": "float64",
    "end_lng": "float64",
}

CHUNKSIZE = 250_000


def _standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    mapping = {}
    for col in df.columns:
        low = col.strip().lower()
        mapping[col] = CANONICAL_COLS[low]

    df = df.rename(columns=mapping)
    return df


def _parse_times(df: pd.DataFrame) -> pd.DataFrame:
    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    return df


def _compute_duration_if_missing(df: pd.DataFrame) -> pd.DataFrame:
    if "trip_duration_sec" not in df.columns:
        df["trip_duration_sec"] = (df["end_time"] - df["start_time"]).dt.total_seconds()
    return df


def _filter_invalid(df: pd.DataFrame) -> pd.DataFrame:
    # Drop invalid times
    if {"start_time", "end_time"}.issubset(df.columns):
        df = df[df["start_time"].notna() & df["end_time"].notna()]
        df = df[df["end_time"] >= df["start_time"]]
    # Duration sanity: 1 minute to 24 hours
    if "trip_duration_sec" in df.columns:
        df = df[
            (df["trip_duration_sec"] >= 60) & (df["trip_duration_sec"] <= 24 * 3600)
        ]
    return df


def _cast_types(df: pd.DataFrame) -> pd.DataFrame:
    for col, dtype in DTYPES_BASE.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype, errors="ignore")
    return df


def _filter_geographic_outliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove trips with stations outside NYC geographic boundaries.

    This filters out erroneous GPS coordinates and stations in other cities
    (e.g., Montreal stations that appeared in the dataset).
    """
    # NYC bounding box (with small buffer for edge stations)
    NYC_LAT_MIN = 40.60
    NYC_LAT_MAX = 40.85
    NYC_LNG_MIN = -74.15
    NYC_LNG_MAX = -73.83

    initial_len = len(df)

    # Filter start station coordinates
    if {"start_lat", "start_lng"}.issubset(df.columns):
        df = df[
            (df["start_lat"] >= NYC_LAT_MIN)
            & (df["start_lat"] <= NYC_LAT_MAX)
            & (df["start_lng"] >= NYC_LNG_MIN)
            & (df["start_lng"] <= NYC_LNG_MAX)
        ]

    # Filter end station coordinates
    if {"end_lat", "end_lng"}.issubset(df.columns):
        df = df[
            (df["end_lat"] >= NYC_LAT_MIN)
            & (df["end_lat"] <= NYC_LAT_MAX)
            & (df["end_lng"] >= NYC_LNG_MIN)
            & (df["end_lng"] <= NYC_LNG_MAX)
        ]

    return df


def load_trips_stream(path: str, limit_chunks: Optional[int] = None) -> pd.DataFrame:
    qa_stats = []
    chunks = []
    for i, chunk in enumerate(pd.read_csv(path, chunksize=CHUNKSIZE, low_memory=True)):
        c0 = len(chunk)
        chunk = _standardize_columns(chunk)
        chunk = _parse_times(chunk)
        chunk = _compute_duration_if_missing(chunk)
        chunk = chunk.dropna()
        chunk = _filter_geographic_outliers(chunk)
        # keep only canonical columns if present
        cols_to_keep = [
            c
            for c in chunk.columns
            if c in CANONICAL_SET or c in {"start_time", "end_time"}
        ]
        chunk = chunk[cols_to_keep]
        chunk = _filter_invalid(chunk)
        chunk = _cast_types(chunk)
        c1 = len(chunk)
        qa_stats.append(
            {
                "chunk": i,
                "rows_dropped": c0 - c1,
                "drop_rate": (c0 - c1) / max(1, c0),
            }
        )
        chunks.append(chunk)
        if limit_chunks is not None and i + 1 >= limit_chunks:
            break
    trips = pd.concat(chunks, ignore_index=True)
    qa = pd.DataFrame(qa_stats)
    print("QA summary:")
    display(qa.describe(include="all"))
    print(f"Total dropped: {qa['rows_dropped'].sum()}")
    return trips


# Preview small sample for speed; set limit_chunks=None to process all
trips_sample = load_trips_stream(RAW_TRIPS_PATH, limit_chunks=None)
trips_sample.head()

In [ ]:
trips_sample.shape

In [ ]:
trips_sample.isnull().sum()

## Station data and activity metrics

Now that we have a clean trip dataset, we need to create a station-level view of the system. 

For each unique station, we will:
1. Extract station metadata: ID, latitude, longitude
2. Aggregate demand metrics:
   - Pickups: Number of trips that started at this station
   - Dropoffs: Number of trips that ended at this station
   - Total activity: Sum of pickups and dropoffs

This station-centric dataset will enable:
- Spatial clustering (grouping nearby stations)
- Demand pattern analysis (identifying high/low activity stations)

This aggregation transforms 17.5M trip records into 846 station profiles that will help in the futire for clustering and forecasting.

In [ ]:
# Extract start station metadata
start_stations = trips_sample[['start_station_id', 'start_lat', 'start_lng']].copy()
start_stations.columns = ['station_id', 'lat', 'lng']

# Extract end station metadata
end_stations = trips_sample[['end_station_id', 'end_lat', 'end_lng']].copy()
end_stations.columns = ['station_id', 'lat', 'lng']

# Combine and remove duplicates
stations = pd.concat([start_stations, end_stations], ignore_index=True)
stations = stations.drop_duplicates(subset='station_id').reset_index(drop=True)

print(f"Total unique stations: {len(stations)}")
stations.head()

In [ ]:
# Count how many trips start/end in each station
start_counts = trips_sample.groupby('start_station_id').size().rename('pickup_count')
end_counts = trips_sample.groupby('end_station_id').size().rename('dropoff_count')
display(start_counts.head())
display(end_counts.head())  

In [ ]:
# Merge with station metadata
stations = stations.merge(start_counts, left_on='station_id', right_index=True, how='left')
stations = stations.merge(end_counts, left_on='station_id', right_index=True, how='left')

# Fill NaN with 0 (stations without trips)
stations[['pickup_count', 'dropoff_count']] = stations[['pickup_count', 'dropoff_count']].fillna(0).astype(int)

# Calculate total activity
stations['total_trips'] = stations['pickup_count'] + stations['dropoff_count']

# Sort by activity
stations = stations.sort_values('total_trips', ascending=False).reset_index(drop=True)

print(f"Total unique stations: {len(stations)}")
stations.head(10)

## Visualization (geographic distribution) of bike stations


Below is an interactive map showing all 846 Citi Bike stations in NYC. Each marker represents a station, with:
- **Color intensity**: Indicates total trip volume (darker = higher activity)
- **Popup info**: Station ID, coordinates, and trip counts (pickups/dropoffs)

This visualization helps identify:
- High-demand areas (Manhattan Midtown, Financial District)
- Geographic coverage and density patterns
- Potential clusters for spatial aggregation

In [ ]:
# dependencies we will use
import folium
from folium.plugins import MarkerCluster
import numpy as np

In [ ]:
# Create base map centered on NYC
nyc_center = [stations["lat"].mean(), stations["lng"].mean()]
m = folium.Map(location=nyc_center, zoom_start=12, tiles="CartoDB Positron")

NYC_LAT_MIN = 40.60
NYC_LAT_MAX = 40.85
NYC_LNG_MIN = -74.15
NYC_LNG_MAX = -73.83
# NYC bounding box
folium.Rectangle(
    bounds=[[NYC_LAT_MIN, NYC_LNG_MIN], [NYC_LAT_MAX, NYC_LNG_MAX]],
    color="blue",
    fill=False,
    weight=3,
    opacity=0.8,
    popup="NYC Bounding Box",
).add_to(m)

# Normalize total_trips for color scaling
min_trips = stations["total_trips"].min()
max_trips = stations["total_trips"].max()

# Add markers for each station
for idx, row in stations.iterrows():
    normalized = (row["total_trips"] - min_trips) / (max_trips - min_trips)

    if normalized > 0.7:
        color = "red"
    elif normalized > 0.4:
        color = "orange"
    elif normalized > 0.2:
        color = "yellow"
    else:
        color = "lightgreen"

    # Create popup with station info
    popup_html = f"""
    <div style="font-family: Arial; font-size: 12px;">
        <b>Station ID:</b> {row['station_id']}<br>
        <b>Location:</b> ({row['lat']:.4f}, {row['lng']:.4f})<br>
        <b>Pickups:</b> {row['pickup_count']:,}<br>
        <b>Dropoffs:</b> {row['dropoff_count']:,}<br>
        <b>Total Trips:</b> {row['total_trips']:,}
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=5,
        popup=folium.Popup(popup_html, max_width=250),
        color=color,
        fill=True,
        fillColor=color,
        fillOpacity=0.8,
        weight=1,
    ).add_to(m)

# Add legend
legend_html = """
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 180px; height: 150px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:12px; padding: 10px">
<p><b>Station Activity</b></p>
<p><i class="fa fa-circle" style="color:red"></i> Very High (70%+)</p>
<p><i class="fa fa-circle" style="color:orange"></i> High (40-70%)</p>
<p><i class="fa fa-circle" style="color:yellow"></i> Medium (20-40%)</p>
<p><i class="fa fa-circle" style="color:lightgreen"></i> Low (<20%)</p>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
m

## Spatial Clustering Strategy

To reduce model complexity and capture local demand patterns, we will group nearby stations into geographic clusters.

### Clustering Approach: K-Means

We'll use **K-Means clustering** on station coordinates (lat/lng) to create 20-30 spatial clusters. This approach:

1. **Groups geographically proximate stations** → Reduces 846 stations to ~25 clusters
2. **Preserves local demand patterns** → Stations in the same neighborhood share similar characteristics
3. **Simplifies forecasting** → Predict demand at cluster level instead of individual stations
4. **Enables bike rebalancing** → Identify clusters with pickup/dropoff imbalances

### Cluster Selection Criteria:
- **Minimum 20 clusters** (per project requirements)
- **Balanced cluster sizes** → Avoid clusters with too few/many stations
- **Geographic coherence** → Clusters should represent meaningful neighborhoods

The map below shows a preview of how stations will be grouped into clusters based on their geographic proximity.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import joblib

# Perform K-Means clustering
n_clusters = 30

# Prepare coordinates for clustering
coords = stations[["lat", "lng"]].values

# Fit K-Means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
stations["cluster"] = kmeans.fit_predict(coords)
joblib.dump(kmeans, "kmeans_model.pkl")

# Get cluster centers
cluster_centers = kmeans.cluster_centers_

print(f"Created {n_clusters} clusters")
print(f"Cluster size distribution:")
print(stations["cluster"].value_counts().describe())

# Aggregate metrics by cluster
cluster_stats = (
    stations.groupby("cluster")
    .agg(
        {
            "station_id": "count",
            "pickup_count": "sum",
            "dropoff_count": "sum",
            "total_trips": "sum",
            "lat": "mean",
            "lng": "mean",
        }
    )
    .rename(columns={"station_id": "num_stations"})
)

cluster_stats["balance"] = (
    cluster_stats["pickup_count"] - cluster_stats["dropoff_count"]
)
cluster_stats = cluster_stats.sort_values("total_trips", ascending=False)

print("\n📍 Top 5 clusters by activity:")
display(cluster_stats.head())

# Create clustered map
m_clustered = folium.Map(location=nyc_center, zoom_start=12, tiles="CartoDB positron")

# Define color palette for clusters
colors = plt.cm.tab20(np.linspace(0, 1, n_clusters))
color_map = {
    i: f"#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}"
    for i, c in enumerate(colors)
}

for cluster_id in range(n_clusters):
    cluster_stations = stations[stations["cluster"] == cluster_id]

    # Add individual station markers
    for idx, row in cluster_stations.iterrows():
        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px;">
            <b>Cluster:</b> {cluster_id}<br>
            <b>Station ID:</b> {row['station_id']}<br>
            <b>Total Trips:</b> {row['total_trips']:,}
        </div>
        """

        folium.CircleMarker(
            location=[row["lat"], row["lng"]],
            radius=4,
            popup=folium.Popup(popup_html, max_width=200),
            color=color_map[cluster_id],
            fill=True,
            fillColor=color_map[cluster_id],
            fillOpacity=0.6,
            weight=1,
        ).add_to(m_clustered)

    # Add cluster center marker
    center_lat, center_lng = cluster_centers[cluster_id]
    cluster_info = cluster_stats.loc[cluster_id]

    center_popup = f"""
    <div style="font-family: Arial; font-size: 13px;">
        <b>Cluster {cluster_id}</b><br>
        <b>Stations:</b> {cluster_info['num_stations']}<br>
        <b>Total Trips:</b> {cluster_info['total_trips']:,}<br>
        <b>Pickups:</b> {cluster_info['pickup_count']:,}<br>
        <b>Dropoffs:</b> {cluster_info['dropoff_count']:,}<br>
        <b>Balance:</b> {cluster_info['balance']:+,}
    </div>
    """

    folium.Marker(
        location=[center_lat, center_lng],
        popup=folium.Popup(center_popup, max_width=250),
        icon=folium.DivIcon(
            html=f"""
        <div style="
            font-size: 18px; 
            font-weight: 900; 
            color: {color_map[cluster_id]}; 
            background-color: white;
            border: 2px solid {color_map[cluster_id]};
            border-radius: 4px;
            width: 35px;
            height: 35px;
            display: flex;
            align-items: center;
            justify-content: center;
            box-shadow: 0 2px 6px rgba(0,0,0,0.3);
        ">
            {cluster_id}
        </div>
    """
        ),
        tooltip=center_popup,
    ).add_to(m_clustered)

# Add title
title_html = f"""
<div style="position: fixed; 
            top: 10px; left: 50px; width: 400px; height: 50px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:16px; padding: 10px">
<b>Citi Bike Stations - K-Means Clustering (k={n_clusters})</b>
</div>
"""
m_clustered.get_root().html.add_child(folium.Element(title_html))

# Display clustered map
m_clustered

In [ ]:
stations

stations.to_csv("cluster_data")



In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Cluster size distribution
axes[0, 0].bar(cluster_stats.index, cluster_stats["num_stations"], color="steelblue")
axes[0, 0].set_xlabel("Cluster ID")
axes[0, 0].set_ylabel("Number of Stations")
axes[0, 0].set_title("Stations per Cluster")
axes[0, 0].grid(axis="y", alpha=0.3)

# 2. Total trips per cluster
top_10 = cluster_stats.nlargest(10, "total_trips")
axes[0, 1].barh(top_10.index.astype(str), top_10["total_trips"], color="coral")
axes[0, 1].set_xlabel("Total Trips")
axes[0, 1].set_ylabel("Cluster ID")
axes[0, 1].set_title("Top 10 Clusters by Activity")
axes[0, 1].invert_yaxis()

# 3. Pickup vs Dropoff balance
axes[1, 0].scatter(
    cluster_stats["pickup_count"],
    cluster_stats["dropoff_count"],
    s=cluster_stats["num_stations"] * 10,
    alpha=0.6,
    c=cluster_stats.index,
    cmap="tab20",
)
axes[1, 0].plot(
    [0, cluster_stats["pickup_count"].max()],
    [0, cluster_stats["pickup_count"].max()],
    "k--",
    alpha=0.3,
    label="Perfect balance",
)
axes[1, 0].set_xlabel("Total Pickups")
axes[1, 0].set_ylabel("Total Dropoffs")
axes[1, 0].set_title("Cluster Balance: Pickups vs Dropoffs")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Imbalance distribution
axes[1, 1].hist(
    cluster_stats["balance"], bins=15, color="purple", alpha=0.7, edgecolor="black"
)
axes[1, 1].axvline(0, color="red", linestyle="--", linewidth=2, label="Perfect balance")
axes[1, 1].set_xlabel("Pickup - Dropoff Balance")
axes[1, 1].set_ylabel("Number of Clusters")
axes[1, 1].set_title("Distribution of Cluster Imbalances")
axes[1, 1].legend()
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("Clustering Summary:")
print(f"   • Total clusters: {n_clusters}")
print(f"   • Avg stations per cluster: {cluster_stats['num_stations'].mean():.1f}")
print(
    f"   • Most balanced cluster: {cluster_stats['balance'].abs().idxmin()} (balance: {cluster_stats['balance'].abs().min():+,.0f})"
)
print(
    f"   • Most imbalanced cluster: {cluster_stats['balance'].abs().idxmax()} (balance: {cluster_stats.loc[cluster_stats['balance'].abs().idxmax(), 'balance']:+,.0f})"
)

# Feature Engineering

In [ ]:
stations.head()

In [ ]:
trips_sample.head()

In [ ]:
trips_sample.groupby('user_type').size()

## Feature Engineering for Demand Forecasting

Now that we have clean trip data and station clusters, we need to transform this into a **time-series forecasting dataset**.

### Goal
Create a dataset where each row represents:
- **One cluster** at **one specific hour**
- Target variables: `pickups` and `dropoffs` (number of trips)
- Features: temporal patterns, cluster characteristics, historical demand

### Transformation Steps
1. **Aggregate trips to cluster-hour level** (from 17M trips → ~500K cluster-hours)
2. **Engineer temporal features** (month, day of week, season, holidays)
3. **Add cluster-level features** (average demand, station count, location)
4. **Create lag features** (previous day/week demand for time-series context)
5. **Normalize continuous variables** (standardization for model training)

This structure enables us to predict the next 24 hours of demand for each cluster.

In [ ]:
# ============================================================================
# STEP 1: Aggregate Trips to Cluster-Hour Level
# ============================================================================

print("Aggregating trips to cluster-hour level...")

# Add temporal columns to trips
trips_sample['start_hour'] = trips_sample['start_time'].dt.hour
trips_sample['start_date'] = trips_sample['start_time'].dt.date
trips_sample['end_hour'] = trips_sample['end_time'].dt.hour
trips_sample['end_date'] = trips_sample['end_time'].dt.date

# Merge trips with station clusters (for pickups)
trips_with_start_cluster = trips_sample.merge(
    stations[['station_id', 'cluster']], 
    left_on='start_station_id', 
    right_on='station_id',
    how='left'
).drop(columns=['station_id'])

# Merge trips with station clusters (for dropoffs)
trips_with_end_cluster = trips_sample.merge(
    stations[['station_id', 'cluster']], 
    left_on='end_station_id', 
    right_on='station_id',
    how='left',
    suffixes=('', '_end')
).drop(columns=['station_id'])

# Aggregate pickups by cluster-date-hour
pickups_hourly = trips_with_start_cluster.groupby(
    ['start_date', 'start_hour', 'cluster']
).agg({
    'id': 'count',  # Number of pickups
    'trip_duration_sec': 'mean',  # Average trip duration
    'birth_year': 'mean',  # Average birth year (for age calculation)
    'user_type': lambda x: (x == 'Subscriber').sum() / len(x)  # % subscribers
}).reset_index()

pickups_hourly.columns = ['date', 'hour', 'cluster', 'pickups', 'avg_trip_duration', 'avg_birth_year', 'pct_subscribers']

# Aggregate dropoffs by cluster-date-hour
dropoffs_hourly = trips_with_end_cluster.groupby(
    ['end_date', 'end_hour', 'cluster']
).size().reset_index(name='dropoffs')
dropoffs_hourly.columns = ['date', 'hour', 'cluster', 'dropoffs']

# Merge pickups and dropoffs
demand_hourly = pickups_hourly.merge(
    dropoffs_hourly, 
    on=['date', 'hour', 'cluster'], 
    how='outer'
).fillna(0)

# Calculate age from birth year
current_year = 2018
demand_hourly['avg_age'] = current_year - demand_hourly['avg_birth_year']
demand_hourly = demand_hourly.drop(columns=['avg_birth_year'])

print(f"Created {len(demand_hourly):,} cluster-hour observations")
print(f"Date range: {demand_hourly['date'].min()} to {demand_hourly['date'].max()}")
print(f"Clusters: {demand_hourly['cluster'].nunique()}")

demand_hourly.head(10)

In [ ]:
# ============================================================================
# STEP 2: Engineer Temporal Features
# ============================================================================

print("Creating temporal features...")

# Convert date to datetime for feature extraction
demand_hourly['date'] = pd.to_datetime(demand_hourly['date'])

# Basic temporal features
demand_hourly['month'] = demand_hourly['date'].dt.month
demand_hourly['day_of_week'] = demand_hourly['date'].dt.dayofweek  # 0=Monday, 6=Sunday
demand_hourly['day_of_month'] = demand_hourly['date'].dt.day
demand_hourly['week_of_year'] = demand_hourly['date'].dt.isocalendar().week

# Weekend indicator
demand_hourly['is_weekend'] = (demand_hourly['day_of_week'] >= 5).astype(int)

# Season dummies (Northern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 'winter'
    elif month in [3, 4, 5]:
        return 'spring'
    elif month in [6, 7, 8]:
        return 'summer'
    else:  # 9, 10, 11
        return 'fall'

demand_hourly['season'] = demand_hourly['month'].apply(get_season)

# Create season dummies
season_dummies = pd.get_dummies(demand_hourly['season'], prefix='season')
demand_hourly = pd.concat([demand_hourly, season_dummies], axis=1)

print("Temporal features created:")
print(f"  - Month, day_of_week, day_of_month, week_of_year")
print(f"  - is_weekend")
print(f"  - Season dummies: {list(season_dummies.columns)}")

demand_hourly.head()

In [ ]:
# ============================================================================
# STEP 3: Add Holiday and Event Indicators
# ============================================================================

print("Merging holiday and event data...")

# Normalize holiday dates
holidays['date'] = pd.to_datetime(holidays['date']).dt.normalize()
demand_hourly['date_normalized'] = demand_hourly['date'].dt.normalize()

# Merge holidays
demand_hourly = demand_hourly.merge(
    holidays[['date']].assign(is_holiday=1),
    left_on='date_normalized',
    right_on='date',
    how='left',
    suffixes=('', '_holiday')
).drop(columns=['date_holiday'])

demand_hourly['is_holiday'] = demand_hourly['is_holiday'].fillna(0).astype(int)

# For events, check if the cluster-hour falls within any event time window
# (This is more complex - simplified version: mark days with events)
event_dates = pd.to_datetime(events['start_datetime']).dt.date.unique()
demand_hourly['is_special_event'] = demand_hourly['date_normalized'].dt.date.isin(event_dates).astype(int)

demand_hourly = demand_hourly.drop(columns=['date_normalized'])

print(f"Holidays marked: {demand_hourly['is_holiday'].sum()} cluster-hours")
print(f"Special events marked: {demand_hourly['is_special_event'].sum()} cluster-hours")

demand_hourly.head()

In [ ]:
# ============================================================================
# STEP 4: Add Cluster-Level Features
# ============================================================================

print("Creating cluster-level features...")

# Calculate cluster statistics from stations
cluster_stats = stations.groupby('cluster').agg({
    'station_id': 'count',  # Number of stations in cluster
    'lat': 'mean',  # Cluster center latitude
    'lng': 'mean',  # Cluster center longitude
    'total_trips': 'sum'  # Total historical trips in cluster
}).reset_index()

cluster_stats.columns = ['cluster', 'cluster_station_count', 'cluster_center_lat', 'cluster_center_lng', 'cluster_total_trips']

# Merge with demand data
demand_hourly = demand_hourly.merge(cluster_stats, on='cluster', how='left')

print("Cluster features added:")
print(f"  - cluster_station_count (stations per cluster)")
print(f"  - cluster_center_lat, cluster_center_lng")
print(f"  - cluster_total_trips (historical total)")

demand_hourly.head()

In [ ]:
# ============================================================================
# STEP 5: Create Lag Features (Time Series Context)
# ============================================================================

print("Creating lag features for time-series forecasting...")

# Sort by cluster and datetime
demand_hourly = demand_hourly.sort_values(['cluster', 'date', 'hour']).reset_index(drop=True)

# Create lags for pickups and dropoffs
for target in ['pickups', 'dropoffs']:
    # Previous day same hour (24 hours ago)
    demand_hourly[f'{target}_lag_24h'] = demand_hourly.groupby('cluster')[target].shift(24)
    
    # Previous week same hour (168 hours ago)
    demand_hourly[f'{target}_lag_168h'] = demand_hourly.groupby('cluster')[target].shift(168)
    
    # Rolling average last 24 hours
    demand_hourly[f'{target}_rolling_24h'] = demand_hourly.groupby('cluster')[target].transform(
        lambda x: x.rolling(window=24, min_periods=1).mean()
    )

print("Lag features created:")
print(f"  - pickups/dropoffs_lag_24h (yesterday same hour)")
print(f"  - pickups/dropoffs_lag_168h (last week same hour)")
print(f"  - pickups/dropoffs_rolling_24h (24-hour moving average)")

# Check for NaN values in lag features (expected for first observations)
print(f"\nNaN values in lag features (first {24*7} hours per cluster):")
print(demand_hourly[['pickups_lag_24h', 'pickups_lag_168h']].isnull().sum())

demand_hourly.head(30)

## Time Series Visualization: Understanding Demand Patterns

Before normalizing and training models, let's visualize the temporal patterns in our data. We'll focus on a high-activity cluster to see clear patterns.

These visualizations will help us:
1. **Identify trends and seasonality** (daily, weekly, monthly patterns)
2. **Validate feature engineering** (do holidays/weekends affect demand?)
3. **Detect anomalies** (unexpected spikes or drops)
4. **Understand cluster behavior** (rush hours, weekend patterns)

In [ ]:
# ============================================================================
# Setup for Visualizations
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)

# Select a high-activity cluster for visualization
top_cluster = demand_hourly.groupby('cluster')['pickups'].sum().idxmax()
print(f"Selected cluster {top_cluster} (highest total pickups) for visualization")

# Filter data for this cluster
cluster_data = demand_hourly[demand_hourly['cluster'] == top_cluster].copy()
cluster_data['datetime'] = pd.to_datetime(cluster_data['date']) + pd.to_timedelta(cluster_data['hour'], unit='h')
cluster_data = cluster_data.sort_values('datetime')

print(f"Cluster {top_cluster} has {len(cluster_data):,} hourly observations")
print(f"Date range: {cluster_data['datetime'].min()} to {cluster_data['datetime'].max()}")

### 1. Full Year Time Series: Pickups Over Time
This chart shows the complete demand pattern for the busiest cluster throughout 2018.

In [ ]:
# ============================================================================
# Visualization 1: Full Year Pickups Time Series
# ============================================================================

fig, ax = plt.subplots(figsize=(18, 6))

ax.plot(cluster_data['datetime'], cluster_data['pickups'], 
        linewidth=0.8, alpha=0.7, color='steelblue')

ax.set_title(f'Cluster {top_cluster}: Hourly Pickups Throughout 2018', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Pickups', fontsize=12)

# Format x-axis to show months
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

# Add grid
ax.grid(True, alpha=0.3)

# Add statistics text box
stats_text = f"Mean: {cluster_data['pickups'].mean():.1f} | Median: {cluster_data['pickups'].median():.1f} | Max: {cluster_data['pickups'].max():.0f}"
ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"📊 Observation: Notice any seasonal trends? Summer vs Winter patterns?")

### 2. Time Series with Weekend Highlighting
This visualization highlights weekends to see if demand patterns differ on Saturdays and Sundays.

In [ ]:
# ============================================================================
# Visualization 2: Pickups with Weekend Highlighting
# ============================================================================

# Focus on a 2-month period for clarity
sample_start = '2018-06-01'
sample_end = '2018-07-31'
cluster_sample = cluster_data[(cluster_data['datetime'] >= sample_start) & 
                               (cluster_data['datetime'] <= sample_end)]

fig, ax = plt.subplots(figsize=(18, 6))

# Plot the time series
ax.plot(cluster_sample['datetime'], cluster_sample['pickups'], 
        linewidth=1.2, color='steelblue', label='Pickups')

# Highlight weekends
weekend_dates = cluster_sample[cluster_sample['is_weekend'] == 1]['datetime'].dt.date.unique()
for weekend_date in weekend_dates:
    weekend_start = pd.to_datetime(weekend_date)
    weekend_end = weekend_start + pd.Timedelta(days=1)
    ax.axvspan(weekend_start, weekend_end, alpha=0.2, color='orange', label='Weekend' if weekend_date == weekend_dates[0] else '')

ax.set_title(f'Cluster {top_cluster}: Pickups with Weekend Highlighting (Jun-Jul 2018)', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Pickups', fontsize=12)

# Format x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=45)

ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"📊 Observation: Do weekends show different patterns? Higher or lower demand?")

### 3. Time Series with Holiday Markers
Major holidays are marked with vertical lines to assess their impact on bike demand.

In [ ]:
# ============================================================================
# Visualization 3: Pickups with Holiday Markers
# ============================================================================

fig, ax = plt.subplots(figsize=(18, 6))

# Plot the time series
ax.plot(cluster_data['datetime'], cluster_data['pickups'], 
        linewidth=0.8, alpha=0.7, color='steelblue')

# Mark holidays with vertical lines
holiday_dates = cluster_data[cluster_data['is_holiday'] == 1]['datetime'].dt.date.unique()
for i, holiday_date in enumerate(holiday_dates):
    holiday_datetime = pd.to_datetime(holiday_date)
    ax.axvline(holiday_datetime, color='red', linestyle='--', linewidth=2, 
               alpha=0.6, label='Holiday' if i == 0 else '')
    
    # Add holiday name annotation (if available from holidays dataframe)
    holiday_info = holidays[holidays['date'].dt.date == holiday_date]
    if not holiday_info.empty:
        holiday_name = holiday_info.iloc[0]['name']
        ax.text(holiday_datetime, ax.get_ylim()[1] * 0.95, holiday_name, 
                rotation=90, verticalalignment='top', fontsize=8, color='red')

ax.set_title(f'Cluster {top_cluster}: Pickups with Holiday Markers', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Pickups', fontsize=12)

# Format x-axis
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"📊 Observation: Do holidays show significant drops in demand? Which holidays have the biggest impact?")

## Zoom in to independence day for example:

compare 3 days:
- before
- during
- after

In [ ]:
# compare 3 days: before, during, after a holiday
holiday_date = pd.to_datetime('2018-07-04')  # Independence Day
before = holiday_date - pd.Timedelta(days=1)
after = holiday_date + pd.Timedelta(days=1)

holiday_window = cluster_data[cluster_data['datetime'].between(before, after + pd.Timedelta(days=1))]

fig, ax = plt.subplots(figsize=(14, 6))
for date in [before, holiday_date, after]:
    day_data = holiday_window[holiday_window['datetime'].dt.date == date.date()]
    label = 'Holiday' if date == holiday_date else date.strftime('%b %d')
    linestyle = '--' if date == holiday_date else '-'
    linewidth = 3 if date == holiday_date else 1.5
    ax.plot(day_data['hour'], day_data['pickups'], marker='o', 
            label=label, linestyle=linestyle, linewidth=linewidth)

ax.set_title(f'Cluster {top_cluster}: Independence Day Impact (July 3-5, 2018)', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Pickups', fontsize=12)
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Although July 4, 2018, was a Wednesday, the chart shows that it behaves like a weekend, with a noticeable surge in pickups in the afternoon

### 4. Daily Pattern: Average Hourly Demand by Day of Week
This heatmap-style visualization shows typical demand patterns for each hour and day of the week.

In [ ]:
# ============================================================================
# Visualization 4: Average Hourly Demand by Day of Week
# ============================================================================

# Calculate average pickups by hour and day of week
hourly_pattern = cluster_data.groupby(['day_of_week', 'hour'])['pickups'].mean().reset_index()
hourly_pivot = hourly_pattern.pivot(index='hour', columns='day_of_week', values='pickups')

# Rename columns to day names
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
hourly_pivot.columns = [day_names[int(col)] for col in hourly_pivot.columns]

fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(hourly_pivot, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': 'Average Pickups'}, ax=ax, linewidths=0.5)

ax.set_title(f'Cluster {top_cluster}: Average Hourly Demand by Day of Week', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Hour of Day', fontsize=12)

plt.tight_layout()
plt.show()

print(f"📊 Observation: Can you identify rush hours? Are weekday patterns different from weekends?")

### 5. Pickups vs Dropoffs: Balance Analysis
Understanding the balance between pickups and dropoffs helps identify rebalancing needs.

In [ ]:
# ============================================================================
# Visualization 5: Pickups vs Dropoffs Over Time
# ============================================================================

# Focus on a 2-week period for clarity
sample_start = '2018-09-01'
sample_end = '2018-09-06'
cluster_sample = cluster_data[(cluster_data['datetime'] >= sample_start) & 
                               (cluster_data['datetime'] <= sample_end)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

# Top plot: Pickups and Dropoffs
ax1.plot(cluster_sample['datetime'], cluster_sample['pickups'], 
         linewidth=1.5, color='steelblue', label='Pickups', alpha=0.8)
ax1.plot(cluster_sample['datetime'], cluster_sample['dropoffs'], 
         linewidth=1.5, color='coral', label='Dropoffs', alpha=0.8)
ax1.fill_between(cluster_sample['datetime'], cluster_sample['pickups'], 
                  cluster_sample['dropoffs'], alpha=0.2, color='gray')

ax1.set_title(f'Cluster {top_cluster}: Pickups vs Dropoffs (Sep 1-6, 2018)', 
              fontsize=16, fontweight='bold', pad=20)
ax1.set_ylabel('Number of Trips', fontsize=12)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Bottom plot: Net Flow (Pickups - Dropoffs)
cluster_sample['net_flow'] = cluster_sample['pickups'] - cluster_sample['dropoffs']
colors = ['green' if x >= 0 else 'red' for x in cluster_sample['net_flow']]
ax2.bar(cluster_sample['datetime'], cluster_sample['net_flow'], 
        color=colors, alpha=0.6, width=0.03)
ax2.axhline(0, color='black', linewidth=1, linestyle='--')

ax2.set_title('Net Flow (Pickups - Dropoffs): Positive = Bikes Leaving, Negative = Bikes Arriving', 
              fontsize=12, style='italic')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Net Flow', fontsize=12)
ax2.grid(True, alpha=0.3)

# Format x-axis
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"📊 Observation: When does this cluster need bike rebalancing? Morning or evening?")

### 6. Monthly Aggregation: Seasonal Trends
This chart shows monthly totals to identify seasonal patterns (summer vs winter usage).

In [ ]:
# ============================================================================
# Visualization 6: Monthly Pickups and Dropoffs
# ============================================================================

# Aggregate by month
cluster_data['month_date'] = cluster_data['datetime'].dt.to_period('M').dt.to_timestamp()
monthly_demand = cluster_data.groupby('month_date').agg({
    'pickups': 'sum',
    'dropoffs': 'sum'
}).reset_index()

fig, ax = plt.subplots(figsize=(16, 6))

x = range(len(monthly_demand))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], monthly_demand['pickups'], 
               width, label='Pickups', color='steelblue', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], monthly_demand['dropoffs'], 
               width, label='Dropoffs', color='coral', alpha=0.8)

ax.set_title(f'Cluster {top_cluster}: Monthly Demand Throughout 2018', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Trips', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(monthly_demand['month_date'].dt.strftime('%b %Y'), rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"📊 Observation: Which months have highest demand? Is there a winter dip?")

### 7. Comparison: Top 5 Clusters
Compare demand patterns across multiple high-activity clusters to see if they behave similarly.

In [ ]:
# ============================================================================
# Visualization 7: Compare Top 5 Clusters
# ============================================================================

# Get top 5 clusters by total pickups
top_5_clusters = demand_hourly.groupby('cluster')['pickups'].sum().nlargest(5).index

fig, axes = plt.subplots(5, 1, figsize=(18, 15), sharex=True)

for idx, cluster_id in enumerate(top_5_clusters):
    cluster_subset = demand_hourly[demand_hourly['cluster'] == cluster_id].copy()
    cluster_subset['datetime'] = pd.to_datetime(cluster_subset['date']) + pd.to_timedelta(cluster_subset['hour'], unit='h')
    cluster_subset = cluster_subset.sort_values('datetime')
    
    # Focus on a 1-month period for clarity
    sample_start = '2018-07-01'
    sample_end = '2018-07-31'
    cluster_month = cluster_subset[(cluster_subset['datetime'] >= sample_start) & 
                                    (cluster_subset['datetime'] <= sample_end)]
    
    axes[idx].plot(cluster_month['datetime'], cluster_month['pickups'], 
                   linewidth=1, color=f'C{idx}', alpha=0.8)
    axes[idx].set_ylabel(f'Cluster {cluster_id}\nPickups', fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    
    # Add mean line
    mean_pickups = cluster_month['pickups'].mean()
    axes[idx].axhline(mean_pickups, color='red', linestyle='--', 
                      linewidth=1, alpha=0.5, label=f'Mean: {mean_pickups:.1f}')
    axes[idx].legend(loc='upper right', fontsize=8)

axes[0].set_title('Top 5 Clusters: Demand Comparison (July 2018)', 
                  fontsize=16, fontweight='bold', pad=20)
axes[-1].set_xlabel('Date', fontsize=12)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"📊 Observation: Do all high-activity clusters follow similar patterns?")

## Comparing average weekday vs weekend hourly patterns:

In [ ]:
# Comparar patrones horarios: Weekday vs Weekend
weekday_pattern = cluster_data[cluster_data['is_weekend'] == 0].groupby('hour')['pickups'].mean()
weekend_pattern = cluster_data[cluster_data['is_weekend'] == 1].groupby('hour')['pickups'].mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(weekday_pattern.index, weekday_pattern.values, marker='o', linewidth=2, 
        label='Weekday Average', color='steelblue')
ax.plot(weekend_pattern.index, weekend_pattern.values, marker='s', linewidth=2, 
        label='Weekend Average', color='coral')
ax.fill_between(weekday_pattern.index, weekday_pattern.values, weekend_pattern.values, 
                alpha=0.2, color='gray')
ax.set_title(f'Cluster {top_cluster}: Weekday vs Weekend Hourly Patterns', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Average Pickups', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Advanced Feature Engineering: Rush Hours and Demand Intensity

We'll create two types of temporal features:

### 1. Fixed Rush Hours (Domain Knowledge)
Based on typical urban commuting patterns:
- Morning rush: 7-9 AM (commute to work)
- Evening rush: 4-7 PM (commute home)
- Lunch time: 12-2 PM (midday activity)
- Night: 12-6 AM (low activity)

These capture **universal human behavior patterns** that apply across all clusters.

### 2. Dynamic Demand Intensity (Data-Driven)
Using percentiles within each cluster to identify:
- High demand hours (top 25% for that cluster)
- Very high demand hours (top 10% for that cluster)

This captures **cluster-specific patterns** (e.g., residential vs commercial areas have different peak times).

In [ ]:
# ============================================================================
# Feature Engineering: Rush Hours (Fixed - Domain Knowledge)
# ============================================================================

# Universal urban patterns
demand_hourly['is_morning_rush'] = demand_hourly['hour'].between(7, 9).astype(int)
demand_hourly['is_evening_rush'] = demand_hourly['hour'].between(16, 19).astype(int)
demand_hourly['is_lunch_time'] = demand_hourly['hour'].between(12, 14).astype(int)
demand_hourly['is_night'] = (demand_hourly['hour'] < 6).astype(int)

# Interaction with weekends (rush hours behave differently)
demand_hourly['weekday_morning_rush'] = (
    (demand_hourly['is_morning_rush'] == 1) & 
    (demand_hourly['is_weekend'] == 0)
).astype(int)

demand_hourly['weekday_evening_rush'] = (
    (demand_hourly['is_evening_rush'] == 1) & 
    (demand_hourly['is_weekend'] == 0)
).astype(int)

# Cyclic encoding for hour: Converts hour (0-23) into sin/cos components
# This preserves the circular nature of time (hour 23 is close to hour 0)
# Without this, models would think 11 PM is far from midnight, when they're actually adjacent
# The model will use both hour_sin and hour_cos to understand temporal patterns correctly
demand_hourly['hour_sin'] = np.sin(2 * np.pi * demand_hourly['hour'] / 24)
demand_hourly['hour_cos'] = np.cos(2 * np.pi * demand_hourly['hour'] / 24)

print("✓ Fixed rush hour features created")
print(f"  Morning rush observations: {demand_hourly['is_morning_rush'].sum():,}")
print(f"  Evening rush observations: {demand_hourly['is_evening_rush'].sum():,}")

In [ ]:
# ============================================================================
# Feature Engineering: Dynamic Demand Intensity (Data-Driven)
# ============================================================================

print("Creating dynamic demand intensity indicators (percentile-based)...")

# Calculate percentile rank within each cluster for PICKUPS
demand_hourly['pickups_percentile'] = demand_hourly.groupby('cluster')['pickups'].transform(
    lambda x: x.rank(pct=True)
)

# Calculate percentile rank within each cluster for DROPOFFS
demand_hourly['dropoffs_percentile'] = demand_hourly.groupby('cluster')['dropoffs'].transform(
    lambda x: x.rank(pct=True)
)

# Create binary indicators for high-demand PICKUPS
demand_hourly['is_high_demand'] = (demand_hourly['pickups_percentile'] > 0.75).astype(int)  # Top 25%
demand_hourly['is_very_high_demand'] = (demand_hourly['pickups_percentile'] > 0.90).astype(int)  # Top 10%
demand_hourly['is_low_demand'] = (demand_hourly['pickups_percentile'] < 0.25).astype(int)  # Bottom 25%

# Create binary indicators for high-demand DROPOFFS
demand_hourly['is_high_dropoff'] = (demand_hourly['dropoffs_percentile'] > 0.75).astype(int)  # Top 25%
demand_hourly['is_very_high_dropoff'] = (demand_hourly['dropoffs_percentile'] > 0.90).astype(int)  # Top 10%
demand_hourly['is_low_dropoff'] = (demand_hourly['dropoffs_percentile'] < 0.25).astype(int)  # Bottom 25%

# Interaction: High pickups + Low dropoffs = Critical rebalancing need
demand_hourly['critical_rebalancing_need'] = (
    (demand_hourly['is_high_demand'] == 1) & 
    (demand_hourly['is_low_dropoff'] == 1)
).astype(int)

print("✓ Dynamic demand intensity features created")
print(f"  Pickups - High demand: {demand_hourly['is_high_demand'].sum():,} ({demand_hourly['is_high_demand'].mean():.1%})")
print(f"  Dropoffs - High demand: {demand_hourly['is_high_dropoff'].sum():,} ({demand_hourly['is_high_dropoff'].mean():.1%})")
print(f"  Critical rebalancing need: {demand_hourly['critical_rebalancing_need'].sum():,} ({demand_hourly['critical_rebalancing_need'].mean():.1%})")

# Show example
example_cluster = demand_hourly.groupby('cluster')['pickups'].sum().idxmax()
example_data = demand_hourly[demand_hourly['cluster'] == example_cluster].groupby('hour').agg({
    'is_morning_rush': 'first',
    'is_evening_rush': 'first',
    'is_high_demand': 'mean',
    'is_high_dropoff': 'mean',
    'critical_rebalancing_need': 'mean',
    'pickups': 'mean',
    'dropoffs': 'mean'
}).round(2)

print(f"\nExample: Cluster {example_cluster} - Feature Summary by Hour")
print(example_data[['is_morning_rush', 'is_high_demand', 'is_high_dropoff', 'critical_rebalancing_need']])

In [ ]:
# ============================================================================
# Feature Engineering: Net Flow
# ============================================================================

# Calculate instantaneous net flow (pickups - dropoffs)
demand_hourly['net_flow'] = demand_hourly['pickups'] - demand_hourly['dropoffs']

# Sort by cluster and time
demand_hourly = demand_hourly.sort_values(['cluster', 'date', 'hour']).reset_index(drop=True)

# Cluster type: Source (bikes leave) vs Sink (bikes arrive)
cluster_avg_net = demand_hourly.groupby('cluster')['net_flow'].mean()
demand_hourly['cluster_is_source'] = demand_hourly['cluster'].map(
    lambda x: 1 if cluster_avg_net.get(x, 0) > 0 else 0
)

print("✓ Net flow features created")
print(f"  Mean net flow: {demand_hourly['net_flow'].mean():.2f}")
print(f"  Net flow std: {demand_hourly['net_flow'].std():.2f}")
print(f"  Max deficit (single hour): {demand_hourly['net_flow'].min():.0f} bikes")
print(f"  Max surplus (single hour): {demand_hourly['net_flow'].max():.0f} bikes")

demand_hourly[['cluster', 'hour', 'pickups', 'dropoffs', 'net_flow', 
               'cluster_is_source', 'is_morning_rush', 'is_high_demand']].head(24)

### Comparing Fixed vs Dynamic Rush Hour Detection

Let's visualize how fixed (domain knowledge) and dynamic (percentile-based) rush hours differ across clusters.

In [ ]:
# ============================================================================
# Visualization: Fixed vs Dynamic Rush Hours
# ============================================================================

# Compare 3 different cluster types
top_clusters = demand_hourly.groupby('cluster')['pickups'].sum().nlargest(3).index

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for idx, cluster_id in enumerate(top_clusters):
    cluster_subset = demand_hourly[demand_hourly['cluster'] == cluster_id]
    
    # Average pickups by hour
    hourly_avg = cluster_subset.groupby('hour').agg({
        'pickups': 'mean',
        'is_morning_rush': 'first',
        'is_evening_rush': 'first',
        'is_high_demand': 'mean'  # % of times this hour is high-demand
    })
    
    ax = axes[idx]
    
    # Plot average pickups
    ax.bar(hourly_avg.index, hourly_avg['pickups'], alpha=0.6, color='steelblue', label='Avg Pickups')
    
    # Highlight fixed rush hours (domain knowledge)
    morning_rush_hours = hourly_avg[hourly_avg['is_morning_rush'] == 1].index
    evening_rush_hours = hourly_avg[hourly_avg['is_evening_rush'] == 1].index
    
    for hour in morning_rush_hours:
        ax.axvspan(hour - 0.5, hour + 0.5, alpha=0.2, color='orange', 
                   label='Fixed Morning Rush' if hour == morning_rush_hours[0] else '')
    for hour in evening_rush_hours:
        ax.axvspan(hour - 0.5, hour + 0.5, alpha=0.2, color='red', 
                   label='Fixed Evening Rush' if hour == evening_rush_hours[0] else '')
    
    # Overlay dynamic high-demand indicator (line)
    ax2 = ax.twinx()
    ax2.plot(hourly_avg.index, hourly_avg['is_high_demand'] * 100, 
             color='green', linewidth=2.5, marker='o', label='% Days This Hour Was High-Demand')
    ax2.set_ylabel('% High Demand Hours', fontsize=10, color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    ax2.set_ylim(0, 100)
    
    ax.set_title(f'Cluster {cluster_id}: Fixed vs Dynamic Rush Hour Detection', 
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Avg Pickups', fontsize=10)
    ax.legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

axes[-1].set_xlabel('Hour of Day', fontsize=12)
axes[-1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

print("📊 Observation:")
print("  - Orange/Red zones: Fixed rush hours (same for all clusters)")
print("  - Green line: Dynamic high-demand hours (cluster-specific)")
print("  - Notice how some clusters have peaks outside traditional rush hours!")

In [ ]:
list(demand_hourly.columns)


## Final Dataset Preparation and Train/Validation/Test Split

Now that we have engineered all features, we need to:

1. **Select and organize features** for modeling
2. **Standardize continuous variables** (preserve model performance and interpretability)
3. **Handle missing values** from lag features (first days have no historical data)
4. **Split data chronologically** into train (70%) and test (30%) sets
   - Train: January - September (70% of year)
   - Test: October - December (30% of year)
   - **No shuffling** - time series must maintain temporal order

This final dataset will be ready for time series forecasting models.

In [ ]:
# ============================================================================
# Step 1: Organize Features by Type
# ============================================================================

print("Organizing features by type...")

# Target variables (what we want to predict)
target_vars = ['pickups', 'dropoffs']

# Identifiers (not used in modeling, but needed for tracking)
id_vars = ['date', 'cluster']

# Continuous features (need standardization)
continuous_features = [
    'avg_trip_duration',
    'pct_subscribers',
    'avg_age',
    'cluster_station_count',
    'cluster_center_lat',
    'cluster_center_lng',
    'cluster_total_trips',
    'pickups_lag_24h',
    'pickups_lag_168h',
    'pickups_rolling_24h',
    'dropoffs_lag_24h',
    'dropoffs_lag_168h',
    'dropoffs_rolling_24h',
    'hour_sin',
    'hour_cos',
    'net_flow',
    'pickups_percentile',
    'dropoffs_percentile'
]

# Categorical/Binary features (already encoded, no standardization needed)
categorical_features = [
    'hour',
    'month',
    'day_of_week',
    'day_of_month',
    'week_of_year',
    'is_weekend',
    'season_fall',
    'season_spring',
    'season_summer',
    'season_winter',
    'is_holiday',
    'is_special_event',
    'is_morning_rush',
    'is_evening_rush',
    'is_lunch_time',
    'is_night',
    'weekday_morning_rush',
    'weekday_evening_rush',
    'is_high_demand',
    'is_very_high_demand',
    'is_low_demand',
    'is_high_dropoff',
    'is_very_high_dropoff',
    'is_low_dropoff',
    'critical_rebalancing_need',
    'cluster_is_source'
]

# All feature columns (for modeling)
feature_cols = continuous_features + categorical_features

print(f"Feature organization:")
print(f"  - Target variables: {len(target_vars)}")
print(f"  - Continuous features: {len(continuous_features)}")
print(f"  - Categorical features: {len(categorical_features)}")
print(f"  - Total features for modeling: {len(feature_cols)}")

In [ ]:
# ============================================================================
# Step 2: Handle Missing Values (from lag features)
# ============================================================================

print("\nHandling missing values...")

# Check missing values
missing_summary = demand_hourly[feature_cols + target_vars].isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)

if len(missing_summary) > 0:
    print(f"Missing values found in {len(missing_summary)} columns:")
    print(missing_summary)
    
    # Strategy: Drop rows with missing lag features (first week of data)
    # This is acceptable because we have 365 days and only lose ~7 days
    initial_rows = len(demand_hourly)
    demand_hourly_clean = demand_hourly.dropna(subset=feature_cols + target_vars).copy()
    dropped_rows = initial_rows - len(demand_hourly_clean)
    
    print(f"\nDropped {dropped_rows:,} rows with missing values ({dropped_rows/initial_rows:.1%})")
    print(f"  Remaining: {len(demand_hourly_clean):,} observations")
else:
    demand_hourly_clean = demand_hourly.copy()
    print("No missing values found")

# Sort by date and hour (ensure chronological order)
demand_hourly_clean = demand_hourly_clean.sort_values(['date', 'hour', 'cluster']).reset_index(drop=True)

print(f"\nFinal dataset shape: {demand_hourly_clean.shape}")
print(f"Date range: {demand_hourly_clean['date'].min()} to {demand_hourly_clean['date'].max()}")

In [ ]:
# ============================================================================
# Step 3: Standardize Continuous Features
# ============================================================================

print("\nStandardizing continuous features...")

from sklearn.preprocessing import StandardScaler

# Create a copy for standardization
demand_final = demand_hourly_clean.copy()

# Initialize scaler
scaler = StandardScaler()

# Fit scaler on continuous features and transform
demand_final[continuous_features] = scaler.fit_transform(demand_final[continuous_features])

print(f"Standardized {len(continuous_features)} continuous features")
print(f"  Mean ≈ 0, Std ≈ 1 for all continuous features")

# Verify standardization
print("\nStandardization check (first 5 continuous features):")
for col in continuous_features[:5]:
    print(f"  {col}: mean={demand_final[col].mean():.4f}, std={demand_final[col].std():.4f}")

## Time Series Validation and Feature Selection

Before splitting our data and training models, we need to:

1. **Check stationarity** using the Augmented Dickey-Fuller (ADF) test
2. **Analyze autocorrelation** (ACF/PACF) to understand temporal dependencies
3. **Clean up features** - remove redundant or problematic features
4. **Validate data quality** - ensure no issues before modeling

These steps are critical for time series forecasting success.

In [ ]:
# ============================================================================
# Time Series Validation: Stationarity Check (ADF Test)
# ============================================================================

from statsmodels.tsa.stattools import adfuller

# Test for pickups (aggregate across all clusters)
pickups_series = demand_hourly.groupby("date")["pickups"].sum()

result_pickups = adfuller(pickups_series)
print("\n📊 Augmented Dickey-Fuller Test for PICKUPS:")
print(f"   ADF Statistic: {result_pickups[0]:.6f}")
print(f"   p-value: {result_pickups[1]:.6f}")
print(f"   Critical Values:")
for key, value in result_pickups[4].items():
    print(f"      {key}: {value:.3f}")

if result_pickups[1] < 0.05:
    print(f"RESULT: Series is STATIONARY (p-value < 0.05)")
else:
    print(f"RESULT: Series is NON-STATIONARY (p-value >= 0.05)")

# Test for dropoffs
dropoffs_series = demand_hourly.groupby("date")["dropoffs"].sum()

result_dropoffs = adfuller(dropoffs_series)
print("\n📊 Augmented Dickey-Fuller Test for DROPOFFS:")
print(f"   ADF Statistic: {result_dropoffs[0]:.6f}")
print(f"   p-value: {result_dropoffs[1]:.6f}")
print(f"   Critical Values:")
for key, value in result_dropoffs[4].items():
    print(f"      {key}: {value:.3f}")

if result_dropoffs[1] < 0.05:
    print(f"RESULT: Series is STATIONARY (p-value < 0.05)")
else:
    print(f"RESULT: Series is NON-STATIONARY (p-value >= 0.05)")

print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# Time Series Validation: Autocorrelation Analysis
# ============================================================================

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

print("Analyzing autocorrelation patterns...")

# Select a representative cluster for analysis
top_cluster = demand_hourly.groupby('cluster')['pickups'].sum().idxmax()
cluster_pickups = demand_hourly[demand_hourly['cluster'] == top_cluster].sort_values('date')['pickups']

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# ACF plot
plot_acf(cluster_pickups, lags=72, ax=axes[0])  # 72 hours = 3 days
axes[0].set_title(f'Autocorrelation Function (ACF) - Cluster {top_cluster}', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lag (hours)', fontsize=12)
axes[0].set_ylabel('Correlation', fontsize=12)

# PACF plot
plot_pacf(cluster_pickups, lags=72, ax=axes[1])
axes[1].set_title(f'Partial Autocorrelation Function (PACF) - Cluster {top_cluster}', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lag (hours)', fontsize=12)
axes[1].set_ylabel('Partial Correlation', fontsize=12)

plt.tight_layout()
plt.show()

### Time Series Analysis: Key Findings

#### 1. Stationarity Test Results

The Augmented Dickey-Fuller (ADF) test indicates that both pickups and dropoffs are **non-stationary** (p-value > 0.05). This is expected for real-world demand data, which typically exhibits:
- Seasonal trends (summer vs. winter usage)
- Weekly patterns (weekday vs. weekend)
- Daily cycles (rush hours)

**Implication:** For classical ARIMA models, differencing (d=1) would be required. However, modern machine learning models (Gradient Boosting, Neural Networks, Random Forest) can handle non-stationary data directly when provided with appropriate features.

#### 2. Autocorrelation Analysis (ACF/PACF)

The ACF and PACF plots reveal strong temporal patterns:

**ACF (Autocorrelation Function):**
- Strong correlation at **lag 24** (and multiples: 48, 72) confirms clear **daily seasonality**
- Gradual decay suggests the presence of trend and seasonal components
- Persistent patterns validate our choice of 24-hour and 168-hour lag features

**PACF (Partial Autocorrelation Function):**
- Significant spikes at **lags 1-3** suggest an AR(3) process
- Strong spike at **lag 24** indicates direct daily dependence
- Most other lags fall within confidence bands, suggesting our lag features capture the main temporal dependencies

#### 3. Modeling Strategy

Given these findings, our approach is well-suited:
- ✅ **Lag features** (24h, 168h, rolling averages) capture temporal dependencies
- ✅ **Seasonal features** (month, season dummies, day of week) handle seasonality
- ✅ **Rush hour indicators** capture intra-day patterns
- ✅ Modern ML models can learn from these features without requiring explicit differencing

For this project, we will proceed with **supervised learning models** (Gradient Boosting, Neural Networks) that leverage our engineered features, rather than classical ARIMA approaches. This allows us to incorporate external variables (holidays, weather, cluster characteristics) more naturally.

## 📖 Final Feature Dictionary

Before proceeding to model training, let's document all features in our final dataset. This will help ensure we understand what information we're providing to our models.

### 🆔 Identifier Columns (not used as features)
- **`date`**: Date of observation (YYYY-MM-DD)
- **`hour`**: Hour of day (0-23)
- **`cluster`**: Spatial cluster ID (0-24)

### 🎯 Target Variables
- **`pickups`**: Number of bike pickups in this cluster-hour
- **`dropoffs`**: Number of bike dropoffs in this cluster-hour

### ⏰ Temporal Features
- **`month`**: Month of year (1-12)
- **`day_of_week`**: Day of week (0=Monday, 6=Sunday)
- **`day_of_month`**: Day of month (1-31)
- **`week_of_year`**: ISO week number (1-53)
- **`is_weekend`**: Binary flag for Saturday/Sunday
- **`hour_sin`**, **`hour_cos`**: Cyclic encoding of hour (captures circular nature of time)

### 🌦️ Seasonal Features
- **`season_fall`**, **`season_spring`**, **`season_summer`**, **`season_winter`**: One-hot encoded seasons

### 📅 Special Day Indicators
- **`is_holiday`**: Binary flag for NYC public holidays
- **`is_special_event`**: Binary flag for major NYC events (marathons, parades, etc.)

### 🚴 Historical Demand (Lag Features)
- **`pickups_lag_24h`**: Pickups exactly 24 hours ago (same hour yesterday)
- **`pickups_lag_168h`**: Pickups exactly 168 hours ago (same hour last week)
- **`pickups_rolling_24h`**: Average pickups over past 24 hours
- **`dropoffs_lag_24h`**: Dropoffs exactly 24 hours ago
- **`dropoffs_lag_168h`**: Dropoffs exactly 168 hours ago
- **`dropoffs_rolling_24h`**: Average dropoffs over past 24 hours

### 🕐 Rush Hour Indicators (Fixed Time Windows)
- **`is_morning_rush`**: 7-9 AM (binary)
- **`is_evening_rush`**: 5-7 PM (binary)
- **`is_lunch_time`**: 12-2 PM (binary)
- **`is_night`**: 10 PM - 5 AM (binary)
- **`weekday_morning_rush`**: Morning rush on weekdays only
- **`weekday_evening_rush`**: Evening rush on weekdays only

### 📊 Dynamic Demand Intensity (Percentile-Based)
- **`is_high_demand`**: Pickups in top 25% for this cluster-hour combination
- **`is_very_high_demand`**: Pickups in top 10%
- **`is_low_demand`**: Pickups in bottom 25%
- **`is_high_dropoff`**: Dropoffs in top 25%
- **`is_very_high_dropoff`**: Dropoffs in top 10%
- **`is_low_dropoff`**: Dropoffs in bottom 25%
- **`critical_rebalancing_need`**: Both high pickups AND low dropoffs (bikes running out)

### 🔄 Net Flow Features (for Task 3: Bike Repositioning)
- **`net_flow`**: Instantaneous imbalance (pickups - dropoffs)
- **`cluster_is_source`**: Binary flag indicating if cluster typically has net outflow

### 🗺️ Cluster Characteristics (Static)
- **`cluster_station_count`**: Number of stations in this cluster
- **`cluster_center_lat`**, **`cluster_center_lng`**: Geographic centroid of cluster
- **`cluster_total_trips`**: Total historical trips in this cluster

### 👥 User Behavior Aggregates
- **`avg_trip_duration`**: Average trip duration (seconds) for trips starting in this cluster-hour
- **`pct_subscribers`**: Percentage of riders who are annual subscribers (vs. casual customers)
- **`avg_age`**: Average age of riders in this cluster-hour

---

### 🧹 Feature Cleanup Recommendations

After reviewing all features, here are some that could potentially be removed:

**✅ KEEP (Essential for prediction):**
- All lag features (24h, 168h, rolling) - capture temporal dependencies
- Temporal features (month, day_of_week, hour_sin/cos, is_weekend) - capture cycles
- Seasonal dummies - capture yearly patterns
- Rush hour indicators - capture intra-day patterns
- Cluster characteristics - capture spatial differences
- User behavior (avg_trip_duration, pct_subscribers, avg_age) - capture demand quality

**⚠️ CONSIDER REMOVING (Potential issues):**
1. **`day_of_month`** (1-31): Not very informative; month + day_of_week are more useful
2. **`week_of_year`** (1-53): Redundant with month + seasonal features
3. **`is_lunch_time`**: Less relevant for bike demand compared to morning/evening rush
4. **`is_night`**: Low activity period; model can learn this from hour_sin/cos
5. **Dynamic demand flags** (`is_high_demand`, `is_very_high_demand`, etc.): These are **derived from the target variable** and could cause data leakage. They're based on historical percentiles, which is okay, but they add complexity without much predictive power that lag features don't already provide.
6. **`critical_rebalancing_need`**: Derived from current hour's data; not useful for prediction

**🎯 RECOMMENDED ACTION:**
Remove: `day_of_month`, `week_of_year`, `is_lunch_time`, `is_night`, `is_high_demand`, `is_very_high_demand`, `is_low_demand`, `is_high_dropoff`, `is_very_high_dropoff`, `is_low_dropoff`, `critical_rebalancing_need`

This would reduce feature count from **46 to 35** while keeping the most predictive features and reducing overfitting risk.

In [ ]:
# ============================================================================
# Feature Cleanup: Remove Redundant and Problematic Features
# ============================================================================

# Current feature count
print(f"\n Current dataset: {demand_hourly.shape}")
print(f"   Columns: {len(demand_hourly.columns)}")

# Features to remove
features_to_remove = [
    "season",  # String version - we have season dummies (season_winter, etc.)
    "pickups_percentile",  # Intermediate calculation - not needed for modeling
    "dropoffs_percentile",  # Intermediate calculation - not needed for modeling
    "day_of_month",  # Redundant with month + day_of_week
    "week_of_year",  # Redundant with month + seasonal features
    "is_lunch_time",  # Less relevant for bike demand
    "is_night",  # Model can learn from hour_sin/cos
    "is_high_demand",  # Derived from target, redundant with lags
    "is_very_high_demand",  # Derived from target, redundant with lags
    "is_low_demand",  # Derived from target, redundant with lags
    "is_high_dropoff",  # Derived from target, redundant with lags
    "is_very_high_dropoff",  # Derived from target, redundant with lags
    "is_low_dropoff",  # Derived from target, redundant with lags
    "critical_rebalancing_need",  # Derived from current hour, not predictive
]

removed_count = 0
for feat in features_to_remove:
    if feat in demand_hourly.columns:
        demand_hourly = demand_hourly.drop(columns=[feat])
        removed_count += 1

print(f"\nRemoved {removed_count} features")
print(f"Updated dataset: {demand_hourly.shape}")
print(f"   Remaining columns: {len(demand_hourly.columns)}")

# Display remaining features
print(f"\nRemaining features ({len(demand_hourly.columns)} total):")
print(demand_hourly.columns.tolist())

In [ ]:
# ============================================================================
# Train/Validation/Test Split (Chronological)
# ============================================================================

print("Splitting data chronologically into Train/Validation/Test sets...")
print("=" * 80)

# Sort by date to ensure chronological order
demand_hourly = demand_hourly.sort_values(['date', 'hour', 'cluster']).reset_index(drop=True)

# Get unique dates
unique_dates = sorted(demand_hourly['date'].unique())
n_dates = len(unique_dates)

print(f"\n📅 Dataset date range:")
print(f"   Start: {unique_dates[0]}")
print(f"   End: {unique_dates[-1]}")
print(f"   Total days: {n_dates}")

# Calculate split points
# Train: 60% (Jan-Jul), Validation: 10% (Aug), Test: 30% (Sep-Dec)
train_end_idx = int(n_dates * 0.60)
val_end_idx = int(n_dates * 0.70)

train_end_date = unique_dates[train_end_idx - 1]
val_start_date = unique_dates[train_end_idx]
val_end_date = unique_dates[val_end_idx - 1]
test_start_date = unique_dates[val_end_idx]

print(f"\n📊 Split strategy (chronological, no shuffling):")
print(f"   Train:      {unique_dates[0]} to {train_end_date} ({train_end_idx} days, {train_end_idx/n_dates:.1%})")
print(f"   Validation: {val_start_date} to {val_end_date} ({val_end_idx - train_end_idx} days, {(val_end_idx - train_end_idx)/n_dates:.1%})")
print(f"   Test:       {test_start_date} to {unique_dates[-1]} ({n_dates - val_end_idx} days, {(n_dates - val_end_idx)/n_dates:.1%})")

# Split data
train_data = demand_hourly[demand_hourly['date'] <= train_end_date].copy()
val_data = demand_hourly[(demand_hourly['date'] >= val_start_date) & (demand_hourly['date'] <= val_end_date)].copy()
test_data = demand_hourly[demand_hourly['date'] >= test_start_date].copy()

print(f"\n✅ Data split completed:")
print(f"   Train set: {len(train_data):,} observations ({len(train_data)/len(demand_hourly):.1%})")
print(f"   Validation set: {len(val_data):,} observations ({len(val_data)/len(demand_hourly):.1%})")
print(f"   Test set: {len(test_data):,} observations ({len(test_data)/len(demand_hourly):.1%})")

# Define feature columns and target variables
id_cols = ['date', 'cluster']
target_cols = ['pickups', 'dropoffs']
feature_cols = [col for col in demand_hourly.columns if col not in id_cols + target_cols]

print(f"\n📋 Feature organization:")
print(f"   ID columns: {id_cols}")
print(f"   Target variables: {target_cols}")
print(f"   Feature count: {len(feature_cols)}")

# Verify no data leakage
print(f"\n🔍 Verifying chronological split (no data leakage):")
print(f"   Train max date: {train_data['date'].max()}")
print(f"   Val min date: {val_data['date'].min()}")
print(f"   Val max date: {val_data['date'].max()}")
print(f"   Test min date: {test_data['date'].min()}")
print(f"   Test max date: {test_data['date'].max()}")

print("\n" + "=" * 80)
print("✅ Ready for model training!")

## ✅ Data Preparation Complete

### Summary of Final Dataset

We have successfully prepared our data for model training with the following characteristics:

**Dataset Structure:**
- **Temporal granularity**: Hourly observations
- **Spatial granularity**: 25 clusters of bike stations
- **Time period**: Full year 2018 (365 days)
- **Total observations**: ~219,000 cluster-hour combinations

**Data Split Strategy:**
- **Train set (60%)**: January - July → Used for model training
- **Validation set (10%)**: August → Used for hyperparameter tuning
- **Test set (30%)**: September - December → Final evaluation (simulates real-world deployment)

**Why chronological split?**
Time series data requires chronological splitting to prevent data leakage. We cannot use future information to predict the past. This split simulates a realistic scenario where we train on historical data and predict future demand.

**Feature Categories:**
1. ⏰ **Temporal**: hour, month, day_of_week, is_weekend, seasons
2. 📊 **Historical demand**: 6 lag features (24h, 168h, rolling averages)
3. 🕐 **Rush hours**: morning/evening rush indicators
4. 📅 **Special days**: holidays and events
5. 🗺️ **Cluster characteristics**: station count, location, total trips
6. 👥 **User behavior**: trip duration, subscriber %, average age
7. 🔄 **Net flow**: instantaneous imbalance and cluster type
8. 🔢 **Cyclic encoding**: hour_sin, hour_cos

**Target Variables:**
- `pickups`: Number of bikes picked up (departures)
- `dropoffs`: Number of bikes dropped off (arrivals)

---

### Next Steps

With our clean, feature-rich dataset split chronologically, we're ready to:

1. **Baseline Models**: Start with simple models (Linear Regression, Ridge, Lasso)
2. **Advanced Models**: Gradient Boosting (XGBoost, LightGBM), Random Forest
3. **Neural Networks**: LSTM or Dense networks for time series
4. **Evaluation**: Compare models using MAE, RMSE, and R² on validation set
5. **Task 3**: Use predictions to compute bike repositioning requirements

Let's build some models! 🚀

# Model Training and Evaluation

## Task 2: Demand Prediction Models

Our goal is to build models that can predict bike demand (pickups and dropoffs) for the next 24 hours at the cluster level. We'll train separate models for pickups and dropoffs, then compare multiple algorithms to find the best performer.

### Modeling Strategy

1. Start with simple baseline models (Linear, Ridge, Lasso)
2. Progress to ensemble methods (Random Forest, Gradient Boosting)
3. Evaluate using validation set
4. Final evaluation on test set (Nov-Dec 2018)

### Evaluation Metrics

- **MAE (Mean Absolute Error)**: Average prediction error in number of bikes
- **RMSE (Root Mean Squared Error)**: Penalizes large errors more heavily
- **R² Score**: Proportion of variance explained by the model

In [ ]:
# ============================================================================
# Prepare Data for Modeling
# ============================================================================

# Define columns
id_cols = ['date', 'cluster']
target_cols = ['pickups', 'dropoffs']
feature_cols = [col for col in demand_hourly.columns 
                if col not in id_cols + target_cols]

print(f"\nFeature count: {len(feature_cols)}")
print(f"Target variables: {target_cols}")

# Separate features and targets for each split
X_train = train_data[feature_cols].copy()
y_train_pickups = train_data['pickups'].copy()
y_train_dropoffs = train_data['dropoffs'].copy()

X_val = val_data[feature_cols].copy()
y_val_pickups = val_data['pickups'].copy()
y_val_dropoffs = val_data['dropoffs'].copy()

X_test = test_data[feature_cols].copy()
y_test_pickups = test_data['pickups'].copy()
y_test_dropoffs = test_data['dropoffs'].copy()

# Check for missing values in features
print(f"\nMissing values in train features: {X_train.isnull().sum().sum()}")
print(f"Missing values in val features: {X_val.isnull().sum().sum()}")
print(f"Missing values in test features: {X_test.isnull().sum().sum()}")

# Handle missing values in lag features (first days of dataset)
if X_train.isnull().sum().sum() > 0:
    print("\nHandling missing values in lag features...")
    # Option 1: Drop rows with NaN (loses some data)
    # Option 2: Fill with 0 (assumes no demand)
    # Option 3: Forward fill (use previous value)
    
    # We'll use forward fill, then fill remaining with 0
    X_train = X_train.fillna(method='ffill').fillna(0)
    X_val = X_val.fillna(method='ffill').fillna(0)
    X_test = X_test.fillna(method='ffill').fillna(0)
    
    print("Missing values filled")

print(f"\nFinal dataset shapes:")
print(f"  X_train: {X_train.shape}, y_train: {y_train_pickups.shape}")
print(f"  X_val: {X_val.shape}, y_val: {y_val_pickups.shape}")
print(f"  X_test: {X_test.shape}, y_test: {y_test_pickups.shape}")

print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# Baseline Models: Linear Regression
# ============================================================================

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Training baseline models...")
print("=" * 80)

# Store results
results = []

# Models to test
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Ridge (alpha=10.0)': Ridge(alpha=10.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1, max_iter=5000),
    'Lasso (alpha=1.0)': Lasso(alpha=1.0, max_iter=5000),
}

# Train models for PICKUPS
print("\n" + "=" * 40)
print("PREDICTING PICKUPS")
print("=" * 40)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train
    model.fit(X_train, y_train_pickups)
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    mae = mean_absolute_error(y_val_pickups, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val_pickups, y_pred))
    r2 = r2_score(y_val_pickups, y_pred)
    
    print(f"  MAE: {mae:.2f} bikes")
    print(f"  RMSE: {rmse:.2f} bikes")
    print(f"  R²: {r2:.4f}")
    
    results.append({
        'Model': name,
        'Target': 'Pickups',
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

# Train models for DROPOFFS
print("\n" + "=" * 40)
print("PREDICTING DROPOFFS")
print("=" * 40)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train
    model.fit(X_train, y_train_dropoffs)
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    mae = mean_absolute_error(y_val_dropoffs, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val_dropoffs, y_pred))
    r2 = r2_score(y_val_dropoffs, y_pred)
    
    print(f"  MAE: {mae:.2f} bikes")
    print(f"  RMSE: {rmse:.2f} bikes")
    print(f"  R²: {r2:.4f}")
    
    results.append({
        'Model': name,
        'Target': 'Dropoffs',
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

# Summary
import pandas as pd
results_df = pd.DataFrame(results)
print("\n" + "=" * 80)
print("BASELINE MODELS SUMMARY")
print("=" * 80)
print(results_df.to_string(index=False))

### Baseline Model Results: Analysis

Our baseline linear models achieved strong performance:

**Key Findings:**
- **R² Score: 0.87** - The models explain 87% of demand variance, indicating our features capture the main patterns effectively
- **MAE: ~26 bikes/hour** - On average, predictions are off by 26 bikes per cluster per hour
- **Consistent performance** - Linear Regression, Ridge, and Lasso produced nearly identical results, suggesting:
  - Features are well-scaled and balanced
  - No significant multicollinearity issues
  - Minimal overfitting risk

**Model Comparison:**
- **Lasso (alpha=0.1)** performed slightly better (MAE: 26.17) due to mild feature selection
- **Ridge and Linear Regression** were essentially equivalent, indicating regularization isn't critical
- **Pickups and Dropoffs** are equally predictable (same MAE and R²)

**Interpretation:**
An MAE of 26 bikes means that for a cluster with typical demand of 50-100 bikes/hour, our predictions are within 25-50% accuracy. This is reasonable for a linear baseline but leaves room for improvement.

---

## Advanced Models: Capturing Non-Linear Patterns

Linear models assume relationships between features and demand are linear. However, bike demand likely has non-linear patterns:
- Rush hour effects may interact with weather and seasons
- Cluster characteristics may have threshold effects
- Lag features may have diminishing returns

We'll now test ensemble methods that can capture these complex interactions:

**Random Forest:**
- Builds multiple decision trees on random subsets of data
- Captures non-linear relationships and feature interactions
- Robust to outliers and doesn't require feature scaling

**Gradient Boosting:**
- Builds trees sequentially, each correcting errors from previous ones
- Often achieves best performance on tabular data
- More sensitive to hyperparameters but very powerful

**Expected Improvement:**
We aim to reduce MAE to 20-23 bikes (15-20% improvement) and increase R² to 0.90+.

In [ ]:
# ============================================================================
# Advanced Models: Random Forest and Gradient Boosting
# ============================================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import time

print("Training advanced models...")
print("=" * 80)

# Models to test
advanced_models = {
    'Random Forest (n=100)': RandomForestRegressor(
        n_estimators=100, 
        max_depth=15,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ),
    'Random Forest (n=200)': RandomForestRegressor(
        n_estimators=200, 
        max_depth=20,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting (lr=0.1)': GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ),
    'Gradient Boosting (lr=0.05)': GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    ),
}

advanced_results = []

# Train models for PICKUPS
print("\n" + "=" * 40)
print("PREDICTING PICKUPS")
print("=" * 40)

for name, model in advanced_models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    
    # Train
    model.fit(X_train, y_train_pickups)
    train_time = time.time() - start_time
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    mae = mean_absolute_error(y_val_pickups, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val_pickups, y_pred))
    r2 = r2_score(y_val_pickups, y_pred)
    
    print(f"  Training time: {train_time:.1f}s")
    print(f"  MAE: {mae:.2f} bikes")
    print(f"  RMSE: {rmse:.2f} bikes")
    print(f"  R²: {r2:.4f}")
    
    advanced_results.append({
        'Model': name,
        'Target': 'Pickups',
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Train_Time': train_time
    })

# Train models for DROPOFFS
print("\n" + "=" * 40)
print("PREDICTING DROPOFFS")
print("=" * 40)

for name, model in advanced_models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    
    # Train
    model.fit(X_train, y_train_dropoffs)
    train_time = time.time() - start_time
    
    # Predict on validation set
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    mae = mean_absolute_error(y_val_dropoffs, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val_dropoffs, y_pred))
    r2 = r2_score(y_val_dropoffs, y_pred)
    
    print(f"  Training time: {train_time:.1f}s")
    print(f"  MAE: {mae:.2f} bikes")
    print(f"  RMSE: {rmse:.2f} bikes")
    print(f"  R²: {r2:.4f}")
    
    advanced_results.append({
        'Model': name,
        'Target': 'Dropoffs',
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Train_Time': train_time
    })

# Summary
advanced_results_df = pd.DataFrame(advanced_results)
print("\n" + "=" * 80)
print("ADVANCED MODELS SUMMARY")
print("=" * 80)
print(advanced_results_df.to_string(index=False))

# Compare with best baseline
print("\n" + "=" * 80)
print("COMPARISON: Best Baseline vs Best Advanced")
print("=" * 80)
print("\nBest Baseline (Lasso alpha=0.1):")
print("  Pickups  - MAE: 26.99, R²: 0.8713")
print("  Dropoffs - MAE: 26.99, R²: 0.8718")
print("\nBest Advanced (will show after running):")
best_pickups = advanced_results_df[advanced_results_df['Target']=='Pickups'].nsmallest(1, 'MAE')
best_dropoffs = advanced_results_df[advanced_results_df['Target']=='Dropoffs'].nsmallest(1, 'MAE')
print(f"  Pickups  - {best_pickups['Model'].values[0]}")
print(f"             MAE: {best_pickups['MAE'].values[0]:.2f}, R²: {best_pickups['R2'].values[0]:.4f}")
print(f"  Dropoffs - {best_dropoffs['Model'].values[0]}")
print(f"             MAE: {best_dropoffs['MAE'].values[0]:.2f}, R²: {best_dropoffs['R2'].values[0]:.4f}")

## Final Model Evaluation on Test Set

We've identified Random Forest (n=200) as our best model. Now we'll:

1. Retrain on combined train+validation data (Jan-Aug)
2. Evaluate on test set (Sep-Dec) to simulate real-world performance
3. Analyze prediction quality across different clusters and time periods
4. Generate predictions for Task 3 (bike repositioning)

This final evaluation will tell us how well our model generalizes to completely unseen future data.

In [ ]:
# ============================================================================
# Final Model: Train on Train+Val, Evaluate on Test
# ============================================================================

print("Training final model on combined train+validation data...")
print("=" * 80)

# Combine train and validation sets
X_train_val = pd.concat([X_train, X_val], ignore_index=True)
y_train_val_pickups = pd.concat([y_train_pickups, y_val_pickups], ignore_index=True)
y_train_val_dropoffs = pd.concat([y_train_dropoffs, y_val_dropoffs], ignore_index=True)

print(f"\nCombined training data: {X_train_val.shape}")
print(f"Test data: {X_test.shape}")

# Initialize best model
final_model_pickups = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

final_model_dropoffs = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

# Train on combined data
print("\nTraining model for PICKUPS...")
start_time = time.time()
final_model_pickups.fit(X_train_val, y_train_val_pickups)
print(f"  Training time: {time.time() - start_time:.1f}s")

print("\nTraining model for DROPOFFS...")
start_time = time.time()
final_model_dropoffs.fit(X_train_val, y_train_val_dropoffs)
print(f"  Training time: {time.time() - start_time:.1f}s")

# Predict on test set
print("\nGenerating predictions on test set...")
y_pred_pickups = final_model_pickups.predict(X_test)
y_pred_dropoffs = final_model_dropoffs.predict(X_test)

# Ensure non-negative predictions
y_pred_pickups = np.maximum(0, y_pred_pickups)
y_pred_dropoffs = np.maximum(0, y_pred_dropoffs)

# Calculate final metrics
print("\n" + "=" * 80)
print("FINAL TEST SET PERFORMANCE")
print("=" * 80)

print("\nPICKUPS:")
mae_pickups = mean_absolute_error(y_test_pickups, y_pred_pickups)
rmse_pickups = np.sqrt(mean_squared_error(y_test_pickups, y_pred_pickups))
r2_pickups = r2_score(y_test_pickups, y_pred_pickups)
print(f"  MAE:  {mae_pickups:.2f} bikes")
print(f"  RMSE: {rmse_pickups:.2f} bikes")
print(f"  R²:   {r2_pickups:.4f}")

print("\nDROPOFFS:")
mae_dropoffs = mean_absolute_error(y_test_dropoffs, y_pred_dropoffs)
rmse_dropoffs = np.sqrt(mean_squared_error(y_test_dropoffs, y_pred_dropoffs))
r2_dropoffs = r2_score(y_test_dropoffs, y_pred_dropoffs)
print(f"  MAE:  {mae_dropoffs:.2f} bikes")
print(f"  RMSE: {rmse_dropoffs:.2f} bikes")
print(f"  R²:   {r2_dropoffs:.4f}")

# Create predictions dataframe for Task 3
predictions_df = test_data[['date', 'cluster', 'hour']].copy()
predictions_df['actual_pickups'] = y_test_pickups.values
predictions_df['predicted_pickups'] = y_pred_pickups
predictions_df['actual_dropoffs'] = y_test_dropoffs.values
predictions_df['predicted_dropoffs'] = y_pred_dropoffs

print("\n" + "=" * 80)
print("Predictions saved for Task 3 analysis")
print(f"Total predictions: {len(predictions_df):,} cluster-hour observations")
print("=" * 80)

# Preview predictions
predictions_df.head(24)

## Task 3: Bike Repositioning Strategy

### Problem Statement

Every night, Citi Bike manually repositions bikes across clusters to ensure demand can be met the next day. Our goal is to determine the optimal number of bikes each cluster needs at the start of the day (midnight) to avoid shortages.

### Methodology

For each cluster and each day in the test period:

1. **Calculate hourly net flow**: `net_flow[h] = pickups[h] - dropoffs[h]`
   - Positive = bikes leaving (deficit)
   - Negative = bikes arriving (surplus)

2. **Compute cumulative flow**: `cumulative[h] = sum(net_flow[0:h])`
   - Tracks running balance throughout the day
   - Minimum value indicates maximum deficit

3. **Determine bikes needed**: `bikes_required = max(0, -min(cumulative))`
   - If cumulative never goes negative, no bikes needed
   - If it reaches -50, we need 50 bikes at start of day

### Business Impact

This analysis helps operations teams:
- Optimize nightly bike distribution
- Reduce bike shortages during peak hours
- Minimize repositioning costs by focusing on high-need clusters

In [ ]:
# ============================================================================
# Task 3: Calculate Daily Bike Repositioning Requirements
# ============================================================================

print("Calculating bike repositioning requirements...")
print("=" * 80)

# Calculate net flow for predictions
predictions_df['predicted_net_flow'] = (
    predictions_df['predicted_pickups'] - predictions_df['predicted_dropoffs']
)
predictions_df['actual_net_flow'] = (
    predictions_df['actual_pickups'] - predictions_df['actual_dropoffs']
)

# Sort by cluster, date, hour
predictions_df = predictions_df.sort_values(['cluster', 'date', 'hour']).reset_index(drop=True)

# Calculate cumulative net flow per day (predicted)
predictions_df['predicted_cumulative'] = (
    predictions_df.groupby(['cluster', 'date'])['predicted_net_flow'].cumsum()
)

# Calculate cumulative net flow per day (actual)
predictions_df['actual_cumulative'] = (
    predictions_df.groupby(['cluster', 'date'])['actual_net_flow'].cumsum()
)

# For each cluster-day, find the minimum cumulative (maximum deficit)
daily_requirements = predictions_df.groupby(['cluster', 'date']).agg({
    'predicted_cumulative': 'min',
    'actual_cumulative': 'min',
    'predicted_pickups': 'sum',
    'predicted_dropoffs': 'sum',
    'actual_pickups': 'sum',
    'actual_dropoffs': 'sum'
}).reset_index()

# Calculate bikes needed (if cumulative goes negative, we need that many bikes)
daily_requirements['predicted_bikes_needed'] = daily_requirements['predicted_cumulative'].apply(
    lambda x: max(0, -x)
)
daily_requirements['actual_bikes_needed'] = daily_requirements['actual_cumulative'].apply(
    lambda x: max(0, -x)
)

# Calculate prediction error
daily_requirements['bikes_error'] = (
    daily_requirements['predicted_bikes_needed'] - daily_requirements['actual_bikes_needed']
)

print("\n" + "=" * 80)
print("REPOSITIONING REQUIREMENTS SUMMARY")
print("=" * 80)

print(f"\nTotal cluster-days analyzed: {len(daily_requirements):,}")

print("\nPREDICTED REQUIREMENTS:")
print(f"  Average bikes needed per cluster-day: {daily_requirements['predicted_bikes_needed'].mean():.1f}")
print(f"  Median: {daily_requirements['predicted_bikes_needed'].median():.1f}")
print(f"  Max: {daily_requirements['predicted_bikes_needed'].max():.0f}")
print(f"  Days with no bikes needed: {(daily_requirements['predicted_bikes_needed'] == 0).sum():,}")

print("\nACTUAL REQUIREMENTS:")
print(f"  Average bikes needed per cluster-day: {daily_requirements['actual_bikes_needed'].mean():.1f}")
print(f"  Median: {daily_requirements['actual_bikes_needed'].median():.1f}")
print(f"  Max: {daily_requirements['actual_bikes_needed'].max():.0f}")
print(f"  Days with no bikes needed: {(daily_requirements['actual_bikes_needed'] == 0).sum():,}")

print("\nPREDICTION ACCURACY:")
print(f"  Mean Absolute Error: {abs(daily_requirements['bikes_error']).mean():.1f} bikes")
print(f"  Median Absolute Error: {abs(daily_requirements['bikes_error']).median():.1f} bikes")

# Identify critical clusters (highest average requirements)
cluster_avg_needs = daily_requirements.groupby('cluster').agg({
    'predicted_bikes_needed': 'mean',
    'actual_bikes_needed': 'mean'
}).round(1)
cluster_avg_needs.columns = ['Predicted Avg', 'Actual Avg']
cluster_avg_needs = cluster_avg_needs.sort_values('Actual Avg', ascending=False)

print("\n" + "=" * 80)
print("TOP 10 CLUSTERS BY REPOSITIONING NEEDS")
print("=" * 80)
print(cluster_avg_needs.head(10))

print("\n" + "=" * 80)
print("BOTTOM 10 CLUSTERS (Least repositioning needed)")
print("=" * 80)
print(cluster_avg_needs.tail(10))

# Save for visualization
daily_requirements.head(10)

In [ ]:
# ============================================================================
# Task 3 Visualizations
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

print("Creating Task 3 visualizations...")
print("=" * 80)

# Set style
sns.set_style("whitegrid")
fig = plt.figure(figsize=(16, 12))

# 1. Predicted vs Actual bikes needed (scatter)
ax1 = plt.subplot(2, 3, 1)
plt.scatter(daily_requirements['actual_bikes_needed'], 
           daily_requirements['predicted_bikes_needed'],
           alpha=0.3, s=10)
max_val = max(daily_requirements['actual_bikes_needed'].max(),
              daily_requirements['predicted_bikes_needed'].max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')
plt.xlabel('Actual Bikes Needed')
plt.ylabel('Predicted Bikes Needed')
plt.title('Prediction Accuracy: Bikes Required per Day')
plt.legend()

# 2. Top 10 clusters comparison
ax2 = plt.subplot(2, 3, 2)
top_10 = cluster_avg_needs.head(10)
x = range(len(top_10))
width = 0.35
plt.bar([i - width/2 for i in x], top_10['Actual Avg'], width, label='Actual', alpha=0.8)
plt.bar([i + width/2 for i in x], top_10['Predicted Avg'], width, label='Predicted', alpha=0.8)
plt.xlabel('Cluster ID')
plt.ylabel('Average Bikes Needed')
plt.title('Top 10 Clusters: Repositioning Requirements')
plt.xticks(x, top_10.index)
plt.legend()

# 3. Distribution of bikes needed
ax3 = plt.subplot(2, 3, 3)
plt.hist(daily_requirements['actual_bikes_needed'], bins=50, alpha=0.5, label='Actual', edgecolor='black')
plt.hist(daily_requirements['predicted_bikes_needed'], bins=50, alpha=0.5, label='Predicted', edgecolor='black')
plt.xlabel('Bikes Needed')
plt.ylabel('Frequency (cluster-days)')
plt.title('Distribution of Daily Repositioning Needs')
plt.legend()
plt.xlim(0, 400)

# 4. Time series for critical cluster (Cluster 4)
ax4 = plt.subplot(2, 3, 4)
cluster_4_data = daily_requirements[daily_requirements['cluster'] == 4].sort_values('date')
plt.plot(cluster_4_data['date'], cluster_4_data['actual_bikes_needed'], 
         label='Actual', linewidth=2, alpha=0.7)
plt.plot(cluster_4_data['date'], cluster_4_data['predicted_bikes_needed'], 
         label='Predicted', linewidth=2, alpha=0.7)
plt.xlabel('Date')
plt.ylabel('Bikes Needed')
plt.title('Cluster 4 (Highest Demand): Daily Requirements')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

# 5. Prediction error by cluster
ax5 = plt.subplot(2, 3, 5)
cluster_mae = daily_requirements.groupby('cluster').apply(
    lambda x: abs(x['bikes_error']).mean()
).sort_values(ascending=False)
plt.bar(range(len(cluster_mae)), cluster_mae.values)
plt.xlabel('Cluster (sorted by error)')
plt.ylabel('Mean Absolute Error (bikes)')
plt.title('Prediction Error by Cluster')
plt.axhline(y=21.1, color='r', linestyle='--', label='Overall MAE')
plt.legend()

# 6. Cumulative bikes repositioned over time
ax6 = plt.subplot(2, 3, 6)
daily_total = daily_requirements.groupby('date').agg({
    'predicted_bikes_needed': 'sum',
    'actual_bikes_needed': 'sum'
}).sort_index()
plt.plot(daily_total.index, daily_total['actual_bikes_needed'].cumsum(), 
         label='Actual (cumulative)', linewidth=2)
plt.plot(daily_total.index, daily_total['predicted_bikes_needed'].cumsum(), 
         label='Predicted (cumulative)', linewidth=2)
plt.xlabel('Date')
plt.ylabel('Cumulative Bikes Repositioned')
plt.title('Total Repositioning Effort Over Time')
plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print("\nVisualizations complete!")
print("=" * 80)

In [ ]:
# ============================================================================
# Detailed Prediction Analysis: Cluster 2 - Weekday vs Weekend
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("Analyzing hourly predictions for Cluster 2...")
print("=" * 80)

# Filter for Cluster 2 in December
cluster_2_dec = predictions_df[
    (predictions_df['cluster'] == 2) & 
    (predictions_df['date'].dt.month == 12)
].copy()

# Add day of week
cluster_2_dec['day_of_week'] = cluster_2_dec['date'].dt.dayofweek
cluster_2_dec['day_name'] = cluster_2_dec['date'].dt.day_name()

# Find a Wednesday (day_of_week = 2) and Saturday (day_of_week = 5)
wednesdays = cluster_2_dec[cluster_2_dec['day_of_week'] == 2]['date'].unique()
saturdays = cluster_2_dec[cluster_2_dec['day_of_week'] == 5]['date'].unique()

# Pick random dates
np.random.seed(42)
wednesday_date = pd.Timestamp(np.random.choice(wednesdays))
saturday_date = pd.Timestamp(np.random.choice(saturdays))

print(f"\nSelected dates for Cluster 2:")
print(f"  Wednesday: {wednesday_date.date()}")
print(f"  Saturday: {saturday_date.date()}")

# Get data for these specific days
wed_data = cluster_2_dec[cluster_2_dec['date'] == wednesday_date].sort_values('hour')
sat_data = cluster_2_dec[cluster_2_dec['date'] == saturday_date].sort_values('hour')

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Cluster 2: Hourly Demand Prediction - Weekday vs Weekend', 
             fontsize=16, fontweight='bold')

# Wednesday - Pickups
ax1 = axes[0, 0]
ax1.plot(wed_data['hour'], wed_data['actual_pickups'], 
         'o-', linewidth=2, markersize=6, label='Actual', color='#2E86AB')
ax1.plot(wed_data['hour'], wed_data['predicted_pickups'], 
         's--', linewidth=2, markersize=6, label='Predicted', color='#A23B72')
ax1.set_xlabel('Hour of Day', fontsize=11)
ax1.set_ylabel('Number of Pickups', fontsize=11)
ax1.set_title(f'Wednesday ({wednesday_date.date()}) - Pickups', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(0, 24, 2))

# Calculate metrics for Wednesday pickups
wed_mae_pickups = abs(wed_data['actual_pickups'] - wed_data['predicted_pickups']).mean()
ax1.text(0.02, 0.98, f'MAE: {wed_mae_pickups:.1f} bikes', 
         transform=ax1.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Wednesday - Dropoffs
ax2 = axes[0, 1]
ax2.plot(wed_data['hour'], wed_data['actual_dropoffs'], 
         'o-', linewidth=2, markersize=6, label='Actual', color='#2E86AB')
ax2.plot(wed_data['hour'], wed_data['predicted_dropoffs'], 
         's--', linewidth=2, markersize=6, label='Predicted', color='#A23B72')
ax2.set_xlabel('Hour of Day', fontsize=11)
ax2.set_ylabel('Number of Dropoffs', fontsize=11)
ax2.set_title(f'Wednesday ({wednesday_date.date()}) - Dropoffs', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 24, 2))

# Calculate metrics for Wednesday dropoffs
wed_mae_dropoffs = abs(wed_data['actual_dropoffs'] - wed_data['predicted_dropoffs']).mean()
ax2.text(0.02, 0.98, f'MAE: {wed_mae_dropoffs:.1f} bikes', 
         transform=ax2.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Saturday - Pickups
ax3 = axes[1, 0]
ax3.plot(sat_data['hour'], sat_data['actual_pickups'], 
         'o-', linewidth=2, markersize=6, label='Actual', color='#2E86AB')
ax3.plot(sat_data['hour'], sat_data['predicted_pickups'], 
         's--', linewidth=2, markersize=6, label='Predicted', color='#A23B72')
ax3.set_xlabel('Hour of Day', fontsize=11)
ax3.set_ylabel('Number of Pickups', fontsize=11)
ax3.set_title(f'Saturday ({saturday_date.date()}) - Pickups', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xticks(range(0, 24, 2))

# Calculate metrics for Saturday pickups
sat_mae_pickups = abs(sat_data['actual_pickups'] - sat_data['predicted_pickups']).mean()
ax3.text(0.02, 0.98, f'MAE: {sat_mae_pickups:.1f} bikes', 
         transform=ax3.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Saturday - Dropoffs
ax4 = axes[1, 1]
ax4.plot(sat_data['hour'], sat_data['actual_dropoffs'], 
         'o-', linewidth=2, markersize=6, label='Actual', color='#2E86AB')
ax4.plot(sat_data['hour'], sat_data['predicted_dropoffs'], 
         's--', linewidth=2, markersize=6, label='Predicted', color='#A23B72')
ax4.set_xlabel('Hour of Day', fontsize=11)
ax4.set_ylabel('Number of Dropoffs', fontsize=11)
ax4.set_title(f'Saturday ({saturday_date.date()}) - Dropoffs', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)
ax4.set_xticks(range(0, 24, 2))

# Calculate metrics for Saturday dropoffs
sat_mae_dropoffs = abs(sat_data['actual_dropoffs'] - sat_data['predicted_dropoffs']).mean()
ax4.text(0.02, 0.98, f'MAE: {sat_mae_dropoffs:.1f} bikes', 
         transform=ax4.transAxes, fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "=" * 80)
print("PREDICTION QUALITY SUMMARY")
print("=" * 80)

print(f"\nWEDNESDAY ({wednesday_date.date()}):")
print(f"  Pickups  - MAE: {wed_mae_pickups:.1f} bikes, Total Actual: {wed_data['actual_pickups'].sum():.0f}, Total Predicted: {wed_data['predicted_pickups'].sum():.0f}")
print(f"  Dropoffs - MAE: {wed_mae_dropoffs:.1f} bikes, Total Actual: {wed_data['actual_dropoffs'].sum():.0f}, Total Predicted: {wed_data['predicted_dropoffs'].sum():.0f}")

print(f"\nSATURDAY ({saturday_date.date()}):")
print(f"  Pickups  - MAE: {sat_mae_pickups:.1f} bikes, Total Actual: {sat_data['actual_pickups'].sum():.0f}, Total Predicted: {sat_data['predicted_pickups'].sum():.0f}")
print(f"  Dropoffs - MAE: {sat_mae_dropoffs:.1f} bikes, Total Actual: {sat_data['actual_dropoffs'].sum():.0f}, Total Predicted: {sat_data['predicted_dropoffs'].sum():.0f}")

print("\n" + "=" * 80)

## Task 3 Summary: Key Insights

### Operational Recommendations

**1. Focus on High-Demand Clusters**
- Clusters 4, 28, 26, 20, 2, and 12 account for most repositioning needs
- These 6 clusters (24% of total) require 70% of nightly bike movements
- Priority: Ensure these clusters are well-stocked before morning rush

**2. Prediction Reliability**
- Our model predicts repositioning needs with MAE of 21 bikes per cluster-day
- For high-demand clusters (700+ bikes), this is 3% error - excellent for planning
- For low-demand clusters, absolute error is small (5-10 bikes)

**3. Seasonal Patterns**
- Test period (Sep-Dec) shows declining demand as winter approaches
- Summer months likely require more aggressive repositioning
- Consider seasonal adjustment factors for operational planning

**4. Cost-Benefit Analysis**
- Average nightly repositioning: ~2,500 bikes across 25 clusters
- Over-provisioning by 20 bikes per cluster costs less than bike shortages
- Recommendation: Add 10-15% buffer to predicted requirements for critical clusters

### Model Performance Summary

**Task 2 (Demand Prediction):**
- R² = 0.94 (explains 94% of demand variance)
- MAE = 14 bikes per hour per cluster
- Robust across different time periods and clusters

**Task 3 (Repositioning):**
- MAE = 21 bikes per cluster-day
- Successfully identifies critical clusters
- Predictions enable data-driven nightly operations

---

## Project Complete

We have successfully:
1. Cleaned and prepared 17M+ bike trips
2. Engineered 31 predictive features
3. Built and validated demand forecasting models
4. Calculated optimal bike repositioning requirements
5. Provided actionable insights for operations

The Random Forest model is production-ready and can be deployed to support Citi Bike's daily operations.

## Step-by-Step: How We Predict Next Day's Demand

Let's walk through a concrete example to understand how the prediction model works and how we calculate bike repositioning needs.

### Scenario: Predict January 1, 2019 for Cluster 2

We'll demonstrate:
1. How to prepare features for a future day
2. How the model predicts 24 hours of demand
3. How to calculate bikes needed for repositioning

This is the exact process Citi Bike would use every night at midnight to plan the next day.

In [ ]:
# ============================================================================
# STEP 1: Get Historical Data (What we know up to Dec 31, 2018)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 1: Gather Historical Data")
print("=" * 80)

# We need recent data to compute lag features
historical_data = demand_hourly[
    (demand_hourly['cluster'] == 10) & 
    (demand_hourly['date'] >= '2018-12-25') &
    (demand_hourly['date'] < '2019-01-01')
].copy()

print(f"\nWe have real data from {historical_data['date'].min()} to {historical_data['date'].max()}")
print(f"Total observations: {len(historical_data)} (cluster-hour combinations)")

# Show last day we have (Dec 31)
dec_31_data = historical_data[historical_data['date'] == '2018-12-31'].sort_values('hour')
print(f"\nLast day of real data (Dec 31, 2018) for Cluster 2:")
print(f"  Total pickups: {dec_31_data['pickups'].sum():.0f}")
print(f"  Total dropoffs: {dec_31_data['dropoffs'].sum():.0f}")
print(f"  Peak hour: {dec_31_data.loc[dec_31_data['pickups'].idxmax(), 'hour']:.0f}:00 ({dec_31_data['pickups'].max():.0f} pickups)")

# ============================================================================
# STEP 2: Create Template for January 1 (24 hours we want to predict)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 2: Create Prediction Template")
print("=" * 80)

# Create 24 rows (one for each hour of Jan 1)
jan_1_template = pd.DataFrame({
    'date': pd.Timestamp('2019-01-01'),
    'cluster': 2,
    'hour': range(24)
})

print(f"\nCreated template for January 1, 2019:")
print(f"  Date: {jan_1_template['date'].iloc[0].date()}")
print(f"  Cluster: {jan_1_template['cluster'].iloc[0]}")
print(f"  Hours to predict: {len(jan_1_template)} (0-23)")

# ============================================================================
# STEP 3: Add Temporal Features (Calendar information)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 3: Add Temporal Features")
print("=" * 80)

jan_1_template['month'] = 1  # January
jan_1_template['day_of_week'] = 1  # Tuesday (Jan 1, 2019 is a Tuesday)
jan_1_template['is_weekend'] = 0  # Not a weekend
jan_1_template['is_holiday'] = 1  # New Year's Day!

# Season
jan_1_template['season_fall'] = 0
jan_1_template['season_spring'] = 0
jan_1_template['season_summer'] = 0
jan_1_template['season_winter'] = 1  # Winter

# Rush hours
jan_1_template['is_morning_rush'] = jan_1_template['hour'].isin([7, 8]).astype(int)
jan_1_template['is_evening_rush'] = jan_1_template['hour'].isin([17, 18]).astype(int)
jan_1_template['weekday_morning_rush'] = jan_1_template['is_morning_rush']  # It's a weekday
jan_1_template['weekday_evening_rush'] = jan_1_template['is_evening_rush']

# Cyclic hour encoding
jan_1_template['hour_sin'] = np.sin(2 * np.pi * jan_1_template['hour'] / 24)
jan_1_template['hour_cos'] = np.cos(2 * np.pi * jan_1_template['hour'] / 24)

jan_1_template['is_special_event'] = 0

print("\nTemporal features added:")
print(f"  Month: January (1)")
print(f"  Day: Tuesday (weekday)")
print(f"  Holiday: YES (New Year's Day)")
print(f"  Season: Winter")

# ============================================================================
# STEP 4: Add Lag Features (Recent demand history)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 4: Compute Lag Features from Historical Data")
print("=" * 80)

# For each hour of Jan 1, get the corresponding historical values
lag_features = []

for hour in range(24):
    # Lag 24h: Same hour yesterday (Dec 31)
    dec_31_hour = historical_data[
        (historical_data['date'] == '2018-12-31') & 
        (historical_data['hour'] == hour)
    ]
    
    # Lag 168h: Same hour last week (Dec 25)
    dec_25_hour = historical_data[
        (historical_data['date'] == '2018-12-25') & 
        (historical_data['hour'] == hour)
    ]
    
    # Rolling 24h: Average of last 24 hours before this hour
    # (simplified: use Dec 31 average)
    rolling_avg_pickups = historical_data[historical_data['date'] == '2018-12-31']['pickups'].mean()
    rolling_avg_dropoffs = historical_data[historical_data['date'] == '2018-12-31']['dropoffs'].mean()
    
    lag_features.append({
        'hour': hour,
        'pickups_lag_24h': dec_31_hour['pickups'].values[0] if len(dec_31_hour) > 0 else 0,
        'dropoffs_lag_24h': dec_31_hour['dropoffs'].values[0] if len(dec_31_hour) > 0 else 0,
        'pickups_lag_168h': dec_25_hour['pickups'].values[0] if len(dec_25_hour) > 0 else 0,
        'dropoffs_lag_168h': dec_25_hour['dropoffs'].values[0] if len(dec_25_hour) > 0 else 0,
        'pickups_rolling_24h': rolling_avg_pickups,
        'dropoffs_rolling_24h': rolling_avg_dropoffs
    })

lag_df = pd.DataFrame(lag_features)
jan_1_template = jan_1_template.merge(lag_df, on='hour', how='left')

print("\nLag features computed:")
print(f"  Using Dec 31 data for 24h lags")
print(f"  Using Dec 25 data for 168h lags (1 week ago)")
print(f"\nExample for Hour 8 (8:00 AM):")
hour_8 = jan_1_template[jan_1_template['hour'] == 8].iloc[0]
print(f"  Pickups 24h ago (Dec 31, 8am): {hour_8['pickups_lag_24h']:.0f}")
print(f"  Pickups 168h ago (Dec 25, 8am): {hour_8['pickups_lag_168h']:.0f}")

# ============================================================================
# STEP 5: Add Cluster Features (Static characteristics)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 5: Add Cluster Characteristics")
print("=" * 80)

# Get cluster info from training data
cluster_info = demand_hourly[demand_hourly['cluster'] == 2].iloc[0]

jan_1_template['cluster_station_count'] = cluster_info['cluster_station_count']
jan_1_template['cluster_center_lat'] = cluster_info['cluster_center_lat']
jan_1_template['cluster_center_lng'] = cluster_info['cluster_center_lng']
jan_1_template['cluster_total_trips'] = cluster_info['cluster_total_trips']
jan_1_template['cluster_is_source'] = cluster_info['cluster_is_source']

print(f"\nCluster 2 characteristics:")
print(f"  Number of stations: {cluster_info['cluster_station_count']:.0f}")
print(f"  Location: ({cluster_info['cluster_center_lat']:.4f}, {cluster_info['cluster_center_lng']:.4f})")
print(f"  Historical trips: {cluster_info['cluster_total_trips']:.0f}")
print(f"  Type: {'Source (bikes leave)' if cluster_info['cluster_is_source'] == 1 else 'Sink (bikes arrive)'}")

# ============================================================================
# STEP 6: Add User Behavior Features
# ============================================================================

print("\n" + "=" * 80)
print("STEP 6: Add User Behavior Patterns")
print("=" * 80)

# Use December averages for this cluster
dec_user_behavior = demand_hourly[
    (demand_hourly['cluster'] == 2) & 
    (demand_hourly['date'].dt.month == 12)
].agg({
    'avg_trip_duration': 'mean',
    'pct_subscribers': 'mean',
    'avg_age': 'mean'
})

jan_1_template['avg_trip_duration'] = dec_user_behavior['avg_trip_duration']
jan_1_template['pct_subscribers'] = dec_user_behavior['pct_subscribers']
jan_1_template['avg_age'] = dec_user_behavior['avg_age']
jan_1_template['net_flow'] = 0  # Unknown for future

print(f"\nUser behavior (Dec 2018 average):")
print(f"  Avg trip duration: {dec_user_behavior['avg_trip_duration']:.0f} seconds")
print(f"  Subscriber rate: {dec_user_behavior['pct_subscribers']:.1%}")
print(f"  Avg age: {dec_user_behavior['avg_age']:.0f} years")

# ============================================================================
# STEP 7: Make Predictions!
# ============================================================================

print("\n" + "=" * 80)
print("STEP 7: Generate Predictions Using Trained Model")
print("=" * 80)

# Prepare feature matrix
X_jan1 = jan_1_template[feature_cols].fillna(0)

print(f"\nFeature matrix ready: {X_jan1.shape[0]} hours × {X_jan1.shape[1]} features")
print("\nRunning Random Forest model...")

# Predict
jan_1_template['predicted_pickups'] = final_model_pickups.predict(X_jan1).clip(min=0)
jan_1_template['predicted_dropoffs'] = final_model_dropoffs.predict(X_jan1).clip(min=0)

print("✓ Predictions complete!")

# Display predictions
print("\n" + "=" * 80)
print("PREDICTED DEMAND FOR CLUSTER 2 ON JANUARY 1, 2019")
print("=" * 80)

result = jan_1_template[['hour', 'predicted_pickups', 'predicted_dropoffs']].copy()
result['predicted_pickups'] = result['predicted_pickups'].round(0)
result['predicted_dropoffs'] = result['predicted_dropoffs'].round(0)
result.columns = ['Hour', 'Predicted Pickups', 'Predicted Dropoffs']

print("\n" + result.to_string(index=False))

print(f"\nDAILY TOTALS:")
print(f"  Total pickups: {jan_1_template['predicted_pickups'].sum():.0f} bikes")
print(f"  Total dropoffs: {jan_1_template['predicted_dropoffs'].sum():.0f} bikes")
print(f"  Peak pickup hour: {jan_1_template.loc[jan_1_template['predicted_pickups'].idxmax(), 'hour']:.0f}:00")
print(f"  Peak dropoff hour: {jan_1_template.loc[jan_1_template['predicted_dropoffs'].idxmax(), 'hour']:.0f}:00")

# ============================================================================
# STEP 8: Calculate Bikes Needed (Task 3)
# ============================================================================

print("\n" + "=" * 80)
print("STEP 8: Calculate Bikes Needed for Repositioning")
print("=" * 80)

# Calculate net flow (pickups - dropoffs)
jan_1_template['net_flow'] = jan_1_template['predicted_pickups'] - jan_1_template['predicted_dropoffs']

# Calculate cumulative net flow throughout the day
jan_1_template['cumulative_net_flow'] = jan_1_template['net_flow'].cumsum()

# Find the minimum (maximum deficit)
min_cumulative = jan_1_template['cumulative_net_flow'].min()
bikes_needed = max(0, -min_cumulative)

print("\nNet Flow Analysis:")
print(f"  Net flow = Pickups - Dropoffs")
print(f"  Positive = bikes leaving (deficit)")
print(f"  Negative = bikes arriving (surplus)")

print("\n" + jan_1_template[['hour', 'predicted_pickups', 'predicted_dropoffs', 'net_flow', 'cumulative_net_flow']].round(1).to_string(index=False))

print(f"\n" + "=" * 80)
print(f"REPOSITIONING REQUIREMENT:")
print(f"=" * 80)
print(f"\nMinimum cumulative flow: {min_cumulative:.1f}")
print(f"Bikes needed at start of day: {bikes_needed:.0f}")

if bikes_needed > 0:
    worst_hour = jan_1_template.loc[jan_1_template['cumulative_net_flow'].idxmin(), 'hour']
    print(f"\nInterpretation:")
    print(f"  If we start with {bikes_needed:.0f} bikes at midnight,")
    print(f"  we'll have just enough to meet demand all day.")
    print(f"  The critical moment is at hour {worst_hour:.0f}:00,")
    print(f"  when we'll be at our lowest inventory.")
else:
    print(f"\nInterpretation:")
    print(f"  This cluster is balanced - bikes arriving ≈ bikes leaving")
    print(f"  No repositioning needed!")

print("\n" + "=" * 80)

# Load 2018 & 2019 data

In [ ]:
# Load 2018 data
print("Loading 2018 trips...")
TRIPS_2018_PATH = os.path.join(DATA_DIR, 'Trips_2018.csv')
trips_2018 = load_trips_stream(TRIPS_2018_PATH, limit_chunks=None)
print(f"\n2018 trips loaded: {trips_2018.shape}")
print(f"Date range: {trips_2018['start_time'].min()} to {trips_2018['start_time'].max()}")

# Load 2019 data
print("\nLoading 2019 trips...")
TRIPS_2019_PATH = os.path.join(DATA_DIR, 'Trips_2019.csv')
trips_2019 = load_trips_stream(TRIPS_2019_PATH, limit_chunks=None)
print(f"\n2019 trips loaded: {trips_2019.shape}")
print(f"Date range: {trips_2019['start_time'].min()} to {trips_2019['start_time'].max()}")

## Extract station information from both years

In [ ]:
# Extract 2018 stations (we already have these with clusters from earlier in the notebook)
# Let's extract them again from trips_2018 for consistency

start_stations_2018 = trips_2018[['start_station_id', 'start_lat', 'start_lng']].copy()
start_stations_2018.columns = ['station_id', 'lat', 'lng']

end_stations_2018 = trips_2018[['end_station_id', 'end_lat', 'end_lng']].copy()
end_stations_2018.columns = ['station_id', 'lat', 'lng']

stations_2018 = pd.concat([start_stations_2018, end_stations_2018], ignore_index=True)
stations_2018 = stations_2018.drop_duplicates(subset='station_id').reset_index(drop=True)

print(f"Total unique stations in 2018: {len(stations_2018)}")

# Extract 2019 stations
start_stations_2019 = trips_2019[['start_station_id', 'start_lat', 'start_lng']].copy()
start_stations_2019.columns = ['station_id', 'lat', 'lng']

end_stations_2019 = trips_2019[['end_station_id', 'end_lat', 'end_lng']].copy()
end_stations_2019.columns = ['station_id', 'lat', 'lng']

stations_2019 = pd.concat([start_stations_2019, end_stations_2019], ignore_index=True)
stations_2019 = stations_2019.drop_duplicates(subset='station_id').reset_index(drop=True)

print(f"Total unique stations in 2019: {len(stations_2019)}")

# Check overlap
common_stations = set(stations_2018['station_id']) & set(stations_2019['station_id'])
print(f"\nStations in both years: {len(common_stations)}")
print(f"New stations in 2019: {len(set(stations_2019['station_id']) - set(stations_2018['station_id']))}")
print(f"Removed stations from 2018: {len(set(stations_2018['station_id']) - set(stations_2019['station_id']))}")

## Assign clusters to stations using cluster_data.csv

Using the pre-computed cluster assignments from cluster_data.csv to ensure consistency. New stations in 2019 will be assigned using KMeans if needed.

In [ ]:
import joblib

# Load cluster assignments from cluster_data.csv
cluster_data_path = os.path.join(DATA_DIR, 'cluster_data.csv')
cluster_mapping = pd.read_csv(cluster_data_path)
print(f"Loaded cluster mappings for {len(cluster_mapping)} stations")
print(f"Cluster mapping columns: {cluster_mapping.columns.tolist()}")

# Create a station_id -> cluster mapping dictionary
station_cluster_map = dict(zip(cluster_mapping['station_id'], cluster_mapping['cluster']))

# Assign clusters to 2018 stations using the mapping
stations_2018['cluster'] = stations_2018['station_id'].map(station_cluster_map)

# Check for 2018 stations not in the mapping
missing_2018 = stations_2018[stations_2018['cluster'].isna()]
if len(missing_2018) > 0:
    print(f"\n⚠️ WARNING: {len(missing_2018)} stations in 2018 data not found in cluster_data.csv:")
    print(missing_2018[['station_id', 'lat', 'lng']].head(10))
else:
    print(f"\n✓ All {len(stations_2018)} stations in 2018 found in cluster mapping")

print("\n2018 Stations cluster distribution:")
print(stations_2018['cluster'].value_counts().sort_index())

# Assign clusters to 2019 stations using the mapping
stations_2019['cluster'] = stations_2019['station_id'].map(station_cluster_map)

# Check for NEW stations in 2019 not in the mapping
new_stations_2019 = stations_2019[stations_2019['cluster'].isna()].copy()

if len(new_stations_2019) > 0:
    print(f"\n🆕 FOUND {len(new_stations_2019)} NEW STATIONS in 2019 not in cluster_data.csv:")
    print("=" * 80)
    print(new_stations_2019[['station_id', 'lat', 'lng']])
    
    # Assign clusters to new stations using KMeans prediction
    print("\n📍 Assigning new stations to clusters using KMeans model...")
    coords_new = new_stations_2019[['lat', 'lng']].values
    kmeans = joblib.load("kmeans_model.pkl") # load from file
    new_clusters = kmeans.predict(coords_new)
    
    # Update the cluster assignments for new stations
    stations_2019.loc[stations_2019['cluster'].isna(), 'cluster'] = new_clusters
    
    print(f"✓ Assigned {len(new_stations_2019)} new stations to clusters")
    print("\nNew station cluster assignments:")
    for idx, row in new_stations_2019.iterrows():
        cluster_assigned = stations_2019.loc[idx, 'cluster']
        print(f"  Station {row['station_id']}: Cluster {int(cluster_assigned)}")
else:
    print(f"\n✓ All {len(stations_2019)} stations in 2019 found in cluster mapping - NO NEW STATIONS")

print("\n2019 Stations cluster distribution:")
print(stations_2019['cluster'].value_counts().sort_index())

# Summary
print("\n" + "=" * 80)
print("SUMMARY:")
print(f"  Total 2018 stations: {len(stations_2018)}")
print(f"  Total 2019 stations: {len(stations_2019)}")
print(f"  Stations in cluster_data.csv: {len(cluster_mapping)}")
print(f"  New stations in 2019: {len(new_stations_2019)}")
print("=" * 80)

## Create hourly demand dataset for 2018 and 2019

We'll aggregate trips by cluster and hour to create time series data for analysis.

In [ ]:
# Merge cluster assignments into trip data for 2018
trips_2018 = trips_2018.merge(
    stations_2018[['station_id', 'cluster']].rename(columns={'cluster': 'start_cluster'}),
    left_on='start_station_id',
    right_on='station_id',
    how='left'
).drop('station_id', axis=1)

trips_2018 = trips_2018.merge(
    stations_2018[['station_id', 'cluster']].rename(columns={'cluster': 'end_cluster'}),
    left_on='end_station_id',
    right_on='station_id',
    how='left'
).drop('station_id', axis=1)

# Merge cluster assignments into trip data for 2019
trips_2019 = trips_2019.merge(
    stations_2019[['station_id', 'cluster']].rename(columns={'cluster': 'start_cluster'}),
    left_on='start_station_id',
    right_on='station_id',
    how='left'
).drop('station_id', axis=1)

trips_2019 = trips_2019.merge(
    stations_2019[['station_id', 'cluster']].rename(columns={'cluster': 'end_cluster'}),
    left_on='end_station_id',
    right_on='station_id',
    how='left'
).drop('station_id', axis=1)

print("2018 trips with cluster assignments:")
print(trips_2018[['start_time', 'start_station_id', 'start_cluster', 'end_station_id', 'end_cluster']].head())
print(f"\nMissing cluster assignments in 2018: {trips_2018['start_cluster'].isna().sum()}")

print("\n2019 trips with cluster assignments:")
print(trips_2019[['start_time', 'start_station_id', 'start_cluster', 'end_station_id', 'end_cluster']].head())
print(f"\nMissing cluster assignments in 2019: {trips_2019['start_cluster'].isna().sum()}")

In [ ]:
# Create hourly aggregations for 2018
# Extract hour from start_time for pickups
trips_2018['hour'] = trips_2018['start_time'].dt.floor('H')

# Aggregate pickups by cluster and hour for 2018
pickups_2018 = trips_2018.groupby(['start_cluster', 'hour']).size().reset_index(name='pickups')
pickups_2018.rename(columns={'start_cluster': 'cluster'}, inplace=True)

# Aggregate dropoffs by cluster and hour for 2018
trips_2018['end_hour'] = trips_2018['end_time'].dt.floor('H')
dropoffs_2018 = trips_2018.groupby(['end_cluster', 'end_hour']).size().reset_index(name='dropoffs')
dropoffs_2018.rename(columns={'end_cluster': 'cluster', 'end_hour': 'hour'}, inplace=True)

# Merge pickups and dropoffs for 2018
hourly_2018 = pickups_2018.merge(dropoffs_2018, on=['cluster', 'hour'], how='outer').fillna(0)
hourly_2018['pickups'] = hourly_2018['pickups'].astype(int)
hourly_2018['dropoffs'] = hourly_2018['dropoffs'].astype(int)
hourly_2018['year'] = 2018

print(f"2018 hourly demand shape: {hourly_2018.shape}")
print(hourly_2018.head())
print(f"\nDate range: {hourly_2018['hour'].min()} to {hourly_2018['hour'].max()}")

In [ ]:
# Create hourly aggregations for 2019
trips_2019['hour'] = trips_2019['start_time'].dt.floor('H')

# Aggregate pickups by cluster and hour for 2019
pickups_2019 = trips_2019.groupby(['start_cluster', 'hour']).size().reset_index(name='pickups')
pickups_2019.rename(columns={'start_cluster': 'cluster'}, inplace=True)

# Aggregate dropoffs by cluster and hour for 2019
trips_2019['end_hour'] = trips_2019['end_time'].dt.floor('H')
dropoffs_2019 = trips_2019.groupby(['end_cluster', 'end_hour']).size().reset_index(name='dropoffs')
dropoffs_2019.rename(columns={'end_cluster': 'cluster', 'end_hour': 'hour'}, inplace=True)

# Merge pickups and dropoffs for 2019
hourly_2019 = pickups_2019.merge(dropoffs_2019, on=['cluster', 'hour'], how='outer').fillna(0)
hourly_2019['pickups'] = hourly_2019['pickups'].astype(int)
hourly_2019['dropoffs'] = hourly_2019['dropoffs'].astype(int)
hourly_2019['year'] = 2019

print(f"2019 hourly demand shape: {hourly_2019.shape}")
print(hourly_2019.head())
print(f"\nDate range: {hourly_2019['hour'].min()} to {hourly_2019['hour'].max()}")

In [ ]:
# Combine 2018 and 2019 hourly data
hourly_combined = pd.concat([hourly_2018, hourly_2019], ignore_index=True)
hourly_combined = hourly_combined.sort_values(['cluster', 'hour']).reset_index(drop=True)

print(f"Combined hourly demand shape: {hourly_combined.shape}")
print(f"\nDate range: {hourly_combined['hour'].min()} to {hourly_combined['hour'].max()}")
print(f"\nClusters: {sorted(hourly_combined['cluster'].unique())}")
print(f"\nSample data:")
print(hourly_combined.head(10))

# Check for any missing hours (gaps in time series)
print(f"\nChecking for time gaps...")
for cluster in sorted(hourly_combined['cluster'].unique())[:3]:  # Check first 3 clusters
    cluster_data = hourly_combined[hourly_combined['cluster'] == cluster].sort_values('hour')
    expected_hours = (cluster_data['hour'].max() - cluster_data['hour'].min()).total_seconds() / 3600 + 1
    actual_hours = len(cluster_data)
    print(f"Cluster {cluster}: Expected {int(expected_hours)} hours, got {actual_hours} hours")

## Analyze Missing Hours in Time Series

Before proceeding, let's investigate the missing hours to understand if they represent true gaps or simply hours with zero activity.

In [ ]:
# Detailed analysis of missing hours across all clusters
print("="*80)
print("MISSING HOURS ANALYSIS")
print("="*80)

# Get the overall date range
start_date = hourly_combined['hour'].min()
end_date = hourly_combined['hour'].max()
total_expected_hours = int((end_date - start_date).total_seconds() / 3600) + 1

print(f"\nOverall date range: {start_date} to {end_date}")
print(f"Total expected hours per cluster: {total_expected_hours:,}")

# Analyze each cluster
missing_data_summary = []

for cluster in sorted(hourly_combined['cluster'].unique()):
    cluster_data = hourly_combined[hourly_combined['cluster'] == cluster].sort_values('hour')
    
    # Get actual hours present
    actual_hours = len(cluster_data)
    missing_hours = total_expected_hours - actual_hours
    missing_pct = (missing_hours / total_expected_hours) * 100
    
    # Create complete hour range for this cluster
    complete_hours = pd.date_range(start=start_date, end=end_date, freq='h')
    complete_df = pd.DataFrame({'hour': complete_hours})
    
    # Find which hours are missing
    merged = complete_df.merge(cluster_data[['hour']], on='hour', how='left', indicator=True)
    missing_hour_df = merged[merged['_merge'] == 'left_only']
    
    # Analyze patterns in missing hours
    if len(missing_hour_df) > 0:
        missing_hour_df['hour_of_day'] = missing_hour_df['hour'].dt.hour
        missing_hour_df['day_of_week'] = missing_hour_df['hour'].dt.dayofweek
        missing_hour_df['month'] = missing_hour_df['hour'].dt.month
        missing_hour_df['year'] = missing_hour_df['hour'].dt.year
        
        hour_distribution = missing_hour_df['hour_of_day'].value_counts().sort_index()
        most_common_hour = hour_distribution.idxmax() if len(hour_distribution) > 0 else None
        
        # Get average demand when cluster HAS data
        avg_pickups = cluster_data['pickups'].mean()
        avg_dropoffs = cluster_data['dropoffs'].mean()
    else:
        most_common_hour = None
        avg_pickups = cluster_data['pickups'].mean()
        avg_dropoffs = cluster_data['dropoffs'].mean()
    
    missing_data_summary.append({
        'cluster': int(cluster),
        'actual_hours': actual_hours,
        'missing_hours': missing_hours,
        'missing_pct': missing_pct,
        'avg_pickups': avg_pickups,
        'avg_dropoffs': avg_dropoffs,
        'most_common_missing_hour': most_common_hour
    })

# Create summary dataframe
missing_summary_df = pd.DataFrame(missing_data_summary)
missing_summary_df = missing_summary_df.sort_values('missing_pct', ascending=False)

print("\nMissing Hours Summary (sorted by % missing):")
print(missing_summary_df.to_string(index=False))

print("\n" + "="*80)
print("KEY INSIGHTS:")
print("="*80)
print(f"Clusters with >1% missing data: {len(missing_summary_df[missing_summary_df['missing_pct'] > 1])}")
print(f"Clusters with NO missing data: {len(missing_summary_df[missing_summary_df['missing_hours'] == 0])}")
print(f"\nAverage missing hours per cluster: {missing_summary_df['missing_hours'].mean():.1f}")
print(f"Max missing hours: {missing_summary_df['missing_hours'].max():.0f} (Cluster {missing_summary_df.iloc[0]['cluster']})")
print(f"Min missing hours: {missing_summary_df['missing_hours'].min():.0f}")

In [ ]:
# Deep dive: Examine when missing hours occur
# Pick a few clusters with different amounts of missing data

print("="*80)
print("DETAILED PATTERN ANALYSIS OF MISSING HOURS")
print("="*80)

# Select 3 clusters: worst, middle, best
worst_cluster = missing_summary_df.iloc[0]['cluster']
mid_cluster = missing_summary_df.iloc[len(missing_summary_df)//2]['cluster']
best_cluster = missing_summary_df[missing_summary_df['missing_hours'] > 0].iloc[-1]['cluster'] if len(missing_summary_df[missing_summary_df['missing_hours'] > 0]) > 0 else worst_cluster

for cluster_id in [worst_cluster, mid_cluster, best_cluster]:
    cluster_data = hourly_combined[hourly_combined['cluster'] == cluster_id].sort_values('hour')
    
    # Find missing hours
    complete_hours = pd.date_range(start=start_date, end=end_date, freq='h')
    complete_df = pd.DataFrame({'hour': complete_hours})
    merged = complete_df.merge(cluster_data[['hour']], on='hour', how='left', indicator=True)
    missing_hour_df = merged[merged['_merge'] == 'left_only'].copy()
    
    if len(missing_hour_df) > 0:
        missing_hour_df['hour_of_day'] = missing_hour_df['hour'].dt.hour
        missing_hour_df['day_of_week'] = missing_hour_df['hour'].dt.dayofweek
        missing_hour_df['month'] = missing_hour_df['hour'].dt.month
        
        print(f"\n🔍 Cluster {int(cluster_id)} - Missing {len(missing_hour_df)} hours:")
        print("-"*80)
        
        # Time of day distribution
        print("  Missing by Hour of Day:")
        hour_dist = missing_hour_df['hour_of_day'].value_counts().sort_index()
        for hour, count in hour_dist.head(10).items():
            pct = (count / len(missing_hour_df)) * 100
            print(f"    {hour:02d}:00 - {count:4d} missing ({pct:5.1f}%)")
        
        # Day of week distribution
        print("\n  Missing by Day of Week:")
        dow_dist = missing_hour_df['day_of_week'].value_counts().sort_index()
        dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        for dow, count in dow_dist.items():
            pct = (count / len(missing_hour_df)) * 100
            print(f"    {dow_names[dow]:3s} - {count:4d} missing ({pct:5.1f}%)")
        
        # First few missing timestamps
        print("\n  Sample missing timestamps (first 5):")
        for ts in missing_hour_df['hour'].head(5):
            print(f"    {ts}")
    else:
        print(f"\n✓ Cluster {int(cluster_id)} - No missing hours!")

print("\n" + "="*80)

## Decision: Fill Missing Hours with Zero

**Interpretation:** Missing hours in the aggregated data likely represent periods with **zero rides** rather than data collection issues, because:

1. **Aggregation Process**: We're grouping trips by hour - if no trips occurred in a given hour for a cluster, that hour won't appear in the groupby result
2. **Low-Activity Clusters**: Smaller/less active clusters (especially during off-peak hours like 2-5 AM) may genuinely have zero trips
3. **Time Series Modeling**: For ARIMA/SARIMA and other time series models, we need a complete hourly sequence

**Approach**: Fill missing hours with `pickups=0, dropoffs=0` to create a complete time series for each cluster.

In [ ]:
# Create complete time series for all clusters by filling missing hours with zeros

print("Creating complete hourly time series for all clusters...")
print("="*80)

# Get the overall date range
start_date = hourly_combined['hour'].min()
end_date = hourly_combined['hour'].max()

# Create complete hourly range
complete_hours = pd.date_range(start=start_date, end=end_date, freq='h')

# Get all unique clusters
all_clusters = sorted(hourly_combined['cluster'].unique())

# Create a complete grid of cluster x hour combinations
complete_grid = []
for cluster in all_clusters:
    for hour in complete_hours:
        complete_grid.append({'cluster': cluster, 'hour': hour})

complete_df = pd.DataFrame(complete_grid)

print(f"Created grid: {len(all_clusters)} clusters × {len(complete_hours):,} hours = {len(complete_df):,} records")

# Merge with actual data
hourly_complete = complete_df.merge(
    hourly_combined[['cluster', 'hour', 'pickups', 'dropoffs', 'year']], 
    on=['cluster', 'hour'], 
    how='left'
)

# Fill missing values with 0 for pickups and dropoffs
hourly_complete['pickups'] = hourly_complete['pickups'].fillna(0).astype(int)
hourly_complete['dropoffs'] = hourly_complete['dropoffs'].fillna(0).astype(int)

# Fill year based on the hour timestamp
hourly_complete['year'] = hourly_complete['hour'].dt.year

# Add total_demand column
hourly_complete['total_demand'] = hourly_complete['pickups'] + hourly_complete['dropoffs']

print(f"\nBefore filling - records: {len(hourly_combined):,}")
print(f"After filling  - records: {len(hourly_complete):,}")
print(f"Records added: {len(hourly_complete) - len(hourly_combined):,}")

# Verify completeness
print("\nVerifying completeness per cluster:")
for cluster in all_clusters[:5]:  # Show first 5
    cluster_data = hourly_complete[hourly_complete['cluster'] == cluster]
    expected = len(complete_hours)
    actual = len(cluster_data)
    status = "✓" if actual == expected else "✗"
    print(f"  {status} Cluster {int(cluster):2d}: {actual:,} / {expected:,} hours")

print(f"\n✓ All clusters now have complete hourly time series!")

# Show statistics
print("\nDataset Statistics:")
print(f"  Total records: {len(hourly_complete):,}")
print(f"  Clusters: {len(all_clusters)}")
print(f"  Hours per cluster: {len(complete_hours):,}")
print(f"  Zero-activity hours added: {len(hourly_complete) - len(hourly_combined):,}")
print(f"  Percentage of zero-filled: {((len(hourly_complete) - len(hourly_combined)) / len(hourly_complete)) * 100:.2f}%")

# Replace the original hourly_combined with the complete version
hourly_combined = hourly_complete.copy()

print("\n" + "="*80)

In [ ]:
# Check and fix clusters with duplicate hours
print("Checking for duplicate hour entries per cluster...")
print("="*80)

duplicates_found = False
for cluster in all_clusters:
    cluster_data = hourly_complete[hourly_complete['cluster'] == cluster]
    duplicate_hours = cluster_data[cluster_data.duplicated(subset=['hour'], keep=False)]
    
    if len(duplicate_hours) > 0:
        duplicates_found = True
        print(f"\n⚠️ Cluster {int(cluster)} has {len(duplicate_hours)} duplicate hour entries:")
        print(duplicate_hours[['cluster', 'hour', 'pickups', 'dropoffs']].sort_values('hour').head(10))

if duplicates_found:
    print("\n🔧 Fixing duplicates by aggregating (summing) duplicate hours...")
    
    # Aggregate duplicates by summing pickups and dropoffs
    hourly_complete = hourly_complete.groupby(['cluster', 'hour']).agg({
        'pickups': 'sum',
        'dropoffs': 'sum',
        'year': 'first',  # Take first year value (should all be same)
        'total_demand': 'sum'
    }).reset_index()
    
    print(f"✓ After deduplication: {len(hourly_complete):,} records")
    
    # Verify again
    print("\nVerifying completeness after deduplication:")
    for cluster in all_clusters[:5]:
        cluster_data = hourly_complete[hourly_complete['cluster'] == cluster]
        expected = len(complete_hours)
        actual = len(cluster_data)
        status = "✓" if actual == expected else "✗"
        print(f"  {status} Cluster {int(cluster):2d}: {actual:,} / {expected:,} hours")
else:
    print("✓ No duplicates found - data is clean!")

# Final verification
total_records = len(hourly_complete)
expected_records = len(all_clusters) * len(complete_hours)
print(f"\n{'='*80}")
print(f"FINAL VERIFICATION:")
print(f"  Expected: {expected_records:,} records ({len(all_clusters)} clusters × {len(complete_hours):,} hours)")
print(f"  Actual:   {total_records:,} records")
print(f"  Match: {'✓ YES' if total_records == expected_records else '✗ NO'}")
print("="*80)

## Save Complete Time Series Dataset

In [ ]:
# Save the complete hourly demand dataset (with zeros filled in)
output_path = os.path.join(DATA_DIR, 'hourly_demand_2018_2019_complete.csv')
hourly_complete.to_csv(output_path, index=False)
print(f"✓ Saved complete hourly demand to: {output_path}")
print(f"  Total records: {len(hourly_complete):,}")
print(f"  Clusters: {len(hourly_complete['cluster'].unique())}")
print(f"  Date range: {hourly_complete['hour'].min()} to {hourly_complete['hour'].max()}")

# Also update the hourly_combined variable to use the complete version
hourly_combined = hourly_complete.copy()
print(f"\n✓ Updated hourly_combined variable with complete time series")

---

## Summary: Missing Data Handling

### Analysis Results:
- **Original aggregated data**: 479,685 records (incomplete - missing hours where no trips occurred)
- **Complete dataset**: 526,110 records (all 30 clusters × 17,537 hours)
- **Hours added**: 46,425 hours with zero activity
- **Pattern**: Zero-activity hours primarily occur during **late night/early morning** (2-5 AM) in less active clusters
- **Interpretation**: These represent **genuine zero-activity periods**, not data collection gaps

### Why "Missing" Hours Occurred:
1. When we aggregated trips by `groupby(['cluster', 'hour'])`, hours with **zero trips** don't appear in the result
2. Smaller/peripheral clusters have lower activity, especially during off-peak hours (some have 0 trips)
3. This is expected behavior - not all clusters have continuous 24/7 activity
4. Example: Cluster 23 (only 3-6 stations) has many hours with no trips

### Solution Implemented:
✅ **Created complete hourly time series with zeros for all clusters**
- Built complete hourly grid: 30 clusters × 17,537 hours = 526,110 records
- Original data had 479,685 records (hours with at least 1 trip)
- Added 46,425 hours with `pickups=0, dropoffs=0` (genuine zero-activity periods)
- Resolved 14 duplicate hour entries by aggregating (summing)
- **Final dataset**: 526,110 complete hourly records ✓

### Why This Approach is Correct:
1. **Time Series Models Requirement**: ARIMA/SARIMA need continuous sequences without gaps
2. **Logical Interpretation**: No trips recorded = zero demand (not missing data)
3. **Statistical Validity**: Zeros are actual observations, not imputed values
4. **Model Performance**: Captures true demand patterns including low-activity periods
5. **Data Integrity**: Every cluster now has exactly 17,537 hourly records

### Impact on Analysis:
- **Cluster 2 Example**: 
  - Complete with 17,537 hours
  - Only 44 hours (0.25%) have zero total demand
  - 94 hours (0.54%) have zero pickups
  - 90 hours (0.51%) have zero dropoffs
- Time series now ready for forecasting models
- Can apply seasonal decomposition without gaps
- Stationarity tests work on complete sequences

---

# Time Series Exploratory Analysis - Cluster 2

Following the analysis plan, we'll focus on Cluster 2 for detailed time series analysis including:
- Data visualization over the full 2-year period
- Seasonal decomposition
- Stationarity testing
- Additional exploratory patterns

## 1. Extract and Prepare Cluster 2 Data

In [ ]:
# Filter data for Cluster 2 - now with complete hourly data (zeros filled)
cluster_2_data = hourly_combined[hourly_combined['cluster'] == 2].copy()
cluster_2_data = cluster_2_data.sort_values('hour').reset_index(drop=True)

print(f"Cluster 2 hourly data shape: {cluster_2_data.shape}")
print(f"Date range: {cluster_2_data['hour'].min()} to {cluster_2_data['hour'].max()}")
print(f"\nTotal pickups: {cluster_2_data['pickups'].sum():,}")
print(f"Total dropoffs: {cluster_2_data['dropoffs'].sum():,}")

# Check completeness
expected_hours = int((cluster_2_data['hour'].max() - cluster_2_data['hour'].min()).total_seconds() / 3600) + 1
print(f"\nCompleteness check:")
print(f"  Expected hours: {expected_hours:,}")
print(f"  Actual hours: {len(cluster_2_data):,}")
print(f"  Complete: {'✓ YES' if len(cluster_2_data) == expected_hours else '✗ NO'}")

print(f"\nZero-activity hours:")
zero_pickups = (cluster_2_data['pickups'] == 0).sum()
zero_dropoffs = (cluster_2_data['dropoffs'] == 0).sum()
zero_both = ((cluster_2_data['pickups'] == 0) & (cluster_2_data['dropoffs'] == 0)).sum()
print(f"  Hours with 0 pickups: {zero_pickups:,} ({zero_pickups/len(cluster_2_data)*100:.2f}%)")
print(f"  Hours with 0 dropoffs: {zero_dropoffs:,} ({zero_dropoffs/len(cluster_2_data)*100:.2f}%)")
print(f"  Hours with 0 total demand: {zero_both:,} ({zero_both/len(cluster_2_data)*100:.2f}%)")

print(f"\nSample data:")
cluster_2_data.head(10)

In [ ]:
# Prepare data for time series analysis
# Since we already have complete hourly data with zeros filled, we just need to prepare it

# Add total_demand column if not present
if 'total_demand' not in cluster_2_data.columns:
    cluster_2_data['total_demand'] = cluster_2_data['pickups'] + cluster_2_data['dropoffs']

# Create indexed version for time series operations
cluster_2_ts = cluster_2_data.copy()
cluster_2_ts_indexed = cluster_2_ts.set_index('hour')

print(f"Time series data ready for analysis:")
print(f"  Shape: {cluster_2_ts.shape}")
print(f"  Date range: {cluster_2_ts['hour'].min()} to {cluster_2_ts['hour'].max()}")
print(f"\nBasic statistics:")
print(cluster_2_ts[['pickups', 'dropoffs', 'total_demand']].describe())

## 2. Visualize Hourly Demand Over 2-Year Period

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Create visualization of the full time series
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# Plot 1: Pickups
axes[0].plot(cluster_2_ts['hour'], cluster_2_ts['pickups'], linewidth=0.5, color='blue', alpha=0.7)
axes[0].set_title('Cluster 2: Hourly Pickups (2018-2019)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Pickups', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# Plot 2: Dropoffs
axes[1].plot(cluster_2_ts['hour'], cluster_2_ts['dropoffs'], linewidth=0.5, color='red', alpha=0.7)
axes[1].set_title('Cluster 2: Hourly Dropoffs (2018-2019)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Dropoffs', fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

# Plot 3: Total demand
axes[2].plot(cluster_2_ts['hour'], cluster_2_ts['total_demand'], linewidth=0.5, color='green', alpha=0.7)
axes[2].set_title('Cluster 2: Total Hourly Demand (2018-2019)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Total Demand', fontsize=12)
axes[2].set_xlabel('Date', fontsize=12)
axes[2].grid(True, alpha=0.3)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Key observations from the visualization:")
print("- Look for seasonal patterns (yearly, monthly)")
print("- Identify any trends (increasing/decreasing demand)")
print("- Check for anomalies or outliers")
print("- Note any data gaps or missing periods")

## 3. Seasonal Decomposition

Using statsmodels to decompose the time series into trend, seasonal, and residual components.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Set hour as index for time series analysis
cluster_2_ts_indexed = cluster_2_ts.set_index('hour')

# Perform seasonal decomposition on total demand
# Using a weekly period (24*7 = 168 hours) to capture weekly patterns
decomposition = seasonal_decompose( # TODO : why only use weekly. Try yearly as well
    cluster_2_ts_indexed['total_demand'], 
    model='additive',
    period=24*7  # Weekly seasonality
)

# Plot decomposition
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

# Original series
axes[0].plot(cluster_2_ts_indexed.index, cluster_2_ts_indexed['total_demand'], linewidth=0.5)
axes[0].set_title('Original Time Series - Total Demand', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Demand', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Trend
axes[1].plot(decomposition.trend.index, decomposition.trend, linewidth=1, color='red')
axes[1].set_title('Trend Component', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Trend', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Seasonal
axes[2].plot(decomposition.seasonal.index[:24*7*4], decomposition.seasonal[:24*7*4], linewidth=1, color='green')
axes[2].set_title('Seasonal Component (4 weeks shown)', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Seasonal', fontsize=12)
axes[2].grid(True, alpha=0.3)

# Residual
axes[3].plot(decomposition.resid.index, decomposition.resid, linewidth=0.5, color='purple', alpha=0.7)
axes[3].set_title('Residual Component', fontsize=14, fontweight='bold')
axes[3].set_ylabel('Residual', fontsize=12)
axes[3].set_xlabel('Date', fontsize=12)
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDecomposition Summary:")
print(f"Trend range: {decomposition.trend.min():.2f} to {decomposition.trend.max():.2f}")
print(f"Seasonal range: {decomposition.seasonal.min():.2f} to {decomposition.seasonal.max():.2f}")
print(f"Residual std: {decomposition.resid.std():.2f}")

## 4. Stationarity Testing

Testing whether the time series is stationary using:
- Augmented Dickey-Fuller (ADF) test
- KPSS test

A stationary series has constant mean, variance, and autocorrelation over time.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

def test_stationarity(timeseries, name="Series"):
    """
    Perform ADF and KPSS tests for stationarity
    """
    # print(f"\n{'='*60}")
    print(f"Stationarity Tests for: {name}")
    # print(f"{'='*60}")
    
    # ADF Test
    # print("\n1. Augmented Dickey-Fuller Test:")
    # print("-" * 40)
    adf_result = adfuller(timeseries.dropna())
    # adf_result = adfuller(timeseries.diff(12).dropna())
    # print(f"ADF Statistic: {adf_result[0]:.6f}")
    # print(f"p-value: {adf_result[1]:.6f}")
    # print(f"Critical Values:")
    # for key, value in adf_result[4].items():
    #     print(f"   {key}: {value:.3f}")

    if adf_result[1] < 0.05:
        print("\n⚠️  ADF says: STATIONARY (p < 0.05)")
        # print("    But this can be misleading with strong seasonality!")
    else:
        print("\n✗ Result: NON-STATIONARY (p >= 0.05)")
        # print("  The series has a unit root.")
    
    # KPSS Test
    # print("\n2. KPSS Test:")
    # print("-" * 40)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)
        kpss_result = kpss(timeseries.dropna(), regression='c', nlags='auto')
    # kpss_result = kpss(timeseries.diff(12).dropna(), regression='c', nlags='auto')
    # print(f"KPSS Statistic: {kpss_result[0]:.6f}")
    # print(f"p-value: {kpss_result[1]:.6f}")
    # print(f"Critical Values:")
    # for key, value in kpss_result[3].items():
    #     print(f"   {key}: {value:.3f}")
    
    if kpss_result[1] > 0.05:
        print("✓ KPSS says: STATIONARY (p > 0.05)")
    else:
        print("✗ KPSS says: NON-STATIONARY (p <= 0.05)")
        # print("  Strong evidence of non-stationarity")
    
    # Combined interpretation
    # print("\n" + "="*60)
    # print("COMBINED INTERPRETATION:")
    # print("="*60)
    if adf_result[1] < 0.05 and kpss_result[1] <= 0.05:
        print("⚠️  CONTRADICTORY RESULTS - Likely TREND-STATIONARY")
        # print("   ADF: Stationary | KPSS: Non-stationary")
        # print("   → Strong seasonality causing confusion")
        # print("   → Series has changing mean/variance over time")
        # print("   → Needs differencing or detrending for modeling")
    elif adf_result[1] < 0.05 and kpss_result[1] > 0.05:
        print("✓ BOTH AGREE: Stationary")
    elif adf_result[1] >= 0.05 and kpss_result[1] <= 0.05:
        print("✗ BOTH AGREE: Non-stationary")
    else:
        print("⚠️  Mixed results - further investigation needed")
    print(f"\n{'='*60}\n")
    
    return adf_result, kpss_result

# Test 1: Total demand (original series)
print("\n" + "🔍 TESTING ORIGINAL SERIES" + "\n")
test_stationarity(cluster_2_ts_indexed['total_demand'], "Total Demand")

# Test 2: Pickups separately
test_stationarity(cluster_2_ts_indexed['pickups'], "Pickups")

# Test 3: Dropoffs separately  
test_stationarity(cluster_2_ts_indexed['dropoffs'], "Dropoffs")

In [ ]:
# Test 4: First-differenced series (removes trend)
print("\n" + "🔍 TESTING DIFFERENCED SERIES (Removing Trend)" + "\n")

diff_total = cluster_2_ts_indexed['total_demand'].diff(3).dropna()
test_stationarity(diff_total, "Total Demand (1st Difference)")

diff_pickups = cluster_2_ts_indexed['pickups'].diff(3).dropna()
test_stationarity(diff_pickups, "Pickups (1st Difference)")

diff_dropoffs = cluster_2_ts_indexed['dropoffs'].diff(3).dropna()
test_stationarity(diff_dropoffs, "Dropoffs (1st Difference)")

### Understanding the Stationarity Paradox

**Why does ADF say "stationary" when the plot clearly shows non-stationarity?**

1. **ADF Test Limitation**: ADF tests for *unit root* (random walk behavior), not strict stationarity
   - It can say "stationary" for series with strong mean-reversion despite visible trends
   - Strong daily/weekly cycles create mean-reversion that fools the test

2. **KPSS Test is More Reliable Here**: Tests for level/trend stationarity
   - KPSS correctly identifies non-stationarity (p < 0.05)
   - More appropriate for series with strong seasonality

3. **Visual Evidence of Non-Stationarity**:
   - Changing mean: Winter (low) vs Summer (high) demand
   - Changing variance: More variable in peak seasons
   - Strong seasonal patterns at multiple scales (daily, weekly, yearly)

4. **What "Trend-Stationary" Means**:
   - Series becomes stationary after removing trend/seasonal components
   - Common in time series with predictable patterns
   - Need to difference or deseasonalize for forecasting models

In [ ]:
# Visualize why the series is non-stationary
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# Plot 1: Rolling mean and std (30-day window)
rolling_mean = cluster_2_ts_indexed['total_demand'].rolling(window=24*30).mean()
rolling_std = cluster_2_ts_indexed['total_demand'].rolling(window=24*30).std()

axes[0].plot(cluster_2_ts_indexed.index, cluster_2_ts_indexed['total_demand'], 
             linewidth=0.3, alpha=0.5, label='Original', color='blue')
axes[0].plot(rolling_mean.index, rolling_mean, linewidth=2, label='Rolling Mean (30 days)', color='red')
axes[0].plot(rolling_std.index, rolling_std, linewidth=2, label='Rolling Std (30 days)', color='green')
axes[0].set_title('Total Demand: Non-Constant Mean & Variance Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Demand', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: First difference (removes trend)
diff_series = cluster_2_ts_indexed['total_demand'].diff()
axes[1].plot(diff_series.index, diff_series, linewidth=0.5, alpha=0.7, color='purple')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('First Difference: Changes from Hour to Hour', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Change in Demand', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Plot 3: Distribution by season to show non-constant mean
cluster_2_ts_indexed['month'] = cluster_2_ts_indexed.index.month
cluster_2_ts_indexed['season'] = cluster_2_ts_indexed['month'].apply(
    lambda x: 'Winter (Dec-Feb)' if x in [12, 1, 2] 
    else 'Spring (Mar-May)' if x in [3, 4, 5]
    else 'Summer (Jun-Aug)' if x in [6, 7, 8]
    else 'Fall (Sep-Nov)'
)

for season in ['Winter (Dec-Feb)', 'Spring (Mar-May)', 'Summer (Jun-Aug)', 'Fall (Sep-Nov)']:
    season_data = cluster_2_ts_indexed[cluster_2_ts_indexed['season'] == season]['total_demand']
    axes[2].hist(season_data, bins=50, alpha=0.5, label=f'{season} (μ={season_data.mean():.0f})')

axes[2].set_title('Demand Distribution by Season: Different Means = Non-Stationary', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Total Demand', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nSeasonal Statistics:")
print("="*60)
for season in ['Winter (Dec-Feb)', 'Spring (Mar-May)', 'Summer (Jun-Aug)', 'Fall (Sep-Nov)']:
    season_data = cluster_2_ts_indexed[cluster_2_ts_indexed['season'] == season]['total_demand']
    print(f"{season:20s} - Mean: {season_data.mean():6.1f}, Std: {season_data.std():6.1f}")
print("="*60)
print("→ Different seasonal means = NON-STATIONARY series")

## 5. ACF and PACF Analysis

Autocorrelation and Partial Autocorrelation plots help identify patterns and lags in the time series.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# ACF plot
plot_acf(cluster_2_ts_indexed['total_demand'].dropna(), lags=168, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF) - 1 Week of Lags', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lag (hours)', fontsize=12)

# PACF plot
plot_pacf(cluster_2_ts_indexed['total_demand'].dropna(), lags=168, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF) - 1 Week of Lags', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lag (hours)', fontsize=12)

plt.tight_layout()
plt.show()

print("Key patterns to look for:")
print("- Strong correlations at lag 24 (daily pattern)")
print("- Strong correlations at lag 168 (weekly pattern)")
print("- Gradual decay suggests trend/non-stationarity")
print("- Sharp cutoff in PACF suggests AR order")

## 6. Additional Pattern Analysis

Analyzing year-over-year comparison, monthly patterns, and weekday vs weekend differences.

In [ ]:
# Extract temporal features for analysis
cluster_2_ts['year'] = cluster_2_ts['hour'].dt.year
cluster_2_ts['month'] = cluster_2_ts['hour'].dt.month
cluster_2_ts['day_of_week'] = cluster_2_ts['hour'].dt.dayofweek
cluster_2_ts['hour_of_day'] = cluster_2_ts['hour'].dt.hour
cluster_2_ts['is_weekend'] = cluster_2_ts['day_of_week'].isin([5, 6]).astype(int)

# 1. Year-over-year comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Monthly averages by year
monthly_avg = cluster_2_ts.groupby(['year', 'month'])['total_demand'].mean().reset_index()
for year in [2018, 2019]:
    year_data = monthly_avg[monthly_avg['year'] == year]
    axes[0].plot(year_data['month'], year_data['total_demand'], marker='o', label=str(year), linewidth=2)

axes[0].set_title('Monthly Average Demand: 2018 vs 2019', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Month', fontsize=12)
axes[0].set_ylabel('Average Hourly Demand', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(1, 13))

# Weekday vs Weekend
weekday_pattern = cluster_2_ts.groupby(['hour_of_day', 'is_weekend'])['total_demand'].mean().reset_index()
weekday_data = weekday_pattern[weekday_pattern['is_weekend'] == 0]
weekend_data = weekday_pattern[weekday_pattern['is_weekend'] == 1]

axes[1].plot(weekday_data['hour_of_day'], weekday_data['total_demand'], marker='o', label='Weekday', linewidth=2)
axes[1].plot(weekend_data['hour_of_day'], weekend_data['total_demand'], marker='s', label='Weekend', linewidth=2)
axes[1].set_title('Average Hourly Pattern: Weekday vs Weekend', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour of Day', fontsize=12)
axes[1].set_ylabel('Average Demand', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

## Summary of Time Series Exploratory Analysis

### Key Findings:

#### 1. **Data Overview**
- **Time Period**: January 2018 - December 2019 (2 years)
- **Total Records**: 17,537 hourly observations for Cluster 2 (complete, no gaps)
- **Total Pickups**: 3,421,416 trips
- **Total Dropoffs**: 3,488,980 trips
- **Zero-Activity Hours**: 
  - Hours with 0 pickups: 94 (0.54%)
  - Hours with 0 dropoffs: 90 (0.51%)
  - Hours with 0 total demand: 44 (0.25%)
- **Data Completeness**: ✓ 100% complete (zeros filled for low-activity periods)

#### 2. **Seasonal Decomposition**
- **Trend Component**: Shows clear seasonal variation with peaks in summer months
  - Range: 67.4 to 691.1
  - Winter 2018/2019 shows the lowest demand period
  - Peak demand in summer months (June-September)
- **Seasonal Component**: Strong weekly pattern visible
  - Range: -388.3 to 747.9 (updated with complete data including zeros)
  - Daily and weekly cycles are prominent
  - Wider range reflects true low-activity periods
- **Residual**: Standard deviation of 161.6 (improved with complete data)
  - Lower variability indicates better model fit with zeros included

#### 3. **Stationarity Tests**
- **ADF Test**: Series is STATIONARY (p-value < 0.05)
  - Rejects null hypothesis of unit root
- **KPSS Test**: Series is NON-STATIONARY (p-value < 0.05)
  - Suggests non-stationarity around a constant
- **Interpretation**: Contradictory results suggest the series has **trend stationarity** but not strict stationarity
  - Likely needs differencing or detrending for forecasting models

#### 4. **ACF/PACF Analysis**
- **Strong Daily Cycle**: Clear peaks at lag 24 (hourly pattern repeats daily)
- **Weekly Pattern**: Peaks at multiples of 168 hours (7 days)
- **Persistence**: High autocorrelation persists across many lags
- **Implication**: Suggests SARIMA models would be appropriate (Seasonal ARIMA)

#### 5. **Pattern Analysis**

**Year-over-Year Comparison (2018 vs 2019):**
- 2019 shows higher demand than 2018, especially in summer months
- September 2019 peak is significantly higher than 2018
- Winter months (Jan-Mar, Dec) show lower but consistent demand
- Growth trend visible year-over-year

**Weekday vs Weekend Patterns:**
- **Weekday**: Strong bimodal pattern with peaks at:
  - Morning rush: 8-9 AM
  - Evening rush: 5-6 PM
  - Classic commuter pattern
- **Weekend**: More uniform distribution throughout the day
  - Peak around 1-2 PM
  - Higher baseline during midday hours
  - Lower early morning demand

#### 6. **Implications for Forecasting**
1. Need to account for multiple seasonality (daily, weekly, yearly)
2. Consider differencing to achieve stationarity
3. SARIMA or Prophet models would be suitable
4. Should include features for:
   - Hour of day
   - Day of week
   - Month/season
   - Weekday/weekend indicator
5. Weather and special events likely important external factors
6. Zero-filled hours represent genuine low-activity periods (not missing data)
   - Models should preserve these zeros rather than interpolate
   - Important for capturing true demand variability

---

## 3. Demand Forecasting with Prophet

Now we'll build forecasting models using Facebook's Prophet library. Prophet is well-suited for this task because it:
- Handles multiple seasonalities (daily, weekly, yearly)
- Is robust to missing data and outliers
- Provides interpretable components
- Works well with hourly data

### Modeling Approach:
- **Two separate models**: One for pickups, one for dropoffs
- **Seasonalities**: Daily (24h), weekly (168h), and yearly patterns
- **Train/Val/Test split**: 
  - Train: Jan 2018 - Oct 2019 (22 months)
  - Validation: Nov 2019 (1 month) 
  - Test: Dec 2019 (1 month)
- **No external regressors**: Relying on Prophet's automatic seasonality detection

### 3.1 Install and Import Prophet

In [1]:
# Install prophet if not already installed
# !pip install prophet

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import warnings
warnings.filterwarnings('ignore')

print("Prophet library imported successfully!")

/Users/kristian/Documents/GitHub/Analysis-of-NY-Citi-Bike-stations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


Prophet library imported successfully!


### 3.2 Prepare Data for Prophet

Prophet requires data in a specific format:
- `ds`: datetime column
- `y`: value to forecast

In [ ]:
# Prepare data for Prophet - Cluster 2
# Prophet requires columns: 'ds' (datetime) and 'y' (value)

# Prepare pickups data
df_pickups = cluster_2_ts_indexed.reset_index()[['hour', 'pickups']].rename(
    columns={'hour': 'ds', 'pickups': 'y'}
)

# Prepare dropoffs data
df_dropoffs = cluster_2_ts_indexed.reset_index()[['hour', 'dropoffs']].rename(
    columns={'hour': 'ds', 'dropoffs': 'y'}
)

print("Pickups dataset for Prophet:")
print(df_pickups.head())
print(f"\nShape: {df_pickups.shape}")
print(f"Date range: {df_pickups['ds'].min()} to {df_pickups['ds'].max()}")

print("\n" + "="*70)
print("\nDropoffs dataset for Prophet:")
print(df_dropoffs.head())
print(f"\nShape: {df_dropoffs.shape}")
print(f"Date range: {df_dropoffs['ds'].min()} to {df_dropoffs['ds'].max()}")

### 3.3 Data Split Strategy

We'll use a **rolling daily forecast** approach:
- **Initial Training**: Jan 2018 - Oct 2019 (22 months)
- **Forecast Method**: Each day, predict the next full day (24 hours)
  - After observing all of Wednesday → predict all 24 hours of Thursday
  - After observing all of Thursday → predict all 24 hours of Friday
  - etc.
- **Evaluation Periods**:
  - **Validation**: Nov 2019 (30 days = 30 daily forecasts)
  - **Test**: Dec 2019 (31 days = 31 daily forecasts)

In [ ]:
# Define split dates
initial_train_end = pd.to_datetime('2019-10-31 23:00:00')
val_start = pd.to_datetime('2019-11-01 00:00:00')
val_end = pd.to_datetime('2019-11-30 23:00:00')
test_start = pd.to_datetime('2019-12-01 00:00:00')
test_end = df_pickups['ds'].max()

# Get initial training data (up to end of Oct 2019)
initial_train_pickups = df_pickups[df_pickups['ds'] <= initial_train_end].copy()
initial_train_dropoffs = df_dropoffs[df_dropoffs['ds'] <= initial_train_end].copy()

# Get validation and test data for evaluation
val_pickups = df_pickups[(df_pickups['ds'] >= val_start) & (df_pickups['ds'] <= val_end)].copy()
val_dropoffs = df_dropoffs[(df_dropoffs['ds'] >= val_start) & (df_dropoffs['ds'] <= val_end)].copy()

test_pickups = df_pickups[(df_pickups['ds'] >= test_start) & (df_pickups['ds'] <= test_end)].copy()
test_dropoffs = df_dropoffs[(df_dropoffs['ds'] >= test_start) & (df_dropoffs['ds'] <= test_end)].copy()

# Calculate number of days to forecast
val_days = (val_end - val_start).days + 1
test_days = (test_end - test_start).days + 1

print("Rolling Daily Forecast Setup:")
print("="*70)
print(f"Initial Training Period:")
print(f"  {initial_train_pickups['ds'].min()} to {initial_train_pickups['ds'].max()}")
print(f"  Total: {len(initial_train_pickups):,} hours ({len(initial_train_pickups)//24} days)")

print(f"\nValidation Period (Nov 2019):")
print(f"  {val_start} to {val_end}")
print(f"  Total: {len(val_pickups):,} hours ({val_days} days)")
print(f"  → Will make {val_days} separate 24-hour forecasts")

print(f"\nTest Period (Dec 2019):")
print(f"  {test_start} to {test_end}")
print(f"  Total: {len(test_pickups):,} hours ({test_days} days)")
print(f"  → Will make {test_days} separate 24-hour forecasts")

print("\n" + "="*70)
print("Forecast Process:")
print("  1. Train on all data up to end of day D")
print("  2. Predict all 24 hours of day D+1")
print("  3. Observe actual values for day D+1")
print("  4. Add day D+1 to training data")
print("  5. Repeat for next day")

### 3.4 Implement Rolling Daily Forecast

For each day in the evaluation period, we:
1. Train Prophet on all data up to end of previous day
2. Predict next 24 hours
3. Compare predictions to actual values
4. Add actual values to training set (for next iteration)

This simulates real-world usage: predict tomorrow's demand based on all available history.

In [ ]:
# Rolling Daily Forecast Function
def rolling_daily_forecast(df_full, start_date, end_date, initial_train_end, target_col='y'):
    """
    Perform rolling daily forecasts.
    
    For each day in [start_date, end_date]:
    - Train Prophet on all data up to end of previous day
    - Predict next 24 hours
    - Store predictions and actuals
    
    Parameters:
    - df_full: Full dataset with 'ds' and 'y' columns
    - start_date: First day to forecast
    - end_date: Last day to forecast  
    - initial_train_end: End of initial training period
    - target_col: Name of target column ('pickups' or 'dropoffs')
    
    Returns:
    - DataFrame with predictions and actuals for each forecasted hour
    """
    results = []
    
    # Start with initial training data
    current_train_end = initial_train_end
    
    # Get list of dates to forecast
    forecast_dates = pd.date_range(start_date, end_date, freq='D')
    
    print(f"\nStarting rolling daily forecast for {target_col}...")
    print(f"Forecasting {len(forecast_dates)} days: {start_date.date()} to {end_date.date()}")
    print("="*70)
    
    for i, forecast_date in enumerate(forecast_dates, 1):
        # Define the 24 hours to predict (00:00 to 23:00 of forecast_date)
        day_start = pd.Timestamp(forecast_date.date())
        day_end = day_start + pd.Timedelta(hours=23)
        
        # Get training data up to end of previous day
        train_data = df_full[df_full['ds'] <= current_train_end].copy()
        
        # Get actual values for the forecast day (for comparison)
        actual_data = df_full[(df_full['ds'] >= day_start) & (df_full['ds'] <= day_end)].copy()
        
        if len(actual_data) == 0:
            print(f"  Day {i}: No actual data for {forecast_date.date()}, skipping...")
            continue
        
        # Train Prophet model
        model = Prophet(
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=True,
            seasonality_mode='additive',
            interval_width=0.95
        )
        model.fit(train_data)
        
        # Create future dataframe for the 24 hours we want to predict
        future_hours = pd.DataFrame({'ds': pd.date_range(day_start, day_end, freq='H')})
        
        # Make predictions
        forecast = model.predict(future_hours)
        
        # Store results
        for j in range(len(forecast)):
            results.append({
                'ds': forecast['ds'].iloc[j],
                'actual': actual_data['y'].iloc[j] if j < len(actual_data) else None,
                'predicted': forecast['yhat'].iloc[j],
                'lower_bound': forecast['yhat_lower'].iloc[j],
                'upper_bound': forecast['yhat_upper'].iloc[j],
                'forecast_day': i,
                'hour_of_day': j
            })
        
        # Update training data cutoff to include this day
        current_train_end = day_end
        
        # Progress update every 5 days
        if i % 5 == 0 or i == len(forecast_dates):
            print(f"  Completed {i}/{len(forecast_dates)} days ({i/len(forecast_dates)*100:.1f}%)")
    
    print("✓ Rolling forecast complete!")
    return pd.DataFrame(results)

print("Rolling forecast function defined.")

In [ ]:
# Run rolling forecasts for TEST PERIOD (Dec 2019)
# We'll focus on test period for final evaluation

print("\nRunning Rolling Daily Forecasts for TEST PERIOD (December 2019)")
print("="*70)

# Pickups forecast
forecast_results_pickups = rolling_daily_forecast(
    df_full=df_pickups,
    start_date=test_start,
    end_date=test_end,
    initial_train_end=val_end,  # Train on everything up to end of Nov
    target_col='pickups'
)

# Dropoffs forecast  
forecast_results_dropoffs = rolling_daily_forecast(
    df_full=df_dropoffs,
    start_date=test_start,
    end_date=test_end,
    initial_train_end=val_end,  # Train on everything up to end of Nov
    target_col='dropoffs'
)

print(f"\n" + "="*70)
print(f"Pickups forecast results: {len(forecast_results_pickups)} hours")
print(f"Dropoffs forecast results: {len(forecast_results_dropoffs)} hours")

### 3.5 Evaluate Rolling Forecast Performance

Calculate metrics for the rolling daily forecasts.

In [ ]:
# Calculate performance metrics for rolling forecasts
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def calculate_forecast_metrics(results_df, model_name=""):
    """Calculate metrics for rolling forecast results"""
    # Drop any rows with NaN values (in case last day is incomplete)
    clean_df = results_df.dropna(subset=['actual', 'predicted'])
    
    actual = clean_df['actual'].values
    predicted = clean_df['predicted'].values
    
    print(f"\n{model_name} - Data info:")
    print(f"  Total rows: {len(results_df)}, Valid rows: {len(clean_df)}")
    
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    
    # Calculate MAPE avoiding division by zero
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.sum() > 0 else np.inf
    
    r2 = r2_score(actual, predicted)
    
    print(f"\n{model_name} Performance (Rolling Daily Forecast):")
    print("="*70)
    print(f"  MAE:   {mae:8.2f} trips/hour")
    print(f"  RMSE:  {rmse:8.2f} trips/hour")
    print(f"  MAPE:  {mape:8.2f}%")
    print(f"  R²:    {r2:8.4f}")
    
    # Calculate daily-level metrics
    daily_mae = clean_df.groupby('forecast_day').apply(
        lambda x: mean_absolute_error(x['actual'], x['predicted'])
    )
    
    print(f"\nDaily MAE statistics:")
    print(f"  Mean:   {daily_mae.mean():8.2f}")
    print(f"  Median: {daily_mae.median():8.2f}")
    print(f"  Min:    {daily_mae.min():8.2f} (best day)")
    print(f"  Max:    {daily_mae.max():8.2f} (worst day)")
    print(f"  Std:    {daily_mae.std():8.2f}")
    
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'daily_mae': daily_mae}

# Evaluate pickups
metrics_pickups = calculate_forecast_metrics(forecast_results_pickups, "PICKUPS")

# Evaluate dropoffs
metrics_dropoffs = calculate_forecast_metrics(forecast_results_dropoffs, "DROPOFFS")

### 3.6 Visualize Rolling Forecast Results

Compare predictions to actuals for the daily rolling forecasts.

In [ ]:
# Visualize full test period rolling forecasts
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Pickups
axes[0].plot(forecast_results_pickups['ds'], forecast_results_pickups['actual'], 
             label='Actual', color='black', linewidth=1.5, alpha=0.7)
axes[0].plot(forecast_results_pickups['ds'], forecast_results_pickups['predicted'], 
             label='Rolling Forecast', color='#0072B2', linewidth=1.5, linestyle='--')
axes[0].fill_between(forecast_results_pickups['ds'], 
                       forecast_results_pickups['lower_bound'], 
                       forecast_results_pickups['upper_bound'],
                       alpha=0.3, color='#0072B2', label='95% Confidence Interval')
axes[0].set_title('Pickups: Rolling Daily Forecasts - December 2019', 
                   fontsize=14, fontweight='bold')
axes[0].set_ylabel('Pickups per Hour', fontsize=12)
axes[0].legend(loc='upper left', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Dropoffs
axes[1].plot(forecast_results_dropoffs['ds'], forecast_results_dropoffs['actual'], 
             label='Actual', color='black', linewidth=1.5, alpha=0.7)
axes[1].plot(forecast_results_dropoffs['ds'], forecast_results_dropoffs['predicted'], 
             label='Rolling Forecast', color='#D55E00', linewidth=1.5, linestyle='--')
axes[1].fill_between(forecast_results_dropoffs['ds'], 
                       forecast_results_dropoffs['lower_bound'], 
                       forecast_results_dropoffs['upper_bound'],
                       alpha=0.3, color='#D55E00', label='95% Confidence Interval')
axes[1].set_title('Dropoffs: Rolling Daily Forecasts - December 2019', 
                   fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Dropoffs per Hour', fontsize=12)
axes[1].legend(loc='upper left', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.7 Detailed View: First Week of Forecasts

Zoom into the first week to see individual daily predictions.

In [ ]:
# Zoom in: First week of December (first 7 daily forecasts)
first_week_pickups = forecast_results_pickups[forecast_results_pickups['forecast_day'] <= 7].copy()
first_week_dropoffs = forecast_results_dropoffs[forecast_results_dropoffs['forecast_day'] <= 7].copy()

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Pickups - First week detail
axes[0].plot(first_week_pickups['ds'], first_week_pickups['actual'], 
             label='Actual', color='black', linewidth=2, marker='o', markersize=3)
axes[0].plot(first_week_pickups['ds'], first_week_pickups['predicted'], 
             label='Rolling Forecast', color='#0072B2', linewidth=2, marker='s', markersize=3, linestyle='--')
axes[0].fill_between(first_week_pickups['ds'], 
                       first_week_pickups['lower_bound'], 
                       first_week_pickups['upper_bound'],
                       alpha=0.3, color='#0072B2', label='95% CI')

# Add vertical lines to separate days
for day in range(1, 8):
    day_data = first_week_pickups[first_week_pickups['forecast_day'] == day]
    if len(day_data) > 0:
        day_start = day_data['ds'].min()
        axes[0].axvline(x=day_start, color='gray', linestyle=':', alpha=0.5, linewidth=1)

axes[0].set_title('Pickups: First Week Rolling Forecasts (Each Day Predicted Separately)', 
                   fontsize=14, fontweight='bold')
axes[0].set_ylabel('Pickups per Hour', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Dropoffs - First week detail
axes[1].plot(first_week_dropoffs['ds'], first_week_dropoffs['actual'], 
             label='Actual', color='black', linewidth=2, marker='o', markersize=3)
axes[1].plot(first_week_dropoffs['ds'], first_week_dropoffs['predicted'], 
             label='Rolling Forecast', color='#D55E00', linewidth=2, marker='s', markersize=3, linestyle='--')
axes[1].fill_between(first_week_dropoffs['ds'], 
                       first_week_dropoffs['lower_bound'], 
                       first_week_dropoffs['upper_bound'],
                       alpha=0.3, color='#D55E00', label='95% CI')

# Add vertical lines to separate days
for day in range(1, 8):
    day_data = first_week_dropoffs[first_week_dropoffs['forecast_day'] == day]
    if len(day_data) > 0:
        day_start = day_data['ds'].min()
        axes[1].axvline(x=day_start, color='gray', linestyle=':', alpha=0.5, linewidth=1)

axes[1].set_title('Dropoffs: First Week Rolling Forecasts (Each Day Predicted Separately)', 
                   fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Dropoffs per Hour', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze daily forecast accuracy
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Pickups: Daily MAE over time (drop NaN values)
daily_mae_pickups = forecast_results_pickups.dropna().groupby('forecast_day').apply(
    lambda x: mean_absolute_error(x['actual'], x['predicted'])
).reset_index()
daily_mae_pickups.columns = ['forecast_day', 'mae']

axes[0].plot(daily_mae_pickups['forecast_day'], daily_mae_pickups['mae'], 
             marker='o', linewidth=2, markersize=6, color='#0072B2')
axes[0].axhline(y=daily_mae_pickups['mae'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean MAE = {daily_mae_pickups["mae"].mean():.1f}')
axes[0].set_title('Pickups: Daily Forecast Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Forecast Day Number', fontsize=12)
axes[0].set_ylabel('MAE (trips/hour)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Dropoffs: Daily MAE over time (drop NaN values)
daily_mae_dropoffs = forecast_results_dropoffs.dropna().groupby('forecast_day').apply(
    lambda x: mean_absolute_error(x['actual'], x['predicted'])
).reset_index()
daily_mae_dropoffs.columns = ['forecast_day', 'mae']

axes[1].plot(daily_mae_dropoffs['forecast_day'], daily_mae_dropoffs['mae'], 
             marker='o', linewidth=2, markersize=6, color='#D55E00')
axes[1].axhline(y=daily_mae_dropoffs['mae'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f'Mean MAE = {daily_mae_dropoffs["mae"].mean():.1f}')
axes[1].set_title('Dropoffs: Daily Forecast Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Forecast Day Number', fontsize=12)
axes[1].set_ylabel('MAE (trips/hour)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDaily Forecast Accuracy Analysis:")
print("="*70)
print(f"Pickups - Days with MAE < 50: {(daily_mae_pickups['mae'] < 50).sum()}/{len(daily_mae_pickups)}")
print(f"Dropoffs - Days with MAE < 50: {(daily_mae_dropoffs['mae'] < 50).sum()}/{len(daily_mae_dropoffs)}")

### 3.8 Example: Components from One Daily Forecast

Let's examine the Prophet components for one example day (December 15, 2019) to understand what the model learned.

In [ ]:
# Train one example model to show components (for Dec 15, 2019 forecast)
example_date = pd.to_datetime('2019-12-15')
example_train_end = example_date - pd.Timedelta(days=1, hours=1)  # End of Dec 14

# Get training data up to end of Dec 14
example_train_pickups = df_pickups[df_pickups['ds'] <= example_train_end].copy()

# Train model
print(f"Training example model to forecast {example_date.date()}...")
print(f"Training data: {example_train_pickups['ds'].min()} to {example_train_pickups['ds'].max()}")

example_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    seasonality_mode='additive',
    interval_width=0.95
)
example_model.fit(example_train_pickups)

# Create forecast for Dec 15
example_future = pd.DataFrame({
    'ds': pd.date_range(example_date, example_date + pd.Timedelta(hours=23), freq='H')
})
example_forecast = example_model.predict(example_future)

print("✓ Example forecast generated")

# Plot components
fig = example_model.plot_components(example_forecast, figsize=(14, 10))
fig.suptitle(f'Prophet Components - Example Forecast for {example_date.date()}', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
### 3.9 Residual Analysis

Examine residuals from the rolling forecasts.

In [ ]:
# Calculate residuals for rolling forecasts
forecast_results_pickups['residual'] = forecast_results_pickups['actual'] - forecast_results_pickups['predicted']
forecast_results_dropoffs['residual'] = forecast_results_dropoffs['actual'] - forecast_results_dropoffs['predicted']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Pickups: Residuals over time
axes[0, 0].scatter(forecast_results_pickups['ds'], forecast_results_pickups['residual'], 
                   alpha=0.5, s=10, color='#0072B2')
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_title('Pickups: Residuals Over Time (Rolling Forecasts)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date', fontsize=10)
axes[0, 0].set_ylabel('Residual (Actual - Predicted)', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Pickups: Residuals distribution
axes[0, 1].hist(forecast_results_pickups['residual'], bins=50, color='#0072B2', alpha=0.7, edgecolor='black')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
mean_res_p = forecast_results_pickups['residual'].mean()
std_res_p = forecast_results_pickups['residual'].std()
axes[0, 1].set_title(f'Pickups: Residuals Distribution (μ={mean_res_p:.2f}, σ={std_res_p:.2f})', 
                      fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Residual', fontsize=10)
axes[0, 1].set_ylabel('Frequency', fontsize=10)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Dropoffs: Residuals over time
axes[1, 0].scatter(forecast_results_dropoffs['ds'], forecast_results_dropoffs['residual'], 
                   alpha=0.5, s=10, color='#D55E00')
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_title('Dropoffs: Residuals Over Time (Rolling Forecasts)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Date', fontsize=10)
axes[1, 0].set_ylabel('Residual (Actual - Predicted)', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Dropoffs: Residuals distribution
axes[1, 1].hist(forecast_results_dropoffs['residual'], bins=50, color='#D55E00', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
mean_res_d = forecast_results_dropoffs['residual'].mean()
std_res_d = forecast_results_dropoffs['residual'].std()
axes[1, 1].set_title(f'Dropoffs: Residuals Distribution (μ={mean_res_d:.2f}, σ={std_res_d:.2f})', 
                      fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Residual', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nResidual Statistics (Rolling Forecasts):")
print("="*70)
print(f"Pickups:  Mean = {mean_res_p:7.2f}, Std = {std_res_p:7.2f}")
print(f"Dropoffs: Mean = {mean_res_d:7.2f}, Std = {std_res_d:7.2f}")

### 3.10 Summary: Rolling Daily Forecast Results

**Prophet Rolling Forecast Approach for Cluster 2:**

We implemented a **rolling daily forecast** methodology that matches real-world operational needs:

#### **Forecast Method:**
- **Initial Training**: Jan 2018 - Nov 2019 (23 months of historical data)
- **Test Period**: December 2019 (31 days)
- **Process**: Each day, predict the next complete 24-hour period
  - After observing all of Wednesday → Forecast all 24 hours of Thursday
  - After observing all of Thursday → Forecast all 24 hours of Friday
  - Continue for all 31 days of December

#### **Key Advantages of This Approach:**
1. **Realistic simulation**: Matches how the model would be used in production
2. **Fresh predictions**: Each day uses all available historical data up to that point
3. **No look-ahead bias**: Never uses future information to make predictions
4. **Task-aligned**: Directly addresses project requirement to "predict the next 24 hours"

#### **Model Configuration:**
- **Seasonalities**: Daily (24h rush hours), Weekly (weekdays vs weekends), Yearly (seasonal)
- **Mode**: Additive (constant seasonal effects)
- **Confidence**: 95% prediction intervals

#### **Performance Metrics** *(to be filled after running)*:
- Check the metrics output above for MAE, RMSE, MAPE, and R² scores
- Daily MAE variability shows which days were harder/easier to predict
- Residual analysis reveals systematic patterns in errors

#### **Next Steps:**
1. Compare this rolling forecast performance to validation period (Nov 2019)
2. Identify which days had largest errors (holidays? weather events?)
3. Consider adding external regressors (holidays, weather) to improve accuracy
4. Apply same methodology to other clusters for comparison

## 5. Prophet with Weather Data

Now we'll enhance the Prophet model by adding weather features (temperature, precipitation, wind speed) as external regressors to improve forecast accuracy.

### 5.1 Download Weather Data

In [7]:
# Re-import necessary libraries after kernel restart
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

Importing plotly failed. Interactive plots will not work.


In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("aadimator/nyc-weather-2016-to-2022")

print("Path to dataset files:", path)

/Users/kristian/Documents/GitHub/Analysis-of-NY-Citi-Bike-stations/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/kristian/.cache/kagglehub/datasets/aadimator/nyc-weather-2016-to-2022/versions/1


### 5.2 Load and Explore Weather Data

In [8]:
# List files in the downloaded directory
import os
weather_files = os.listdir(path)
print("Weather dataset files:")
for f in weather_files:
    print(f"  - {f}")
    
# Load the weather data
weather_file = [f for f in weather_files if f.endswith('.csv')][0]
weather_path = os.path.join(path, weather_file)
weather_raw = pd.read_csv(weather_path)

print(f"\nLoaded weather data from: {weather_file}")
print(f"Shape: {weather_raw.shape}")
print(f"\nColumns: {list(weather_raw.columns)}")
print(f"\nFirst few rows:")
weather_raw.head()

Weather dataset files:
  - NYC_Weather_2016_2022.csv

Loaded weather data from: NYC_Weather_2016_2022.csv
Shape: (59760, 10)

Columns: ['time', 'temperature_2m (°C)', 'precipitation (mm)', 'rain (mm)', 'cloudcover (%)', 'cloudcover_low (%)', 'cloudcover_mid (%)', 'cloudcover_high (%)', 'windspeed_10m (km/h)', 'winddirection_10m (°)']

First few rows:


,time,temperature_2m (°C),precipitation (mm),rain (mm),cloudcover (%),cloudcover_low (%),cloudcover_mid (%),cloudcover_high (%),windspeed_10m (km/h),winddirection_10m (°)
0,2016-01-01T00:00,7.6,0.0,0.0,69.0,53.0,0.0,72.0,10.0,296.0
1,2016-01-01T01:00,7.5,0.0,0.0,20.0,4.0,0.0,56.0,9.8,287.0
2,2016-01-01T02:00,7.1,0.0,0.0,32.0,3.0,0.0,99.0,9.7,285.0
3,2016-01-01T03:00,6.6,0.0,0.0,35.0,5.0,0.0,100.0,9.2,281.0
4,2016-01-01T04:00,6.3,0.0,0.0,34.0,4.0,0.0,100.0,9.1,279.0


### 5.3 Check Time Granularity and Filter Data

In [9]:
# Check the datetime column and its granularity
print("Checking time granularity...")
print(f"\nDatetime column name: {[col for col in weather_raw.columns if 'time' in col.lower() or 'date' in col.lower()]}")

# Parse datetime - adjust column name based on actual data
datetime_col = [col for col in weather_raw.columns if 'time' in col.lower() or 'date' in col.lower()][0]
weather_raw[datetime_col] = pd.to_datetime(weather_raw[datetime_col])

# Check time differences to determine granularity
time_diffs = weather_raw[datetime_col].diff().value_counts().head(5)
print(f"\nMost common time differences between consecutive records:")
print(time_diffs)

# Check date range
print(f"\nDate range: {weather_raw[datetime_col].min()} to {weather_raw[datetime_col].max()}")

# Filter to 2018-2019 only
weather_2018_2019 = weather_raw[
    (weather_raw[datetime_col] >= '2018-01-01') & 
    (weather_raw[datetime_col] < '2020-01-01')
].copy()

print(f"\nFiltered to 2018-2019: {len(weather_2018_2019)} records")
print(f"Date range: {weather_2018_2019[datetime_col].min()} to {weather_2018_2019[datetime_col].max()}")

Checking time granularity...

Datetime column name: ['time']

Most common time differences between consecutive records:
time
0 days 01:00:00    59759
Name: count, dtype: int64

Date range: 2016-01-01 00:00:00 to 2022-10-25 23:00:00

Filtered to 2018-2019: 17520 records
Date range: 2018-01-01 00:00:00 to 2019-12-31 23:00:00


### 5.4 Prepare Weather Features

In [10]:
# Extract and rename relevant weather features
weather = weather_2018_2019[[datetime_col, 'temperature_2m (°C)', 
                              'precipitation (mm)', 'windspeed_10m (km/h)']].copy()

# Rename columns for simplicity
weather.columns = ['datetime', 'temperature', 'precipitation', 'wind_speed']

# Set datetime as index
weather['datetime'] = pd.to_datetime(weather['datetime'])
weather = weather.set_index('datetime')

print("Weather features prepared:")
print(f"Shape: {weather.shape}")
print(f"\nFeatures summary:")
print(weather.describe())

# Check for missing values
print(f"\nMissing values:")
print(weather.isnull().sum())

# Handle any missing values by forward fill
if weather.isnull().sum().sum() > 0:
    weather = weather.fillna(method='ffill').fillna(method='bfill')
    print(f"\n✓ Missing values filled")
    
weather.head()

Weather features prepared:
Shape: (17520, 3)

Features summary:
        temperature  precipitation    wind_speed
count  17520.000000   17520.000000  17520.000000
mean      12.718139       0.153716     11.367745
std       10.089998       0.581271      5.817428
min      -18.300000       0.000000      0.400000
25%        4.300000       0.000000      7.200000
50%       12.900000       0.000000     10.400000
75%       21.600000       0.000000     14.400000
max       35.500000      15.300000     45.300000

Missing values:
temperature      0
precipitation    0
wind_speed       0
dtype: int64


,temperature,precipitation,wind_speed
datetime,,,
2018-01-01 00:00:00,-11.1,0.0,9.4
2018-01-01 01:00:00,-11.5,0.0,10.1
2018-01-01 02:00:00,-11.8,0.0,12.3
2018-01-01 03:00:00,-12.2,0.0,13.9
2018-01-01 04:00:00,-12.2,0.0,14.3


### 5.5 Load Bike Data and Merge with Weather

In [11]:
# Load the complete 2018-2019 hourly demand data (prepared earlier)
print("Loading complete hourly demand data for 2018-2019...")
hourly_complete_path = 'data/hourly_demand_2018_2019_complete.csv'
hourly_complete = pd.read_csv(hourly_complete_path)
hourly_complete['hour'] = pd.to_datetime(hourly_complete['hour'])

# Filter for Cluster 2
cluster_2_complete = hourly_complete[hourly_complete['cluster'] == 2.0].copy()

# Prepare pickup and dropoff time series
pickups_full = cluster_2_complete[['hour', 'pickups']].copy()
pickups_full.columns = ['ds', 'y']
pickups_full = pickups_full.set_index('ds')

dropoffs_full = cluster_2_complete[['hour', 'dropoffs']].copy()
dropoffs_full.columns = ['ds', 'y']
dropoffs_full = dropoffs_full.set_index('ds')

print(f"Cluster 2 data loaded:")
print(f"  Pickups: {len(pickups_full)} hours")
print(f"  Dropoffs: {len(dropoffs_full)} hours")
print(f"  Date range: {pickups_full.index.min()} to {pickups_full.index.max()}")

Loading complete hourly demand data for 2018-2019...
Cluster 2 data loaded:
  Pickups: 17537 hours
  Dropoffs: 17537 hours
  Date range: 2018-01-01 00:00:00 to 2020-01-01 16:00:00
Cluster 2 data loaded:
  Pickups: 17537 hours
  Dropoffs: 17537 hours
  Date range: 2018-01-01 00:00:00 to 2020-01-01 16:00:00


In [12]:
# Merge pickups with weather (2018-2019)
pickups_with_weather = pickups_full.join(weather, how='inner')
pickups_with_weather = pickups_with_weather.reset_index()
pickups_with_weather.columns = ['ds', 'y', 'temperature', 'precipitation', 'wind_speed']

# Merge dropoffs with weather (2018-2019)
dropoffs_with_weather = dropoffs_full.join(weather, how='inner')
dropoffs_with_weather = dropoffs_with_weather.reset_index()
dropoffs_with_weather.columns = ['ds', 'y', 'temperature', 'precipitation', 'wind_speed']

print("Merged bike demand with weather (2018-2019):")
print(f"  Pickups: {len(pickups_with_weather)} hours")
print(f"  Dropoffs: {len(dropoffs_with_weather)} hours")
print(f"  Date range: {pickups_with_weather['ds'].min()} to {pickups_with_weather['ds'].max()}")
print(f"\nSample of merged data:")
pickups_with_weather.head()

Merged bike demand with weather (2018-2019):
  Pickups: 17520 hours
  Dropoffs: 17520 hours
  Date range: 2018-01-01 00:00:00 to 2019-12-31 23:00:00

Sample of merged data:


,ds,y,temperature,precipitation,wind_speed
0,2018-01-01 00:00:00,15,-11.1,0.0,9.4
1,2018-01-01 01:00:00,20,-11.5,0.0,10.1
2,2018-01-01 02:00:00,8,-11.8,0.0,12.3
3,2018-01-01 03:00:00,5,-12.2,0.0,13.9
4,2018-01-01 04:00:00,4,-12.2,0.0,14.3


In [13]:
# Check data quality
print("\nData Quality Check:")
print(f"Missing values in pickups_with_weather:")
print(pickups_with_weather.isnull().sum())
print(f"\nMissing values in dropoffs_with_weather:")
print(dropoffs_with_weather.isnull().sum())

# Check weather feature distributions
print(f"\nWeather features summary:")
print(pickups_with_weather[['temperature', 'precipitation', 'wind_speed']].describe())


Data Quality Check:
Missing values in pickups_with_weather:
ds               0
y                0
temperature      0
precipitation    0
wind_speed       0
dtype: int64

Missing values in dropoffs_with_weather:
ds               0
y                0
temperature      0
precipitation    0
wind_speed       0
dtype: int64

Weather features summary:
        temperature  precipitation    wind_speed
count  17520.000000   17520.000000  17520.000000
mean      12.718139       0.153716     11.367745
std       10.089998       0.581271      5.817428
min      -18.300000       0.000000      0.400000
25%        4.300000       0.000000      7.200000
50%       12.900000       0.000000     10.400000
75%       21.600000       0.000000     14.400000
max       35.500000      15.300000     45.300000


### 5.6 Prophet Model with Weather Regressors

Now we'll train Prophet models with temperature, precipitation, and wind speed as additional regressors, using the same rolling daily forecast approach as before.

In [14]:
# Define the same train/test split as the baseline model
# Training: Jan 2018 - Oct 2019
# Validation: Nov 2019 (not used for now)
# Test: Dec 2019

initial_train_end_weather = pd.Timestamp('2019-10-31 23:00:00')
val_start_weather = pd.Timestamp('2019-11-01 00:00:00')
val_end_weather = pd.Timestamp('2019-11-30 23:00:00')
test_start_weather = pd.Timestamp('2019-12-01 00:00:00')
test_end_weather = pd.Timestamp('2019-12-31 23:00:00')

print("Data split for Prophet with weather:") 
print(f"  Initial training: 2018-01-01 to {initial_train_end_weather}")
print(f"  Validation: {val_start_weather} to {val_end_weather} (32 days, not used)")
print(f"  Test: {test_start_weather} to {test_end_weather} (31 days)")

# Calculate number of days for rolling forecast
test_days = (test_end_weather - test_start_weather).days + 1
print(f"\\nWill make {test_days} separate 24-hour forecasts for test period")

Data split for Prophet with weather:
  Initial training: 2018-01-01 to 2019-10-31 23:00:00
  Validation: 2019-11-01 00:00:00 to 2019-11-30 23:00:00 (32 days, not used)
  Test: 2019-12-01 00:00:00 to 2019-12-31 23:00:00 (31 days)
\nWill make 31 separate 24-hour forecasts for test period


In [15]:
# Helper function to suppress Prophet's verbose output
import os
import sys
from contextlib import contextmanager

@contextmanager
def suppress_stdout_stderr():
    """Suppress stdout and stderr"""
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

print("Helper functions defined.")

Helper functions defined.


## 6. Prophet with Enhanced Feature Engineering

Based on Random Forest analysis, we'll add:
- **Lag features**: 1h, 2h, 3h, 24h, 168h (capture recent demand)
- **Rolling averages**: 3h, 6h, 24h (smooth trends)
- **Temporal features**: hour, day_of_week, month, cyclic encoding
- **Domain features**: weekends, holidays, events, seasons, rush hours
- **Weather interactions**: temperature × weekend, precipitation × hour

**Critical**: During 24-hour forecasts, lag/rolling features use predicted values (not future actuals)

### 6.1 Load Events and Holidays Data

In [18]:
# Load holidays data
holidays_df = pd.read_csv('data/holidays_2018_2019.csv')
holidays_df['date'] = pd.to_datetime(holidays_df['date'])

# print(f"Holidays loaded: {len(holidays_df)} holidays")
# print(f"Date range: {holidays_df['date'].min()} to {holidays_df['date'].max()}")
# print("\nHolidays:")
# print(holidays_df[['name', 'date', 'category']].to_string(index=False))

# Load events data (CSV has commas in text, so we need to handle it properly)
events_df = pd.read_csv('data/events_2018_2019.csv', on_bad_lines='skip')
events_df['start_datetime'] = pd.to_datetime(events_df['start_datetime'])
events_df['end_datetime'] = pd.to_datetime(events_df['end_datetime'])
events_df['event_date'] = events_df['start_datetime'].dt.date

# print(f"\n\nEvents loaded: {len(events_df)} events")
# print(f"Date range: {events_df['start_datetime'].min()} to {events_df['end_datetime'].max()}")
# print("\nMajor Events:")
# print(events_df[['name', 'start_datetime', 'category']].head(10).to_string(index=False))

## 7. Prophet with All Clusters and Cross-Target Features

**Key improvements:**
- Use **all clusters** (not just Cluster 2) for more diverse training data
- Add **cross-target lag features**: dropoffs when predicting pickups, and vice versa
- Same comprehensive features as before
- This should close the gap with Random Forest

### 7.1 Load Complete Dataset (All Clusters)

In [22]:
# Load complete hourly demand data for ALL clusters
all_clusters_df = pd.read_csv('data/hourly_demand_2018_2019_complete.csv')
all_clusters_df['hour'] = pd.to_datetime(all_clusters_df['hour'])

# Merge with weather data (weather has datetime as index)
weather_reset = weather.reset_index()
all_clusters_with_weather = all_clusters_df.merge(
    weather_reset[['datetime', 'temperature', 'precipitation', 'wind_speed']],
    left_on='hour',
    right_on='datetime',
    how='inner'
).drop(columns=['datetime'])

# Rename for Prophet format
all_clusters_with_weather = all_clusters_with_weather.rename(columns={'hour': 'ds'})

print(f"Complete dataset loaded:")
print(f"  Shape: {all_clusters_with_weather.shape}")
print(f"  Date range: {all_clusters_with_weather['ds'].min()} to {all_clusters_with_weather['ds'].max()}")
print(f"  Clusters: {sorted(all_clusters_with_weather['cluster'].unique())}")
print(f"  Total observations: {len(all_clusters_with_weather):,}")
print(f"\nSample:")
print(all_clusters_with_weather.head())

Complete dataset loaded:
  Shape: (525600, 9)
  Date range: 2018-01-01 00:00:00 to 2019-12-31 23:00:00
  Clusters: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0)]
  Total observations: 525,600

Sample:
   cluster                  ds  pickups  dropoffs  year  total_demand  \
0      0.0 2018-01-01 00:00:00        3         2  2018             5   
1      0.0 2018-01-01 01:00:00        1         5  2018             6   
2      0.0 2018-01-01 02:00:00        4         1  2018             5   
3      0.0 2018-01-01 0

### 7.2 Enhanced Feature Engineering with Cross-Target Lags

In [23]:
def create_features_with_cross_target(df, target_col='pickups'):
    """
    Create comprehensive features INCLUDING cross-target lags.
    When predicting pickups, include dropoffs lags and vice versa.
    Process data by cluster to maintain proper ordering.
    """
    df = df.copy()
    
    # Determine opposite target
    opposite_col = 'dropoffs' if target_col == 'pickups' else 'pickups'
    
    # Process each cluster separately to maintain time ordering
    cluster_dfs = []
    
    for cluster_id in sorted(df['cluster'].unique()):
        cluster_df = df[df['cluster'] == cluster_id].copy()
        cluster_df = cluster_df.sort_values('ds').reset_index(drop=True)
        
        # Rename target column to 'y'
        cluster_df['y'] = cluster_df[target_col]
        
        # ========================================================================
        # 1. SAME-TARGET LAG FEATURES
        # ========================================================================
        cluster_df['lag_1h'] = cluster_df['y'].shift(1)
        cluster_df['lag_2h'] = cluster_df['y'].shift(2)
        cluster_df['lag_3h'] = cluster_df['y'].shift(3)
        cluster_df['lag_24h'] = cluster_df['y'].shift(24)
        cluster_df['lag_168h'] = cluster_df['y'].shift(168)
        
        # ========================================================================
        # 2. CROSS-TARGET LAG FEATURES (KEY ADDITION!)
        # ========================================================================
        cluster_df['opposite_lag_1h'] = cluster_df[opposite_col].shift(1)
        cluster_df['opposite_lag_2h'] = cluster_df[opposite_col].shift(2)
        cluster_df['opposite_lag_3h'] = cluster_df[opposite_col].shift(3)
        cluster_df['opposite_lag_24h'] = cluster_df[opposite_col].shift(24)
        cluster_df['opposite_lag_168h'] = cluster_df[opposite_col].shift(168)
        
        # ========================================================================
        # 3. ROLLING AVERAGES (same and opposite)
        # ========================================================================
        cluster_df['rolling_3h'] = cluster_df['y'].shift(1).rolling(window=3, min_periods=1).mean()
        cluster_df['rolling_6h'] = cluster_df['y'].shift(1).rolling(window=6, min_periods=1).mean()
        cluster_df['rolling_24h'] = cluster_df['y'].shift(1).rolling(window=24, min_periods=1).mean()
        
        cluster_df['opposite_rolling_3h'] = cluster_df[opposite_col].shift(1).rolling(window=3, min_periods=1).mean()
        cluster_df['opposite_rolling_24h'] = cluster_df[opposite_col].shift(1).rolling(window=24, min_periods=1).mean()
        
        # ========================================================================
        # 4. TEMPORAL FEATURES
        # ========================================================================
        cluster_df['hour'] = cluster_df['ds'].dt.hour
        cluster_df['day_of_week'] = cluster_df['ds'].dt.dayofweek
        cluster_df['month'] = cluster_df['ds'].dt.month
        cluster_df['day_of_year'] = cluster_df['ds'].dt.dayofyear
        
        # Cyclic encoding
        cluster_df['hour_sin'] = np.sin(2 * np.pi * cluster_df['hour'] / 24)
        cluster_df['hour_cos'] = np.cos(2 * np.pi * cluster_df['hour'] / 24)
        cluster_df['dow_sin'] = np.sin(2 * np.pi * cluster_df['day_of_week'] / 7)
        cluster_df['dow_cos'] = np.cos(2 * np.pi * cluster_df['day_of_week'] / 7)
        
        # ========================================================================
        # 5. DOMAIN FEATURES
        # ========================================================================
        cluster_df['is_weekend'] = (cluster_df['day_of_week'] >= 5).astype(int)
        cluster_df['is_morning_rush'] = cluster_df['hour'].between(7, 9).astype(int)
        cluster_df['is_evening_rush'] = cluster_df['hour'].between(16, 19).astype(int)
        cluster_df['weekday_morning_rush'] = ((cluster_df['is_morning_rush'] == 1) & (cluster_df['is_weekend'] == 0)).astype(int)
        cluster_df['weekday_evening_rush'] = ((cluster_df['is_evening_rush'] == 1) & (cluster_df['is_weekend'] == 0)).astype(int)
        
        # Season
        month = cluster_df['month']
        cluster_df['season_winter'] = month.isin([12, 1, 2]).astype(int)
        cluster_df['season_spring'] = month.isin([3, 4, 5]).astype(int)
        cluster_df['season_summer'] = month.isin([6, 7, 8]).astype(int)
        cluster_df['season_fall'] = month.isin([9, 10, 11]).astype(int)
        
        # Holidays and events
        cluster_df['date_only'] = cluster_df['ds'].dt.date
        cluster_df['is_holiday'] = cluster_df['date_only'].isin(holidays_df['date'].dt.date).astype(int)
        event_dates = set(events_df['event_date'])
        cluster_df['is_special_event'] = cluster_df['date_only'].isin(event_dates).astype(int)
        
        # ========================================================================
        # 6. WEATHER INTERACTIONS
        # ========================================================================
        cluster_df['temp_weekend'] = cluster_df['temperature'] * cluster_df['is_weekend']
        cluster_df['precip_hour'] = cluster_df['precipitation'] * cluster_df['hour']
        cluster_df['temp_morning_rush'] = cluster_df['temperature'] * cluster_df['is_morning_rush']
        
        # ========================================================================
        # 7. NET FLOW (difference between pickups and dropoffs)
        # ========================================================================
        cluster_df['net_flow'] = cluster_df['pickups'] - cluster_df['dropoffs']
        cluster_df['net_flow_lag_24h'] = cluster_df['net_flow'].shift(24)
        
        cluster_df = cluster_df.drop(columns=['date_only', 'net_flow'])
        cluster_dfs.append(cluster_df)
    
    # Combine all clusters
    result = pd.concat(cluster_dfs, ignore_index=True)
    
    # Fill missing values in lag features
    lag_cols = [col for col in result.columns if 'lag' in col or 'rolling' in col or 'net_flow' in col]
    result[lag_cols] = result[lag_cols].fillna(0)
    
    return result

print("Cross-target feature engineering function defined.")

Cross-target feature engineering function defined.


In [24]:
# Create feature sets for both targets
print("Creating features for PICKUPS (with dropoffs cross-lags)...")
all_pickups_data = create_features_with_cross_target(all_clusters_with_weather.copy(), target_col='pickups')

print("\nCreating features for DROPOFFS (with pickups cross-lags)...")
all_dropoffs_data = create_features_with_cross_target(all_clusters_with_weather.copy(), target_col='dropoffs')

print(f"\n✓ Feature engineering complete:")
print(f"  Pickups dataset: {all_pickups_data.shape}")
print(f"  Dropoffs dataset: {all_dropoffs_data.shape}")
print(f"  Clusters: {sorted(all_pickups_data['cluster'].unique())}")

# Check features
feature_cols_all = [col for col in all_pickups_data.columns if col not in ['ds', 'y', 'cluster', 'pickups', 'dropoffs']]
print(f"\nTotal features: {len(feature_cols_all)}")
print(f"Features: {feature_cols_all}")

Creating features for PICKUPS (with dropoffs cross-lags)...

Creating features for DROPOFFS (with pickups cross-lags)...

Creating features for DROPOFFS (with pickups cross-lags)...

✓ Feature engineering complete:
  Pickups dataset: (525600, 48)
  Dropoffs dataset: (525600, 48)
  Clusters: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), np.float64(26.0), np.float64(27.0), np.float64(28.0), np.float64(29.0)]

Total features: 43
Features: ['year', 'total_demand', 'temperature', 'precipitation', 'wind_speed', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_168h', 'opposite_lag_1h', 'opp

### 7.3 Train/Test Split (Matching Random Forest)

In [25]:
# Use same split as Random Forest: 60% train, 10% val, 30% test
unique_dates = sorted(all_pickups_data['ds'].dt.date.unique())
n_dates = len(unique_dates)

train_end_idx = int(n_dates * 0.60)
val_end_idx = int(n_dates * 0.70)

train_end_date = pd.Timestamp(unique_dates[train_end_idx - 1])
val_start_date = pd.Timestamp(unique_dates[train_end_idx])
val_end_date = pd.Timestamp(unique_dates[val_end_idx - 1])
test_start_date = pd.Timestamp(unique_dates[val_end_idx])
test_end_date = pd.Timestamp(unique_dates[-1])

# For Prophet, combine train + val for training
train_val_end_date = val_end_date

print(f"Dataset split (matching Random Forest):")
print(f"  Total days: {n_dates}")
print(f"  Train+Val: {unique_dates[0]} to {val_end_date.date()} (70%)")
print(f"  Test: {test_start_date.date()} to {test_end_date.date()} (30%)")
print(f"\nTest period: {(test_end_date - test_start_date).days + 1} days")

Dataset split (matching Random Forest):
  Total days: 730
  Train+Val: 2018-01-01 to 2019-05-25 (70%)
  Test: 2019-05-26 to 2019-12-31 (30%)

Test period: 220 days


### 7.4 Train SARIMAX Model

In [26]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pickle
import warnings
warnings.filterwarnings('ignore')

def train_sarimax_model(df_full, train_val_end, target_col, model_save_path=None):
    """
    Train a SARIMAX model on all data up to train_val_end.
    
    SARIMAX: Seasonal AutoRegressive Integrated Moving Average with eXogenous regressors
    - Handles seasonality (weekly/yearly patterns)
    - Supports external features (weather, lags, etc.)
    - Standard statistical approach (part of statsmodels)
    
    Parameters:
    -----------
    df_full : DataFrame
        Full dataset with features (must include 'ds', 'y', and exogenous variables)
    train_val_end : Timestamp
        End of training period
    target_col : str
        'pickups' or 'dropoffs'
    model_save_path : str, optional
        Path to save the trained model
    
    Returns:
    --------
    model : SARIMAXResults
        Trained SARIMAX model
    regressor_cols : list
        List of exogenous variable column names
    """
    # Get regressor columns (exclude metadata and target columns)
    regressor_cols = [col for col in df_full.columns 
                      if col not in ['ds', 'y', 'cluster', 'pickups', 'dropoffs']]
    
    print(f"Training SARIMAX model for {target_col}...")
    print(f"Training data: up to {train_val_end}")
    print(f"Number of exogenous variables: {len(regressor_cols)}")
    
    # Get training data
    train_data = df_full[df_full['ds'] <= train_val_end].copy()
    train_data = train_data.sort_values('ds').reset_index(drop=True)
    print(f"Training samples: {len(train_data):,}")
    
    # Prepare data
    y_train = train_data['y'].values
    X_train = train_data[regressor_cols].values
    
    # Initialize SARIMAX
    # order=(p,d,q): AR order, differencing, MA order
    # seasonal_order=(P,D,Q,s): seasonal AR, seasonal differencing, seasonal MA, seasonality period
    # For hourly data with weekly seasonality: s=168 (24 hours * 7 days)
    print("Fitting SARIMAX model...")
    print("  Order: (1,0,1) - AR(1), no differencing, MA(1)")
    print("  Seasonal order: (1,0,1,168) - Weekly seasonality")
    
    model = SARIMAX(
        y_train,
        exog=X_train,
        order=(1, 0, 1),  # (p,d,q) - simple ARMA
        seasonal_order=(1, 0, 1, 168),  # (P,D,Q,s) - weekly seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    # Fit model
    fitted_model = model.fit(disp=False, maxiter=100)
    
    print("✓ Model training complete")
    print(f"  AIC: {fitted_model.aic:.2f}")
    print(f"  BIC: {fitted_model.bic:.2f}")
    
    # Save model if path provided
    if model_save_path:
        with open(model_save_path, 'wb') as f:
            pickle.dump(fitted_model, f)
        print(f"✓ Model saved to {model_save_path}")
    
    return fitted_model, regressor_cols

print("SARIMAX training function defined.")

SARIMAX training function defined.


### 7.5 Rolling Daily Forecast with Sequential Predictions

In [27]:
def rolling_daily_forecast_sequential(df_pickups, df_dropoffs, pickups_model, dropoffs_model, 
                                      regressor_cols_pickups, regressor_cols_dropoffs,
                                      start_date, end_date):
    """
    Rolling daily forecast using sequential hour-by-hour predictions for BOTH pickups and dropoffs.
    Uses predicted values for intra-day lags/rolling features to avoid data leakage.
    
    Parameters:
    -----------
    df_pickups : DataFrame
        Full pickups dataset with actual features (for initialization)
    df_dropoffs : DataFrame
        Full dropoffs dataset with actual features (for initialization)
    pickups_model : Prophet
        Trained Prophet model for pickups
    dropoffs_model : Prophet
        Trained Prophet model for dropoffs
    regressor_cols_pickups : list
        List of regressor column names for pickups model
    regressor_cols_dropoffs : list
        List of regressor column names for dropoffs model
    start_date : Timestamp
        Start of forecast period
    end_date : Timestamp
        End of forecast period
    
    Returns:
    --------
    tuple: (pickups_results_df, dropoffs_results_df)
    """
    results_pickups = []
    results_dropoffs = []
    forecast_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    total_days = len(forecast_dates)
    # clusters = sorted(df_pickups['cluster'].unique())
    clusters = [2]  # TEST: Only predict cluster 2 for faster execution
    
    print(f"\n{'='*70}")
    print(f"Rolling Daily Forecast: PICKUPS & DROPOFFS")
    print(f"{'='*70}")
    print(f"Forecast period: {start_date.date()} to {end_date.date()}")
    print(f"Total days: {total_days}")
    print(f"Clusters: {len(clusters)}")
    print(f"**TEST MODE: Only predicting cluster {clusters[0]}**")
    
    # Create working dataframes to track predictions for both targets
    df_working_pickups = df_pickups.copy()
    df_working_dropoffs = df_dropoffs.copy()
    
    # For each forecast day
    for day_idx, forecast_day in enumerate(forecast_dates, 1):
        day_start = forecast_day
        day_end = forecast_day + pd.Timedelta(hours=23)
        
        # Get all hours for this day
        forecast_hours = pd.date_range(start=day_start, end=day_end, freq='h')
        
        # Store predictions for this day (separate for pickups and dropoffs)
        day_predictions_pickups = {}
        day_predictions_dropoffs = {}
        
        # Predict hour by hour sequentially
        for hour_idx, current_hour in enumerate(forecast_hours):
            
            # Predict for each cluster at this hour (BOTH pickups and dropoffs)
            for cluster_id in clusters:
                
                # ========== PICKUPS PREDICTION ==========
                mask_p = (df_working_pickups['ds'] == current_hour) & (df_working_pickups['cluster'] == cluster_id)
                if mask_p.any():
                    row_p = df_working_pickups[mask_p].iloc[0].copy()
                    
                    # Update short-term lag features using predictions from earlier hours TODAY
                    if hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_1h'] = day_predictions_pickups[key]
                    
                    if hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_2h'] = day_predictions_pickups[key]
                    
                    if hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_3h'] = day_predictions_pickups[key]
                    
                    # Update rolling averages
                    vals_3h = []
                    for h in range(1, 4):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_3h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_3h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_3h) >= 1:
                        row_p['rolling_3h'] = np.mean(vals_3h)
                    
                    vals_6h = []
                    for h in range(1, 7):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_6h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_6h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_6h) >= 1:
                        row_p['rolling_6h'] = np.mean(vals_6h)
                    
                    # rolling_24h: rolling 24h average of own target
                    vals_24h = []
                    for h in range(1, 25):  # Look back 1-24 hours
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_24h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_24h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_24h) >= 1:
                        row_p['rolling_24h'] = np.mean(vals_24h)
                    
                    # Cross-target lags: use dropoffs predictions if available
                    if 'dropoffs_lag_1h' in regressor_cols_pickups and hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_1h'] = day_predictions_dropoffs[key]
                    
                    if 'dropoffs_lag_2h' in regressor_cols_pickups and hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_2h'] = day_predictions_dropoffs[key]
                    
                    if 'dropoffs_lag_3h' in regressor_cols_pickups and hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_3h'] = day_predictions_dropoffs[key]
                    
                    # opposite_rolling_3h: rolling 3h average of opposite target (dropoffs)
                    if 'dropoffs_rolling_3h' in regressor_cols_pickups:
                        vals_3h = []
                        for h in range(1, 4):  # Look back 1-3 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_dropoffs:
                                vals_3h.append(day_predictions_dropoffs[key])
                            else:
                                hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_3h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                        if len(vals_3h) >= 1:
                            row_p['dropoffs_rolling_3h'] = np.mean(vals_3h)
                    
                    # opposite_rolling_24h: rolling 24h average of opposite target (dropoffs)
                    if 'dropoffs_rolling_24h' in regressor_cols_pickups:
                        vals_24h = []
                        for h in range(1, 25):  # Look back 1-24 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_dropoffs:
                                vals_24h.append(day_predictions_dropoffs[key])
                            else:
                                hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_24h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                        if len(vals_24h) >= 1:
                            row_p['dropoffs_rolling_24h'] = np.mean(vals_24h)
                    
                    # Prepare exogenous variables for SARIMAX prediction
                    exog_values_p = [row_p[col] for col in regressor_cols_pickups]
                    
                    # Make pickups prediction using SARIMAX
                    forecast_p = pickups_model.forecast(steps=1, exog=[exog_values_p])
                    predicted_value_p = forecast_p.iloc[0]
                    day_predictions_pickups[(current_hour, cluster_id)] = predicted_value_p
                    
                    results_pickups.append({
                        'ds': current_hour,
                        'cluster': cluster_id,
                        'actual': row_p['y'],
                        'predicted': predicted_value_p,
                        'forecast_day': day_idx,
                        'hour_of_day': current_hour.hour
                    })
                
                # ========== DROPOFFS PREDICTION ==========
                mask_d = (df_working_dropoffs['ds'] == current_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                if mask_d.any():
                    row_d = df_working_dropoffs[mask_d].iloc[0].copy()
                    
                    # Update short-term lag features using predictions from earlier hours TODAY
                    if hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_1h'] = day_predictions_dropoffs[key]
                    
                    if hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_2h'] = day_predictions_dropoffs[key]
                    
                    if hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_3h'] = day_predictions_dropoffs[key]
                    
                    # Update rolling averages
                    vals_3h = []
                    for h in range(1, 4):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_3h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_3h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_3h) >= 1:
                        row_d['rolling_3h'] = np.mean(vals_3h)
                    
                    # rolling_6h: rolling 6h average of own target
                    vals_6h = []
                    for h in range(1, 7):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_6h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_6h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_6h) >= 1:
                        row_d['rolling_6h'] = np.mean(vals_6h)
                    
                    # rolling_24h: rolling 24h average of own target
                    vals_24h = []
                    for h in range(1, 25):  # Look back 1-24 hours
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_24h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_24h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_24h) >= 1:
                        row_d['rolling_24h'] = np.mean(vals_24h)
                    
                    # Cross-target lags: use pickups predictions if available
                    if 'pickups_lag_1h' in regressor_cols_dropoffs and hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_1h'] = day_predictions_pickups[key]
                    
                    if 'pickups_lag_2h' in regressor_cols_dropoffs and hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_2h'] = day_predictions_pickups[key]
                    
                    if 'pickups_lag_3h' in regressor_cols_dropoffs and hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_3h'] = day_predictions_pickups[key]
                    
                    # opposite_rolling_3h: rolling 3h average of opposite target (pickups)
                    if 'pickups_rolling_3h' in regressor_cols_dropoffs:
                        vals_3h = []
                        for h in range(1, 4):  # Look back 1-3 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_pickups:
                                vals_3h.append(day_predictions_pickups[key])
                            else:
                                hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_3h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                        if len(vals_3h) >= 1:
                            row_d['pickups_rolling_3h'] = np.mean(vals_3h)
                    
                    # opposite_rolling_24h: rolling 24h average of opposite target (pickups)
                    if 'pickups_rolling_24h' in regressor_cols_dropoffs:
                        vals_24h = []
                        for h in range(1, 25):  # Look back 1-24 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_pickups:
                                vals_24h.append(day_predictions_pickups[key])
                            else:
                                hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_24h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                        if len(vals_24h) >= 1:
                            row_d['pickups_rolling_24h'] = np.mean(vals_24h)
                    
                    # Prepare exogenous variables for SARIMAX prediction
                    exog_values_d = [row_d[col] for col in regressor_cols_dropoffs]
                    
                    # Make dropoffs prediction using SARIMAX
                    forecast_d = dropoffs_model.forecast(steps=1, exog=[exog_values_d])
                    predicted_value_d = forecast_d.iloc[0]
                    day_predictions_dropoffs[(current_hour, cluster_id)] = predicted_value_d
                    
                    results_dropoffs.append({
                        'ds': current_hour,
                        'cluster': cluster_id,
                        'actual': row_d['y'],
                        'predicted': predicted_value_d,
                        'forecast_day': day_idx,
                        'hour_of_day': current_hour.hour
                    })
        
        # NOTE: We do NOT update df_working['y'] with predictions
        # Long-term lags (lag_24h, lag_168h) should use ACTUAL values from previous days
        # Short-term lags (lag_1h, lag_2h, lag_3h) and rolling means use predictions via day_predictions dict
        
        # Progress update
        if day_idx % 5 == 0 or day_idx == total_days:
            pct = (day_idx / total_days) * 100
            print(f"Completed {day_idx}/{total_days} days ({pct:.1f}%)")
    
    print("✓ Sequential predictions complete for both pickups and dropoffs")
    return pd.DataFrame(results_pickups), pd.DataFrame(results_dropoffs)

print("Sequential rolling forecast function defined.")
    

Sequential rolling forecast function defined.


### 7.6 Train Models and Run Sequential Forecasts

In [ ]:
# Train Prophet models for pickups and dropoffs
print("="*70)
print("TRAINING PROPHET MODELS")
print("="*70)

model_save_path='sarimax_model_pickups_v1.pkl'

# Train pickups model with SARIMAX
pickups_model, regressor_cols_pickups = train_sarimax_model(
    all_pickups_data,
    train_val_end_date,
    'pickups',
    model_save_path=model_save_path
)
import pickle
with open(model_save_path+"_list", "wb") as f:
    pickle.dump(regressor_cols_pickups, f)


TRAINING PROPHET MODELS
Training SARIMAX model for pickups...
Training data: up to 2019-05-25 00:00:00
Number of exogenous variables: 43
Training samples: 366,510
Fitting SARIMAX model...
  Order: (1,0,1) - AR(1), no differencing, MA(1)
  Seasonal order: (1,0,1,168) - Weekly seasonality
Training samples: 366,510
Fitting SARIMAX model...
  Order: (1,0,1) - AR(1), no differencing, MA(1)
  Seasonal order: (1,0,1,168) - Weekly seasonality


In [ ]:
print("\n" + "-"*70 + "\n")

model_save_path='sarimax_model_dropoffs_v1.pkl'

# Train dropoffs model with SARIMAX
dropoffs_model, regressor_cols_dropoffs = train_sarimax_model(
    all_dropoffs_data,
    train_val_end_date,
    'dropoffs',
    model_save_path=model_save_path
)

import pickle
with open(model_save_path+"_list", "wb") as f:
    pickle.dump(regressor_cols_dropoffs, f)

print("\n" + "="*70)
print("BOTH MODELS TRAINED SUCCESSFULLY")
print("="*70)


----------------------------------------------------------------------

Training Prophet model for dropoffs...
Training data: up to 2019-05-25 00:00:00
Number of regressors: 43
['year', 'total_demand', 'temperature', 'precipitation', 'wind_speed', 'lag_1h', 'lag_2h', 'lag_3h', 'lag_24h', 'lag_168h', 'opposite_lag_1h', 'opposite_lag_2h', 'opposite_lag_3h', 'opposite_lag_24h', 'opposite_lag_168h', 'rolling_3h', 'rolling_6h', 'rolling_24h', 'opposite_rolling_3h', 'opposite_rolling_24h', 'hour', 'day_of_week', 'month', 'day_of_year', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'weekday_morning_rush', 'weekday_evening_rush', 'season_winter', 'season_spring', 'season_summer', 'season_fall', 'is_holiday', 'is_special_event', 'temp_weekend', 'precip_hour', 'temp_morning_rush', 'net_flow_lag_24h']
Training samples: 366,510
Fitting model...
Training samples: 366,510
Fitting model...
✓ Model training complete
✓ Model saved to prophet_model_dr

In [ ]:
import pickle

model_save_path_pickups = "sarimax_model_pickups_v1.pkl" 
model_save_path_dropoffs = "sarimax_model_dropoffs_v1.pkl"

with open(model_save_path_pickups, 'rb') as f:
    pickups_model = pickle.load(f)

with open(model_save_path_dropoffs, 'rb') as f:
    dropoffs_model = pickle.load(f)

with open(model_save_path_pickups+"_list", 'rb') as f:
    regressor_cols_pickups = pickle.load(f)

with open(model_save_path_dropoffs+"_list", 'rb') as f:
    regressor_cols_dropoffs = pickle.load(f)

# Run sequential rolling forecasts for both pickups and dropoffs simultaneously
forecast_results_pickups_sequential, forecast_results_dropoffs_sequential = rolling_daily_forecast_sequential(
    all_pickups_data,
    all_dropoffs_data,
    pickups_model,
    dropoffs_model,
    regressor_cols_pickups,
    regressor_cols_dropoffs,
    test_start_date,
    test_end_date
)

print("\n✓ All sequential forecasts complete!")


Rolling Daily Forecast: PICKUPS & DROPOFFS
Forecast period: 2019-05-26 to 2019-12-31
Total days: 220
Clusters: 1
**TEST MODE: Only predicting cluster 2**
Completed 5/220 days (2.3%)
Completed 5/220 days (2.3%)
Completed 10/220 days (4.5%)
Completed 10/220 days (4.5%)
Completed 15/220 days (6.8%)
Completed 15/220 days (6.8%)
Completed 20/220 days (9.1%)
Completed 20/220 days (9.1%)
Completed 25/220 days (11.4%)
Completed 25/220 days (11.4%)
Completed 30/220 days (13.6%)
Completed 30/220 days (13.6%)
Completed 35/220 days (15.9%)
Completed 35/220 days (15.9%)
Completed 40/220 days (18.2%)
Completed 40/220 days (18.2%)
Completed 45/220 days (20.5%)
Completed 45/220 days (20.5%)
Completed 50/220 days (22.7%)
Completed 50/220 days (22.7%)
Completed 55/220 days (25.0%)
Completed 55/220 days (25.0%)
Completed 60/220 days (27.3%)
Completed 60/220 days (27.3%)
Completed 65/220 days (29.5%)
Completed 65/220 days (29.5%)
Completed 70/220 days (31.8%)
Completed 70/220 days (31.8%)
Completed 75/22

In [ ]:
def train_prophet_model(df_full, train_val_end, target_col, model_save_path="Default_path"):
    """
    Train a Prophet model on all data up to train_val_end.
    
    Parameters:
    -----------
    df_full : DataFrame
        Full dataset with features
    train_val_end : Timestamp
        End of training period
    target_col : str
        'pickups' or 'dropoffs'
    model_save_path : str, optional
        Path to save the trained model
    
    Returns:
    --------
    model : Prophet
        Trained Prophet model
    regressor_cols : list
        List of regressor column names
    """
    # Get regressor columns (exclude metadata and target columns)
    regressor_cols = [col for col in df_full.columns 
                      if col not in ['ds', 'y', 'cluster', 'pickups', 'dropoffs']]
    
    print(f"Training Prophet model for {target_col}...")
    print(f"Training data: up to {train_val_end}")
    print(f"Number of regressors: {len(regressor_cols)}")
    print(regressor_cols)
    
    # Get training data
    train_data = df_full[df_full['ds'] <= train_val_end].copy()
    print(f"Training samples: {len(train_data):,}")
    
    # Initialize Prophet
    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode='additive',
        interval_width=0.95
    )
    
    # Add all regressors
    for col in regressor_cols:
        model.add_regressor(col)
        # Fit model
    print("Fitting model...")
    with suppress_stdout_stderr():
        model.fit(train_data)
    
    print("✓ Model training complete")
    
    # Save model if path provided
    if model_save_path:
        import pickle
        with open(model_save_path, 'wb') as f:
            pickle.dump(model, f)
        print(f"✓ Model saved to {model_save_path}")
    
    return model, regressor_cols

print("Prophet training function defined.")

In [ ]:
def rolling_daily_forecast_sequential_prohpet(df_pickups, df_dropoffs, pickups_model, dropoffs_model, 
                                      regressor_cols_pickups, regressor_cols_dropoffs,
                                      start_date, end_date):
    """
    Rolling daily forecast using sequential hour-by-hour predictions for BOTH pickups and dropoffs.
    Uses predicted values for intra-day lags/rolling features to avoid data leakage.
    
    Parameters:
    -----------
    df_pickups : DataFrame
        Full pickups dataset with actual features (for initialization)
    df_dropoffs : DataFrame
        Full dropoffs dataset with actual features (for initialization)
    pickups_model : Prophet
        Trained Prophet model for pickups
    dropoffs_model : Prophet
        Trained Prophet model for dropoffs
    regressor_cols_pickups : list
        List of regressor column names for pickups model
    regressor_cols_dropoffs : list
        List of regressor column names for dropoffs model
    start_date : Timestamp
        Start of forecast period
    end_date : Timestamp
        End of forecast period
    
    Returns:
    --------
    tuple: (pickups_results_df, dropoffs_results_df)
    """
    results_pickups = []
    results_dropoffs = []
    forecast_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    total_days = len(forecast_dates)
    # clusters = sorted(df_pickups['cluster'].unique())
    clusters = [2]  # TEST: Only predict cluster 2 for faster execution
    
    print(f"\n{'='*70}")
    print(f"Rolling Daily Forecast: PICKUPS & DROPOFFS")
    print(f"{'='*70}")
    print(f"Forecast period: {start_date.date()} to {end_date.date()}")
    print(f"Total days: {total_days}")
    print(f"Clusters: {len(clusters)}")
    print(f"**TEST MODE: Only predicting cluster {clusters[0]}**")
    
    # Create working dataframes to track predictions for both targets
    df_working_pickups = df_pickups.copy()
    df_working_dropoffs = df_dropoffs.copy()
    
    # For each forecast day
    for day_idx, forecast_day in enumerate(forecast_dates, 1):
        day_start = forecast_day
        day_end = forecast_day + pd.Timedelta(hours=23)
        
        # Get all hours for this day
        forecast_hours = pd.date_range(start=day_start, end=day_end, freq='h')
        
        # Store predictions for this day (separate for pickups and dropoffs)
        day_predictions_pickups = {}
        day_predictions_dropoffs = {}
        
        # Predict hour by hour sequentially
        for hour_idx, current_hour in enumerate(forecast_hours):
            
            # Predict for each cluster at this hour (BOTH pickups and dropoffs)
            for cluster_id in clusters:
                
                # ========== PICKUPS PREDICTION ==========
                mask_p = (df_working_pickups['ds'] == current_hour) & (df_working_pickups['cluster'] == cluster_id)
                if mask_p.any():
                    row_p = df_working_pickups[mask_p].iloc[0].copy()
                    
                    # Update short-term lag features using predictions from earlier hours TODAY
                    if hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_1h'] = day_predictions_pickups[key]
                    
                    if hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_2h'] = day_predictions_pickups[key]
                    
                    if hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_p['lag_3h'] = day_predictions_pickups[key]
                    
                    # Update rolling averages
                    vals_3h = []
                    for h in range(1, 4):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_3h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_3h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_3h) >= 1:
                        row_p['rolling_3h'] = np.mean(vals_3h)
                    
                    vals_6h = []
                    for h in range(1, 7):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_6h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_6h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_6h) >= 1:
                        row_p['rolling_6h'] = np.mean(vals_6h)
                    
                    # rolling_24h: rolling 24h average of own target
                    vals_24h = []
                    for h in range(1, 25):  # Look back 1-24 hours
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            vals_24h.append(day_predictions_pickups[key])
                        else:
                            hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_24h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                    if len(vals_24h) >= 1:
                        row_p['rolling_24h'] = np.mean(vals_24h)
                    
                    # Cross-target lags: use dropoffs predictions if available
                    if 'dropoffs_lag_1h' in regressor_cols_pickups and hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_1h'] = day_predictions_dropoffs[key]
                    
                    if 'dropoffs_lag_2h' in regressor_cols_pickups and hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_2h'] = day_predictions_dropoffs[key]
                    
                    if 'dropoffs_lag_3h' in regressor_cols_pickups and hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_p['dropoffs_lag_3h'] = day_predictions_dropoffs[key]
                    
                    # opposite_rolling_3h: rolling 3h average of opposite target (dropoffs)
                    if 'dropoffs_rolling_3h' in regressor_cols_pickups:
                        vals_3h = []
                        for h in range(1, 4):  # Look back 1-3 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_dropoffs:
                                vals_3h.append(day_predictions_dropoffs[key])
                            else:
                                hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_3h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                        if len(vals_3h) >= 1:
                            row_p['dropoffs_rolling_3h'] = np.mean(vals_3h)
                    
                    # opposite_rolling_24h: rolling 24h average of opposite target (dropoffs)
                    if 'dropoffs_rolling_24h' in regressor_cols_pickups:
                        vals_24h = []
                        for h in range(1, 25):  # Look back 1-24 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_dropoffs:
                                vals_24h.append(day_predictions_dropoffs[key])
                            else:
                                hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_24h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                        if len(vals_24h) >= 1:
                            row_p['dropoffs_rolling_24h'] = np.mean(vals_24h)
                    
                    # Create future dataframe for prediction
                    future_row_p = pd.DataFrame({
                        'ds': [current_hour],
                        **{col: [row_p[col]] for col in regressor_cols_pickups}
                    })
                    
                    # Make pickups prediction
                    forecast_p = pickups_model.predict(future_row_p)
                    predicted_value_p = forecast_p['yhat'].iloc[0]
                    day_predictions_pickups[(current_hour, cluster_id)] = predicted_value_p
                    
                    results_pickups.append({
                        'ds': current_hour,
                        'cluster': cluster_id,
                        'actual': row_p['y'],
                        'predicted': predicted_value_p,
                        'forecast_day': day_idx,
                        'hour_of_day': current_hour.hour
                    })
                
                # ========== DROPOFFS PREDICTION ==========
                mask_d = (df_working_dropoffs['ds'] == current_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                if mask_d.any():
                    row_d = df_working_dropoffs[mask_d].iloc[0].copy()
                    
                    # Update short-term lag features using predictions from earlier hours TODAY
                    if hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_1h'] = day_predictions_dropoffs[key]
                    
                    if hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_2h'] = day_predictions_dropoffs[key]
                    
                    if hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            row_d['lag_3h'] = day_predictions_dropoffs[key]
                    
                    # Update rolling averages
                    vals_3h = []
                    for h in range(1, 4):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_3h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_3h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_3h) >= 1:
                        row_d['rolling_3h'] = np.mean(vals_3h)
                    
                    # rolling_6h: rolling 6h average of own target
                    vals_6h = []
                    for h in range(1, 7):
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_6h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_6h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_6h) >= 1:
                        row_d['rolling_6h'] = np.mean(vals_6h)
                    
                    # rolling_24h: rolling 24h average of own target
                    vals_24h = []
                    for h in range(1, 25):  # Look back 1-24 hours
                        prev_hour = current_hour - pd.Timedelta(hours=h)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_dropoffs:
                            vals_24h.append(day_predictions_dropoffs[key])
                        else:
                            hist_mask = (df_working_dropoffs['ds'] == prev_hour) & (df_working_dropoffs['cluster'] == cluster_id)
                            if hist_mask.any():
                                vals_24h.append(df_working_dropoffs[hist_mask]['y'].iloc[0])
                    if len(vals_24h) >= 1:
                        row_d['rolling_24h'] = np.mean(vals_24h)
                    
                    # Cross-target lags: use pickups predictions if available
                    if 'pickups_lag_1h' in regressor_cols_dropoffs and hour_idx >= 1:
                        prev_hour = current_hour - pd.Timedelta(hours=1)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_1h'] = day_predictions_pickups[key]
                    
                    if 'pickups_lag_2h' in regressor_cols_dropoffs and hour_idx >= 2:
                        prev_hour = current_hour - pd.Timedelta(hours=2)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_2h'] = day_predictions_pickups[key]
                    
                    if 'pickups_lag_3h' in regressor_cols_dropoffs and hour_idx >= 3:
                        prev_hour = current_hour - pd.Timedelta(hours=3)
                        key = (prev_hour, cluster_id)
                        if key in day_predictions_pickups:
                            row_d['pickups_lag_3h'] = day_predictions_pickups[key]
                    
                    # opposite_rolling_3h: rolling 3h average of opposite target (pickups)
                    if 'pickups_rolling_3h' in regressor_cols_dropoffs:
                        vals_3h = []
                        for h in range(1, 4):  # Look back 1-3 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_pickups:
                                vals_3h.append(day_predictions_pickups[key])
                            else:
                                hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_3h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                        if len(vals_3h) >= 1:
                            row_d['pickups_rolling_3h'] = np.mean(vals_3h)
                    
                    # opposite_rolling_24h: rolling 24h average of opposite target (pickups)
                    if 'pickups_rolling_24h' in regressor_cols_dropoffs:
                        vals_24h = []
                        for h in range(1, 25):  # Look back 1-24 hours
                            prev_hour = current_hour - pd.Timedelta(hours=h)
                            key = (prev_hour, cluster_id)
                            if key in day_predictions_pickups:
                                vals_24h.append(day_predictions_pickups[key])
                            else:
                                hist_mask = (df_working_pickups['ds'] == prev_hour) & (df_working_pickups['cluster'] == cluster_id)
                                if hist_mask.any():
                                    vals_24h.append(df_working_pickups[hist_mask]['y'].iloc[0])
                        if len(vals_24h) >= 1:
                            row_d['pickups_rolling_24h'] = np.mean(vals_24h)
                    
                    # Create future dataframe for prediction
                    future_row_d = pd.DataFrame({
                        'ds': [current_hour],
                        **{col: [row_d[col]] for col in regressor_cols_dropoffs}
                    })
                    
                    # Make dropoffs prediction
                    forecast_d = dropoffs_model.predict(future_row_d)
                    predicted_value_d = forecast_d['yhat'].iloc[0]
                    day_predictions_dropoffs[(current_hour, cluster_id)] = predicted_value_d
                    
                    results_dropoffs.append({
                        'ds': current_hour,
                        'cluster': cluster_id,
                        'actual': row_d['y'],
                        'predicted': predicted_value_d,
                        'forecast_day': day_idx,
                        'hour_of_day': current_hour.hour
                    })
        
        # NOTE: We do NOT update df_working['y'] with predictions
        # Long-term lags (lag_24h, lag_168h) should use ACTUAL values from previous days
        # Short-term lags (lag_1h, lag_2h, lag_3h) and rolling means use predictions via day_predictions dict
        
        # Progress update
        if day_idx % 5 == 0 or day_idx == total_days:
            pct = (day_idx / total_days) * 100
            print(f"Completed {day_idx}/{total_days} days ({pct:.1f}%)")
    
    print("✓ Sequential predictions complete for both pickups and dropoffs")
    return pd.DataFrame(results_pickups), pd.DataFrame(results_dropoffs)

print("Sequential rolling forecast function defined.")
    

In [ ]:
# Train Prophet models for pickups and dropoffs
print("="*70)
print("TRAINING PROPHET MODELS")
print("="*70)

model_save_path='prophet_model_pickups_v4.pkl'

# Train pickups model
pickups_model, regressor_cols_pickups = train_prophet_model(
    all_pickups_data,
    train_val_end_date,
    'pickups',
    model_save_path=model_save_path
)
import pickle
with open(model_save_path+"_list", "wb") as f:
    pickle.dump(regressor_cols_pickups, f)


In [ ]:
print("\n" + "-"*70 + "\n")

model_save_path='prophet_model_dropoffs_v4.pkl'

# Train dropoffs model  
dropoffs_model, regressor_cols_dropoffs = train_prophet_model(
    all_dropoffs_data,
    train_val_end_date,
    'dropoffs',
    model_save_path=model_save_path
)

import pickle
with open(model_save_path+"_list", "wb") as f:
    pickle.dump(regressor_cols_pickups, f)

print("\n" + "="*70)
print("BOTH MODELS TRAINED SUCCESSFULLY")
print("="*70)

In [ ]:
import pickle

model_save_path_pickups = "prophet_model_pickups_v4.pkl" 
model_save_path_dropoffs = "prophet_model_dropoffs_v4.pkl"

with open(model_save_path_pickups, 'rb') as f:
    pickups_model = pickle.load(f)

with open(model_save_path_dropoffs, 'rb') as f:
    dropoffs_model = pickle.load(f)

# Run sequential rolling forecasts for both pickups and dropoffs simultaneously
forecast_results_pickups_sequential_prophet, forecast_results_dropoffs_sequential_prophet = rolling_daily_forecast_sequential_prohpet(
    all_pickups_data,
    all_dropoffs_data,
    pickups_model,
    dropoffs_model,
    regressor_cols_pickups,
    regressor_cols_dropoffs,
    test_start_date,
    test_end_date
)

print("\n✓ All sequential forecasts complete!")

### Evaluation of performance between prophet and sarimax 

In [ ]:
# Import evaluation functions
from model_evaluation import evaluate_model_complete

# Run complete evaluation for SARIMAX model
evaluate_model_complete(
    forecast_results_pickups_sequential,
    forecast_results_dropoffs_sequential,
    model_name="SARIMAX Sequential"
)

# Run complete evaluation for SARIMAX model
evaluate_model_complete(
    forecast_results_pickups_sequential,
    forecast_results_dropoffs_sequential,
    model_name="SARIMAX Sequential"
)

# Run complete evaluation for Prophet model
evaluate_model_complete(
    forecast_results_pickups_sequential_prophet,
    forecast_results_dropoffs_sequential_prophet,
    model_name="Prophet Sequential"
)


SEQUENTIAL FORECAST EVALUATION (No Data Leakage)

PICKUPS:
  MAE:  9.57
  RMSE: 13.91
  R²:   0.9954

DROPOFFS:
  MAE:  9.76
  RMSE: 14.02
  R²:   0.9956

